# CoAD reimplementation

Own PyTorch implementation of CoAD (KDD 2026), written from the paper (Section 3, Appendix A, Appendix C.1).
The authors' repository is used only for the data loader, the evaluator and reference checks (tests T1, T2, T4).

`FAITHFUL_TO_CODE` switches between the paper description (`False`, main version) and the details of the
authors' code (`True`); the differences D1-D21 are listed in `ke-hoach-reimplement-coad.md`, section 5.
Explanations in Vietnamese: `huong-dan-reimplement-coad.md`.

Run order in every new runtime: 0.1 → 0.2 → 1.1, then any later part.

## 0. Setup

### 0.1. Session, repository and packages

Mount Drive (the account that owns `CoAD_project`), check out the pinned commit and make sure the packages import.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, random, shutil, subprocess, sys, time, zipfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

STORE = Path('/content/drive/MyDrive/CoAD_project')
REPO = Path('/content/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)

CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_reimpl.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    assert install.returncode == 0, 'Installation failed; inspect env/install_reimpl.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_reimpl.log'
print('Commit', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip(),
      '| packages OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

Mounted at /content/drive
Commit 3286813 | packages OK | GPU: Tesla T4


### 0.2. Data, loader, evaluator and reference code

Extracts TSB-AD-U from the archive on Drive (skipped when already extracted) and imports the parts of the repository
that are reused as is: `load_dataset`, `evaluate`, and the reference implementations used only by tests T1, T2 and T4.

In [2]:
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
assert len(list(RAW.iterdir())) == 870, 'raw/ must contain exactly 870 CSVs'

os.chdir(REPO)                       # load_dataset reads the relative path dataset/TSB-AD/raw
sys.path.insert(0, str(REPO))
from toolkit.utils import load_dataset
from evaluation.evaluator import evaluate
from model.frequency import time_to_timefreq       # reference for test T2 only
from toolkit import distort as reference_distort   # reference for test T1 only
from model.patch_soft_mask import PatchFrequencyMask  # reference for test T4 only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORTS = STORE / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Data: 870 files | device:', DEVICE)

Data: 870 files | device: cuda


## 1. Configuration and series

### 1.1. Hyper-parameters, seeds, series loader

Values from Appendix C.1 of the paper; batch size from `runners/run_all_CoAD.sh`.
`load_series` returns one series with the window length and training stride chosen exactly as in
`train_classify_mask.py` (lines 84-105), so both implementations see the same windows.

In [3]:
FAITHFUL_TO_CODE = False   # False: follow the paper (main version); True: follow the authors' code (D1-D21)
SEEDS = [46, 47, 48]
PATCH = 8                  # P
WINDOW_MULTIPLE = 4        # T = 4 x dominant period
N_FFT = 4                  # K, STFT size
LAMBDA = 10                # weight of the reconstruction loss
LR = 0.002
EPOCHS = 300
BATCH_SIZE = 128

def use_code_details(faithful=None):
    """Per-call override of FAITHFUL_TO_CODE (None = use the global switch)."""
    return FAITHFUL_TO_CODE if faithful is None else faithful

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_series(group):
    """One TSB-AD series (min-max scaled on train) plus the window settings of the authors' code."""
    train, test, label, period = [values[0] for values in load_dataset('TSB-AD', [str(group)])]
    multiple = WINDOW_MULTIPLE if len(train) // period > WINDOW_MULTIPLE else len(train) // period
    window = (period * multiple) // PATCH * PATCH
    period = min(period, window)
    n = len(train)
    train_stride = 2 if n < 1000 else 6 if n < 10000 else 10 if n < 60000 else PATCH if n < 100000 else period
    return dict(group=group, train=train, test=test, label=label,
                period=period, window=window, train_stride=train_stride)

print('FAITHFUL_TO_CODE =', FAITHFUL_TO_CODE, '| seeds', SEEDS)

FAITHFUL_TO_CODE = False | seeds [46, 47, 48]


### 1.2. The 5 series (chosen once, before any run)

Group 236 (SED, the pilot of the reproduction) plus one series per remaining family, drawn with
`numpy.random.default_rng(2026)` from the 64-series manifest. The choice is written to `eda/manifest_reimpl.csv`
the first time and only read afterwards, so it cannot depend on any result.

In [4]:
manifest64 = pd.read_csv(STORE / 'eda' / 'manifest_tsbad64.csv')
reimpl_manifest_file = STORE / 'eda' / 'manifest_reimpl.csv'
if not reimpl_manifest_file.exists():
    rng = np.random.default_rng(2026)
    chosen = [236]
    for family in ['MGAB', 'SVDB', 'IOPS', 'TODS']:
        chosen.append(int(rng.choice(sorted(manifest64.loc[manifest64['family'] == family, 'group']))))
    rows = manifest64.set_index('group').loc[chosen].reset_index()
    rows['selection_reason'] = ['Pilot series of the reproduction'] + ['default_rng(2026), one per family'] * 4
    rows.to_csv(reimpl_manifest_file, index=False)
REIMPL_MANIFEST = pd.read_csv(reimpl_manifest_file)
REIMPL_GROUPS = REIMPL_MANIFEST['group'].tolist()

overview = []
for g, family in zip(REIMPL_GROUPS, REIMPL_MANIFEST['family']):
    s = load_series(g)
    overview.append({'group': g, 'family': family, 'n_train': len(s['train']), 'n_test': len(s['test']),
                     'anomaly_%': round(100 * s['label'].mean(), 2), 'period': s['period'],
                     'window': s['window'], 'train_stride': s['train_stride']})
display(pd.DataFrame(overview))

,group,family,n_train,n_test,anomaly_%,period,window,train_stride
0,236,SED,3740,39999,3.68,125,496,6
1,232,MGAB,25000,100000,0.20,49,192,10
2,240,SVDB,18462,130400,4.60,114,456,10
3,260,IOPS,5000,20000,0.07,8,32,6
4,296,TODS,500,5000,4.08,100,400,2


## 2. Module 1: simulated anomalies (Appendix A)

### 2.1. Four distortions

`distort(x, period, kind)` takes clean windows `(B, T, 1)` and returns a distorted copy and point labels `(B, T)`
(1 on the distorted segment). Shared by both modes: segment length `int(period × U(0.1, 1))` (one draw per batch),
segment start drawn per window (the paper only says "from 0 to one dominant period", D21). Mode-specific parts (D12-D15):

| kind | paper (`faithful=False`) | code (`faithful=True`) |
|---|---|---|
| `uniform` | constant drawn in [min, max] of the segment | value at the segment start |
| `mirror` | flip across the y-axis (time reversal) or the x-axis (values mirrored about the segment mid-range), one choice per batch | time reversal |
| `scale` | lengthened or shortened by a factor 2-4 | lengthened only |
| `jitter` | Gaussian noise, variance 0.1 | Gaussian noise, std = 5% of the window range |

In [ ]:
DISTORTIONS = ['uniform', 'scale', 'jitter', 'mirror']

def distort(x, period, kind, faithful=None):
    """x: clean windows (B, T, 1). Returns (distorted copy, point labels (B, T))."""
    faithful = use_code_details(faithful)
    x = x.clone()
    B, T, _ = x.shape
    n = int(period * np.random.uniform(0.1, 1))           # segment length, one draw per batch
    if n == 0:
        n = 2 if kind == 'uniform' else 1
    if n >= T:
        n = T - 2
    start = torch.randint(0, T - n, (B, 1))               # segment start, one per window
    if faithful and kind == 'mirror':
        torch.randint(0, T - n, (B, 1))                   # unused draw of the authors' mirror_flip: keeps the RNG stream aligned
    rows = torch.arange(B)[:, None]

    if kind == 'scale':
        factor = np.random.randint(2, 5)
        if faithful or np.random.rand() < 0.5:            # lengthen: first n / factor points, each repeated
            n = n // factor * factor
            source = start + torch.arange(n) // factor
        else:                                             # shorten: every factor-th point of a longer span
            source = (start + torch.arange(n) * factor).clamp(max=T - 1)
    cols = start + torch.arange(n)
    segment = x[rows, cols]                               # (B, n, 1)

    if kind == 'uniform':
        if faithful:
            x[rows, cols] = x[rows, start]
        else:
            low, high = segment.min(1, keepdim=True).values, segment.max(1, keepdim=True).values
            x[rows, cols] = low + (high - low) * torch.rand(B, 1, 1, device=x.device)
    elif kind == 'scale':
        x[rows, cols] = x[rows, source]
    elif kind == 'jitter':
        if faithful:
            std = 0.05 * (x.max(1, keepdim=True).values - x.min(1, keepdim=True).values)
        else:
            std = 0.1 ** 0.5
        x[rows, cols] = segment + torch.randn(B, n, 1, device=x.device) * std
    elif kind == 'mirror':
        if faithful or np.random.rand() < 0.5:            # across the y-axis: reverse time
            x[rows, cols] = torch.flip(segment, [1])
        else:                                             # across the x-axis: mirror values inside the segment range
            x[rows, cols] = segment.min(1, keepdim=True).values + segment.max(1, keepdim=True).values - segment
    else:
        raise ValueError(f'unknown distortion {kind}')

    labels = torch.zeros(B, T, device=x.device)
    labels[rows, cols] = 1
    return x, labels

### 2.2. Test T1

(a) In code mode, `distort` must give **exactly** the output of the authors' functions for the same random state.
(b) In both modes: points outside the labelled segment are unchanged, and a uniform replacement is constant.
(c) Figure of the four distortions in both modes (compare with Figure 7 of the paper), saved to `reports/`.

In [6]:
REFERENCE = {'uniform': reference_distort.uniform_distort, 'scale': reference_distort.scale_distort,
             'jitter': reference_distort.jitering_distort, 'mirror': reference_distort.mirror_flip}
period, T = 125, 496
t = torch.arange(T, dtype=torch.float32)
x = (0.5 + 0.4 * torch.sin(2 * np.pi * t / period)).repeat(16, 1).unsqueeze(-1)   # 16 sine windows (16, T, 1)

for seed in range(20):
    for kind, reference in REFERENCE.items():
        set_seed(seed)
        expected, expected_labels = reference(x, period)
        set_seed(seed)
        ours, ours_labels = distort(x, period, kind, faithful=True)
        assert torch.allclose(ours, expected) and torch.equal(ours_labels, expected_labels), (kind, seed)

for faithful in [True, False]:
    for seed in range(20):
        for kind in DISTORTIONS:
            set_seed(seed)
            xd, labels = distort(x, period, kind, faithful=faithful)
            assert torch.equal(xd[labels == 0], x[labels == 0]), (kind, faithful)
            if kind == 'uniform':
                values = xd.squeeze(-1)[labels.bool()].view(len(x), -1)
                assert torch.all(values == values[:, :1]), (kind, faithful)

fig, axes = plt.subplots(2, 4, figsize=(16, 5), sharey=True)
for row, faithful in enumerate([False, True]):
    for col, kind in enumerate(DISTORTIONS):
        set_seed(col)
        xd, labels = distort(x[:1], period, kind, faithful=faithful)
        ax = axes[row, col]
        ax.plot(x[0, :, 0], color='0.7', label='original')
        ax.plot(xd[0, :, 0], color='C3', lw=1, label='distorted')
        idx = labels[0].nonzero().squeeze(-1)
        ax.axvspan(idx.min().item(), idx.max().item(), color='C3', alpha=0.1)
        ax.set_title(f"{kind} ({'code' if faithful else 'paper'})")
axes[0, 0].legend(loc='lower left', fontsize=8)
fig.tight_layout()
fig.savefig(REPORTS / 'reimpl_T1_distortions.png', dpi=120)
plt.show()
print('T1 passed: code mode identical to the authors\' functions (20 seeds x 4 kinds); invariants hold in both modes')

T1 passed: code mode identical to the authors' functions (20 seeds x 4 kinds); invariants hold in both modes


## 3. Module 2: patches and STFT (Section 3.3)

### 3.1. Time patches and frequency patches

`stft_features` turns each window into one STFT frame per time point (hop 1, Hann window of `N_FFT` points),
real and imaginary parts concatenated: one-sided spectrum, 3 bins, 6 features per point, in both modes.
The paper's K = 4 is read as the STFT size `n_fft = 4` (D17: a two-sided spectrum would only add the complex
conjugate of bin 1, i.e. no new information).
`make_patches` cuts both into `N = T / P` non-overlapping patches.

In [ ]:
def stft_features(x):
    """x: (B, T) -> (B, T, 2 * bins): real and imaginary part of every bin, one frame per time point."""
    spectrum = torch.stft(x, N_FFT, hop_length=N_FFT // 4, window=torch.hann_window(N_FFT, device=x.device),
                          normalized=True, onesided=True, return_complex=True)
    spectrum = torch.view_as_real(spectrum)            # (B, bins, frames, 2)
    spectrum = spectrum.permute(0, 2, 1, 3).flatten(2)  # (B, frames, bins * 2)
    return spectrum[:, :x.shape[1]]                     # frames = T + 1 (centred STFT); keep T

def make_patches(x, x_freq):
    """x: (B, T), x_freq: (B, T, F) -> time patches (B, N, P) and frequency patches (B, N, P * F)."""
    B, T = x.shape
    return x.reshape(B, T // PATCH, PATCH), x_freq.reshape(B, T // PATCH, -1)

### 3.2. Test T2

`stft_features` must equal the authors' `time_to_timefreq` (`torch.allclose`) and give the patch sizes of the plan
(section 4.2): time `(B, 62, 8)`, frequency `(B, 62, 48)`.

In [8]:
x = torch.rand(4, 496)
expected = time_to_timefreq(x.clone(), n_fft=N_FFT)[:, :x.shape[1]]
assert torch.allclose(stft_features(x), expected)

time_patches, freq_patches = make_patches(x, stft_features(x))
assert time_patches.shape == (4, 62, 8) and freq_patches.shape == (4, 62, 48)
assert torch.equal(time_patches[:, 1], x[:, 8:16])            # patch i holds points 8i .. 8i + 7
print(f'time patches {tuple(time_patches.shape)}, frequency patches {tuple(freq_patches.shape)}')
print('T2 passed: identical to time_to_timefreq')

time patches (4, 62, 8), frequency patches (4, 62, 48)
T2 passed: identical to time_to_timefreq


## 4. Modules 3-5: the model

### 4.1. `CoADReimpl`

Three parts, one class each:

* `TimeFreqClassifier`: module 3 (Eq. 1-4) and the residual classifier of module 5 (Eq. 7-8).
  One Bi-GRU is shared by the time and the frequency branch (Figure 2, "Shared GRU").
* `SoftMaskMAE`: module 4 (Eq. 5-6).
* `CoADReimpl`: training pass (`forward`) and inference (`predict`, Eq. 9).

Mode-specific choices: D1 (GRU layers), D2 (dropout), D4 (linear head or MLP), D5 (hidden size),
D10 (second reconstruction at inference), D18 (SELU after the projections).
Attribute names follow the authors' modules so that test T4 can load their weights.

In [ ]:
def head(in_size, hidden, faithful):
    """Patch classifier: linear in the paper (Eq. 2, 3, 8), two-layer MLP in the code (D4)."""
    if faithful:
        return nn.Sequential(nn.Linear(in_size, hidden), nn.SELU(), nn.Linear(hidden, 1, bias=False))
    return nn.Sequential(nn.Linear(in_size, 1))

def gru(size, faithful):
    """Bi-GRU encoder (Section 3.2): 2 layers and dropout 0.1 in the code, 3 layers in the paper (D1, D2)."""
    return nn.GRU(size, size, num_layers=2 if faithful else 3, batch_first=True, bidirectional=True,
                  dropout=0.1 if faithful else 0.0)

def init_linear(module):
    """Xavier-uniform weights and zero biases for every linear layer, as in the authors' code."""
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


class TimeFreqClassifier(nn.Module):
    """Module 3 (Eq. 1-4) and the residual classifier of module 5 (Eq. 7-8)."""

    def __init__(self, faithful):
        super().__init__()
        self.faithful = faithful
        freq_size = PATCH * 2 * (N_FFT // 2 + 1)                  # 8 points x 3 bins x (real, imaginary) = 48
        hidden = 3 * freq_size if faithful else 24                # D5
        activation = [nn.SELU()] if faithful else []              # D18
        self.proj_freq = nn.Sequential(nn.Linear(freq_size, hidden), *activation)   # W_f^p
        self.proj_time = nn.Sequential(nn.Linear(PATCH, hidden), *activation)       # W_t^p
        self.encoder = gru(hidden, faithful)                      # shared by both branches (D3)
        self.classifier_time = head(2 * hidden, hidden, faithful)       # W_t (created first, as in the authors' code,
        self.classifier_freq = head(2 * hidden, hidden, faithful)       # W_f  so a seed gives the same initial weights)
        self.classifier_residual = head(2 * hidden, hidden, faithful)   # W_r, shared by both residual branches
        init_linear(self)

    def features(self, x):
        """x: (B, T) -> encoder outputs of the frequency and of the time patches, each (B, N, 2H)."""
        time_patches, freq_patches = make_patches(x, stft_features(x))
        projected = torch.cat([self.proj_freq(freq_patches), self.proj_time(time_patches)])
        encoded, _ = self.encoder(projected)
        return encoded.chunk(2)

    def forward(self, x):
        """Eq. 1-4: logit of A = max(A_f, A_t), (B, N, 1). Max of logits = max of probabilities (sigmoid is increasing)."""
        h_freq, h_time = self.features(x)
        return torch.maximum(self.classifier_freq(h_freq), self.classifier_time(h_time))

    def residual_logit(self, x, x_rec):
        """Eq. 7-8: logit of A^r from the feature difference between the input and its reconstruction."""
        h_freq, h_time = self.features(torch.cat([x, x_rec]))
        (freq_in, freq_rec), (time_in, time_rec) = h_freq.chunk(2), h_time.chunk(2)
        return torch.maximum(self.classifier_residual(freq_in - freq_rec),
                             self.classifier_residual(time_in - time_rec))


class SoftMaskMAE(nn.Module):
    """Module 4 (Eq. 5-6): patch embeddings softly replaced by a learned mask embedding, then reconstructed."""

    def __init__(self, faithful):
        super().__init__()
        hidden = 3 * PATCH                                        # 24 in both modes
        self.proj = nn.Linear(PATCH, hidden)                      # W_m
        self.act = nn.SELU() if faithful else nn.Identity()       # D18
        self.encoder = gru(hidden, faithful)
        self.head = nn.Linear(2 * hidden, PATCH)                  # W_o
        self.mask_embedding = nn.Parameter(torch.randn(hidden))   # E_mask, one vector repeated over patches (D19)
        init_linear(self)

    def forward(self, x, prob):
        """x: (B, T), prob: patch anomaly probability (B, N, 1) -> reconstruction X_r (B, T)."""
        B, T = x.shape
        embedding = self.act(self.proj(x.reshape(B, T // PATCH, PATCH)))
        mixed = prob * self.mask_embedding + (1 - prob) * embedding    # Eq. 5
        encoded, _ = self.encoder(mixed)
        return self.head(encoded).reshape(B, T)                         # Eq. 6


class CoADReimpl(nn.Module):
    def __init__(self, faithful=None):
        super().__init__()
        self.faithful = use_code_details(faithful)
        self.classifier = TimeFreqClassifier(self.faithful)
        self.mae = SoftMaskMAE(self.faithful)

    def forward(self, x):
        """Training pass. x: (B, T) -> logits of A and A^r (B, N, 1), reconstruction X_r (B, T)."""
        logit = self.classifier(x)
        x_rec = self.mae(x, torch.sigmoid(logit))
        return logit, self.classifier.residual_logit(x, x_rec), x_rec

    @torch.no_grad()
    def predict(self, x):
        """Inference (call .eval() first). x: (B, T) -> A_c (B, N, 1) (Eq. 9) and the reconstruction (B, T)."""
        prob = torch.sigmoid(self.classifier(x))
        x_rec = self.mae(x, prob)
        prob = (prob + torch.sigmoid(self.classifier.residual_logit(x, x_rec))) / 2
        if self.faithful:                   # D10: reconstruct again, masked with A_c
            x_rec = self.mae(x, prob)
        return prob, x_rec

    def load_official(self, official):
        """Copy the weights of an authors' PatchFrequencyMask (code mode; test T4)."""
        renamed = {k.replace('detection_net.', 'classifier.').replace('reconstruction_net.', 'mae.'): v
                   for k, v in official.state_dict().items()
                   if k.startswith(('detection_net.', 'reconstruction_net.'))}
        self.load_state_dict(renamed)

### 4.2. Test T3: shapes

One random batch through both modes: logits `(B, N, 1)`, reconstruction `(B, T)`, probabilities in [0, 1];
with `X_r = X` the residual is zero and the residual logit must be the same constant for every patch.

In [10]:
x = torch.rand(4, 496)
for faithful in [False, True]:
    model = CoADReimpl(faithful)
    logit, residual_logit, x_rec = model(x)
    prob, rec = model.eval().predict(x)
    assert logit.shape == residual_logit.shape == prob.shape == (4, 62, 1)
    assert x_rec.shape == rec.shape == (4, 496) and ((prob >= 0) & (prob <= 1)).all()
    with torch.no_grad():
        same = model.classifier.residual_logit(x, x)          # X_r = X -> zero residual
    assert torch.allclose(same, same[0, 0].expand_as(same), atol=1e-6)   # zero bias: tolerance must be absolute
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{'code ' if faithful else 'paper'} mode: logits {tuple(logit.shape)}, reconstruction {tuple(x_rec.shape)}, "
          f"{n_params:,} parameters")
print('T3 passed')

paper mode: logits (4, 62, 1), reconstruction (4, 496), 59,195 parameters
code  mode: logits (4, 62, 1), reconstruction (4, 496), 777,656 parameters
T3 passed


### 4.3. Test T4: same weights, same outputs

Loads the authors' model of group 236 saved by the pilot run (`runs/<run_id>/artifacts/model/group_236.pth`,
section 20.3 of the reproduction) into `CoADReimpl` in code mode, and feeds both the same 32 test windows.
Without that file the test uses a freshly initialised authors' model, which checks the architecture just as well.

In [11]:
checkpoints = sorted(STORE.glob('runs/*/artifacts/model/group_236.pth'), key=lambda p: p.stat().st_mtime)
if checkpoints:
    official = torch.load(checkpoints[-1], map_location='cpu', weights_only=False)
    source = 'trained checkpoint of run ' + checkpoints[-1].parents[2].name
else:
    official = PatchFrequencyMask(patch_size=PATCH, expansion_ratio=3, num_layers=2, n_fft=N_FFT, patch_num=None,
                                  recon_mode='soft', detect_mode='freq_time_v2')
    source = "randomly initialised authors' model (no group_236.pth on Drive)"
official = official.float().eval()
ours = CoADReimpl(faithful=True).eval()
ours.load_official(official)

s236 = load_series(236)
starts = np.arange(32) * s236['period']
x = torch.tensor(np.stack([s236['test'][i:i + s236['window']] for i in starts]), dtype=torch.float32)
with torch.no_grad():
    official_prob, official_rec = official.predict(x.unsqueeze(-1))   # (B, T, 1): probability repeated per point
    prob, rec = ours.predict(x)
prob_gap = (prob.repeat_interleave(PATCH, dim=1) - official_prob).abs().max().item()
rec_gap = (rec - official_rec.squeeze(-1)).abs().max().item()
print(f'T4 with {source}: max |difference| probability {prob_gap:.1e}, reconstruction {rec_gap:.1e}')
assert prob_gap < 1e-5 and rec_gap < 1e-5
print("T4 passed: in code mode the reimplementation reproduces the authors' model output")

T4 with trained checkpoint of run 20261017T031415088023Z_coad_sanity_TSB-AD_236: max |difference| probability 0.0e+00, reconstruction 0.0e+00
T4 passed: in code mode the reimplementation reproduces the authors' model output


## 5. Modules 6-7: loss and training

### 5.1. Windows, labels, loss

* `make_windows` / `windows_to_series`: sliding windows with zero padding at the front, so that the last window
  ends on the last point; overlapping window outputs are averaged back (same layout as the authors' code).
* `patch_labels`: a patch is anomalous if at least one of its points was distorted.
* `training_batch`: one random distortion type per batch. Code mode stacks a clean copy (all labels 0) with the
  distorted copy (D9). The reconstruction target is always the clean window.
* `coad_loss`: Eq. 10, `L = L_BCE + λ L_MSE`; code mode uses its own versions of both terms (D6-D8).

In [ ]:
def make_windows(values, window, stride):
    """values (L,) -> windows (M, window) float32 and the front padding length."""
    L = len(values)
    pad = 0 if stride == 1 or (L - window) % stride == 0 else stride - (L - window) % stride
    padded = np.concatenate([np.zeros(pad), values])
    starts = np.arange(0, len(padded) - window + 1, stride)
    return np.stack([padded[s:s + window] for s in starts]).astype(np.float32), pad

def windows_to_series(window_values, length, stride, pad):
    """window_values (M, window) -> (length,): mean over the overlapping windows, padding removed."""
    total, count = np.zeros(pad + length), np.zeros(pad + length)
    for i, values in enumerate(window_values):
        total[i * stride:i * stride + len(values)] += values
        count[i * stride:i * stride + len(values)] += 1
    return total[pad:] / count[pad:]

def patch_labels(labels):
    """Point labels (B, T) -> patch labels (B, N, 1)."""
    B, T = labels.shape
    return labels.reshape(B, T // PATCH, PATCH).amax(-1, keepdim=True)

def training_batch(clean, period, faithful, code_distortion=None):
    """Clean windows (B, T) -> model input, point labels, reconstruction target.
    code_distortion (M6-C only): distortion details D12-D15 chosen apart from the batch layout D9."""
    code_distortion = faithful if code_distortion is None else code_distortion
    distorted, labels = distort(clean.unsqueeze(-1), period, random.choice(DISTORTIONS), code_distortion)
    distorted = distorted.squeeze(-1)
    if faithful:                                    # D9: clean copy + distorted copy
        return torch.cat([clean, distorted]), torch.cat([torch.zeros_like(labels), labels]), torch.cat([clean, clean])
    return distorted, labels, clean

def coad_loss(model, x, labels, target, code_loss=None):
    """Eq. 10. Returns (total, classification term, reconstruction term).
    code_loss (M6-C only): loss details D6-D8, D20 chosen apart from the model."""
    logit, residual_logit, x_rec = model(x)
    y = patch_labels(labels)
    if (model.faithful if code_loss is None else code_loss):
        pos_weight = torch.tensor([2.0], device=x.device)                                    # D6
        classify = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight) +
                    F.binary_cross_entropy_with_logits(residual_logit, y, pos_weight=pos_weight)) / 2   # D7
        error = (x_rec - target).pow(2).reshape(y.shape[0], y.shape[1], PATCH)
        anomalous = (error * y).sum() / (y.sum() * PATCH + 1e-7)
        normal = (error * (1 - y)).sum() / ((1 - y).sum() * PATCH + 1e-7)
        recon = 0.5 * anomalous + 0.5 * normal                                               # D8
    else:
        prob = (torch.sigmoid(logit) + torch.sigmoid(residual_logit)) / 2                    # A_c, Eq. 9
        classify = F.binary_cross_entropy(prob, y)
        recon = PATCH * F.mse_loss(x_rec, target)      # Eq. 10: squared norm summed over a patch, averaged over patches (D20)
    return classify + LAMBDA * recon, classify, recon

### 5.2. Training loop

Adam, lr 0.002, 300 epochs, batch 128, windows of the training part only. Code mode adds the learning-rate
decay (StepLR, γ = 0.99 per epoch) and fp16 mixed precision of the authors' code (D16).

In [ ]:
def train_model(series, faithful=None, seed=46, epochs=None, log_every=50, groups=None):
    """Train CoADReimpl on the training part of one series. Returns the model and the mean losses per epoch.
    groups (M6-C only): set of code-only detail groups (section 12) applied on top of paper mode."""
    epochs = EPOCHS if epochs is None else epochs
    set_seed(seed)
    if groups is None:
        faithful = use_code_details(faithful)
        code_distortion = faithful
        model = CoADReimpl(faithful).to(DEVICE)
    else:
        faithful, code_distortion = 'training' in groups, 'distortion' in groups
        model = HybridCoAD(groups).to(DEVICE)
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99) if faithful else None   # D16
    use_amp = faithful and DEVICE.type == 'cuda'                                                            # D16
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = []
    for epoch in range(epochs):
        model.train()
        sums = np.zeros(3)
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], faithful, code_distortion)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                losses = coad_loss(model, x, labels, target, faithful)
            scaler.scale(losses[0]).backward()
            scaler.step(optimizer)
            scaler.update()
            sums += [value.item() for value in losses]
        if scheduler:
            scheduler.step()
        history.append(dict(zip(['loss', 'classify', 'recon'], sums / len(loader))))
        if log_every and (epoch + 1) % log_every == 0:
            print(f'  epoch {epoch + 1}/{epochs}: ' + ', '.join(f'{k} {v:.4f}' for k, v in history[-1].items()),
                  flush=True)
    return model, pd.DataFrame(history)

### 5.3. Test T5: overfit one batch

500 optimisation steps on one fixed batch of 32 windows of group 236 (one fixed distortion).
A correct model and loss can memorise it: the loss must fall below 10% of its first value.

In [14]:
s236 = load_series(236)
set_seed(0)
model = CoADReimpl().to(DEVICE)
windows, _ = make_windows(s236['train'], s236['window'], s236['train_stride'])
x, labels, target = training_batch(torch.from_numpy(windows[:32]).to(DEVICE), s236['period'], model.faithful)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
losses = []
for step in range(500):
    optimizer.zero_grad()
    loss = coad_loss(model, x, labels, target)[0]
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f'loss: step 1 {losses[0]:.4f} -> step 500 {losses[-1]:.4f} ({100 * losses[-1] / losses[0]:.1f}% of the start)')
assert losses[-1] < 0.1 * losses[0]
print('T5 passed')

loss: step 1 23.3619 -> step 500 1.3638 (5.8% of the start)
T5 passed


### 5.4. Test T6: full training on group 236

300 epochs, seed 46, mode of `FAITHFUL_TO_CODE` (about 3-5 minutes on a T4). No NaN; the mean loss of the last
10 epochs must be lower than that of the first 10. The loss curve is saved to `reports/`.

In [15]:
MODE = 'code' if FAITHFUL_TO_CODE else 'paper'
start = time.time()
model_236, history_236 = train_model(s236, seed=SEEDS[0])
print(f'{MODE} mode, group 236: {time.time() - start:.0f} s')
assert np.isfinite(history_236.values).all()
k = max(1, min(10, len(history_236) // 3))          # 10 epochs, fewer when EPOCHS is small
assert history_236['loss'][-k:].mean() < history_236['loss'][:k].mean()

ax = history_236.plot(logy=True, figsize=(8, 3), title=f'Training loss, group 236, {MODE} mode')
ax.set_xlabel('epoch')
ax.figure.savefig(REPORTS / f'reimpl_T6_loss_236_{MODE}.png', dpi=120, bbox_inches='tight')
plt.show()
print('T6 passed')

  epoch 50/300: loss 2.5952, classify 0.4689, recon 0.2126
  epoch 100/300: loss 1.9931, classify 0.4075, recon 0.1586
  epoch 150/300: loss 1.9718, classify 0.3531, recon 0.1619
  epoch 200/300: loss 1.2856, classify 0.2407, recon 0.1045
  epoch 250/300: loss 0.8999, classify 0.1827, recon 0.0717
  epoch 300/300: loss 1.1582, classify 0.2109, recon 0.0947
paper mode, group 236: 28 s
T6 passed


## 6. Module 8: inference and anomaly score (Eq. 11)

### 6.1. Score of a whole series

Test windows with stride = period, per-point values averaged over overlapping windows.
Paper mode: `a(x) = A_c + |X_r − X|`, then a moving average over one period (Eq. 11).
Code mode (D11): each branch measured relative to its mean on the training part, smoothed, first `period` points
set to 0, then the two branches averaged.

In [ ]:
@torch.no_grad()
def predict_series(model, values, window, stride):
    """Whole series -> per-point A_c and reconstruction, averaged over overlapping windows."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    probs, recs = [], []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        prob, rec = model.predict(batch.to(DEVICE))
        probs.append(prob.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        recs.append(rec.cpu())
    return (windows_to_series(torch.cat(probs).numpy(), len(values), stride, pad),
            windows_to_series(torch.cat(recs).numpy(), len(values), stride, pad))

def moving_average(score, width):
    """Centred moving average keeping the length (np.convolve 'same', as in the authors' code)."""
    return np.convolve(score, np.ones(width) / width, mode='same')

def anomaly_score(model, series):
    """Per-point anomaly score of the test series."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    if not model.faithful:
        return moving_average(test_prob + np.abs(series['test'] - test_rec), period)    # Eq. 11
    train_prob, train_rec = predict_series(model, series['train'], window, period)      # D11
    recon = np.abs(np.abs(series['test'] - test_rec) - np.abs(series['train'] - train_rec).mean())
    classify = np.abs(test_prob - train_prob.mean())
    recon, classify = moving_average(recon, period), moving_average(classify, period)
    recon[:period] = classify[:period] = 0
    return (recon + classify) / 2

METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']

def score_metrics(score, series):
    """Checks of test T7, then the repository evaluator (values in 0-1)."""
    assert len(score) == len(series['label']), 'score and label lengths differ'
    assert np.isfinite(score).all(), 'score contains NaN or Inf'
    result = evaluate(ground_truth=series['label'], anomaly_scores=score, subsequence=series['period'], mode='test')
    row = {k: float(result.point_wise[k]) for k in METRIC_KEYS[:3]}
    row.update({k: float(result.vus[k]) for k in METRIC_KEYS[3:]})
    return row

def plot_scores(series, scores, path=None):
    """Test series on top, one min-max scaled score per row below; labelled anomalies shaded."""
    fig, axes = plt.subplots(len(scores) + 1, 1, figsize=(14, 1.6 * (len(scores) + 1)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    rows = [('test data', series['test'])] + list(scores.items())
    for ax, (name, values) in zip(axes, rows):
        values = np.asarray(values, dtype=float)
        ax.plot((values - values.min()) / (values.max() - values.min() + 1e-12), lw=0.6)
        ax.fill_between(np.arange(len(values)), 0, 1, where=series['label'] == 1, color='C3', alpha=0.25,
                        transform=ax.get_xaxis_transform())
        ax.set_ylabel(name, rotation=0, ha='right', fontsize=8)
        ax.set_yticks([])
    fig.suptitle(f"group {series['group']}")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    plt.show()

### 6.2. Test T7: score of the T6 model

Score length = label length, no NaN/Inf, the evaluator runs and all metrics lie in [0, 1].

In [17]:
score_236 = anomaly_score(model_236, s236)
metrics_236 = score_metrics(score_236, s236)
assert all(0 <= v <= 1 for v in metrics_236.values())
print(f'{MODE} mode, group 236:', ', '.join(f'{k} {100 * v:.2f}' for k, v in metrics_236.items()))
plot_scores(s236, {f'reimpl ({MODE})': score_236}, REPORTS / f'reimpl_T7_score_236_{MODE}.png')
print('T7 passed')

paper mode, group 236: f1_score 81.94, auc_prc 77.91, auc_roc 99.47, R_AUC_ROC 99.92, R_AUC_PR 97.74, VUS_ROC 99.84, VUS_PR 94.88
T7 passed


## 7. Experiments: 64 series × 2 modes × 3 seeds

All 64 TSB-AD series of Table 1, both modes, seeds 46-48: 384 runs. The 30 runs on the 5 chosen series
(section 1.2) always come first, so the core comparison of 8.1 is ready long before the full grid of 8.3.

### 7.1. Run settings and one-run helper

Same conventions as `CoAD_experiments.ipynb`: each session appends to its own results file
(`results/reimpl_results_<SESSION_TAG>.csv`), finished runs are skipped, a failing run is recorded and the loop
goes on. Each run keeps `config.json`, `loss.csv`, `scores.npy`, `metrics.json` and `model.pt` in
`runs/reimpl/<time>_<mode>_<group>_s<seed>/`.

To run several notebook copies at the same time, give each copy its own `SESSION_TAG` and a different part of the
grid, e.g. `RUN_SEEDS = [46]` / `[47]` / `[48]`, or `RUN_MODES = ['paper']` / `['code']`. Two copies must never
share a (mode, seed) pair: a session sees finished runs of the others, not running ones.

In [ ]:
RESULTS_DIR = STORE / 'results'
RUNS = STORE / 'runs' / 'reimpl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

SESSION_TAG = 'gpu-a'            # unique per notebook copy running at the same time
RESULTS = RESULTS_DIR / f'reimpl_results_{SESSION_TAG}.csv'
RUN_MODES = ['paper', 'code']
RUN_GROUPS = REIMPL_GROUPS + [g for g in manifest64['group'] if g not in REIMPL_GROUPS]   # 5 chosen series first, then the other 59
RUN_SEEDS = SEEDS
TIME_BUDGET_MIN = 150            # no new run after this many minutes; re-run 7.2 to continue
COLUMNS = ['run_id', 'mode', 'group', 'seed', 'status', 'seconds', 'error'] + METRIC_KEYS

def read_reimpl_results():
    """Rows of every session, oldest run first."""
    files = sorted(RESULTS_DIR.glob('reimpl_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.sort_values('run_id', kind='stable').reset_index(drop=True)

def run_reimpl(mode, group, seed):
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{mode}_{group}_s{seed}"
    out = RUNS / run_id
    out.mkdir(parents=True)
    row = dict(run_id=run_id, mode=mode, group=group, seed=seed)
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=(mode == 'code'), seed=seed, log_every=0)
        score = anomaly_score(model, series)
        metrics = score_metrics(score, series)
        torch.save(model.state_dict(), out / 'model.pt')
        np.save(out / 'scores.npy', score.astype(np.float32))
        history.to_csv(out / 'loss.csv', index=False)
        (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
        row.update(status='success', **metrics)
    except Exception as error:
        row.update(status='failed', error=f'{type(error).__name__}: {error}'[:300])
    row['seconds'] = round(time.time() - start, 1)
    config = dict(row, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patch=PATCH, n_fft=N_FFT, lam=LAMBDA,
                  commit=commit, device=str(DEVICE))
    (out / 'config.json').write_text(json.dumps(config, indent=2, default=str))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

print('Session', SESSION_TAG, '| writes to', RESULTS.name, '|', len(RUN_MODES) * len(RUN_GROUPS) * len(RUN_SEEDS), 'runs planned')

### 7.2. Run (re-run to continue)

First the 30 runs on the 5 chosen series (seed 46 for both modes, then 47, then 48), then the other 59 series in the
same seed order.

In [19]:
done = read_reimpl_results().query("status == 'success'")
finished = set(zip(done['mode'], done['group'].astype(int), done['seed'].astype(int)))
todo = [(m, g, s) for s in RUN_SEEDS for m in RUN_MODES for g in RUN_GROUPS if (m, g, s) not in finished]
todo.sort(key=lambda run: run[1] not in REIMPL_GROUPS)    # stable sort: runs on the 5 chosen series first
print(len(todo), 'runs to do')
session_start = time.time()
for mode, group, seed in todo:
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    print(f'{datetime.now(timezone.utc):%H:%M} UTC  {mode:5} group {group} seed {seed} ...', end=' ', flush=True)
    row = run_reimpl(mode, group, seed)
    if row['status'] == 'success':
        print(f"{row['seconds']:.0f} s | " + ' '.join(f"{k} {100 * row[k]:.2f}" for k in ['f1_score', 'auc_prc', 'R_AUC_PR', 'VUS_PR']))
    else:
        print('FAILED:', row['error'])

0 runs to do


## 8. Results

### 8.1. Reimplementation vs official run

Official values: the CoAD row of each series in `results/tsbad64_results*.csv` (64-series run, seed 46).
Reimplementation: mean and standard deviation over the finished seeds. Values ×100.
Criterion 1 of the plan (section 6.4): in code mode, the mean VUS-PR over the 5 series lies within ±5 points of
the official one. Saved to `reports/reimpl_vs_official.csv`.

In [20]:
TABLE_KEYS = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
official_rows = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))], ignore_index=True)
official = (official_rows.query("method == 'CoAD' and status == 'success'")
            .sort_values('run_id').drop_duplicates('group', keep='last').set_index('group'))
ours = read_reimpl_results().query("status == 'success'").drop_duplicates(['mode', 'group', 'seed'], keep='last')

missing = [g for g in REIMPL_GROUPS if g not in official.index]
assert not missing, f'no successful official CoAD run for groups {missing}: finish them in CoAD_experiments.ipynb'
rows = []
for g in REIMPL_GROUPS:
    for key, name in TABLE_KEYS.items():
        row = {'group': g, 'metric': name, 'official': 100 * official.loc[g, key]}
        for mode in ['paper', 'code']:
            values = 100 * ours.loc[(ours['mode'] == mode) & (ours['group'] == g), key]
            row.update({f'{mode}_mean': values.mean(), f'{mode}_std': values.std(), f'{mode}_seeds': len(values)})
        rows.append(row)
comparison = pd.DataFrame(rows)
comparison['paper - official'] = comparison['paper_mean'] - comparison['official']
comparison['code - official'] = comparison['code_mean'] - comparison['official']
comparison.to_csv(REPORTS / 'reimpl_vs_official.csv', index=False)
display(comparison.round(2))

complete = comparison.dropna(subset=['paper_mean', 'code_mean'])   # series finished in both modes
n_series = complete['group'].nunique()
assert n_series > 0, 'no series finished in both modes yet: run 7.2 first'
average = complete.groupby('metric', sort=False)[['official', 'paper_mean', 'code_mean']].mean()
print(f'Mean over the {n_series} series finished in both modes:')
display(average.round(2))
gap = average.loc['VUS-PR', 'code_mean'] - average.loc['VUS-PR', 'official']
print(f'Criterion 1 (code mode, mean VUS-PR within ±5 points of official): {gap:+.2f} ->',
      'met' if abs(gap) <= 5 else 'NOT met', '' if n_series == len(REIMPL_GROUPS) else f'(provisional: {n_series} of {len(REIMPL_GROUPS)} series)')

,group,metric,official,paper_mean,paper_std,paper_seeds,code_mean,code_std,code_seeds,paper - official,code - official
0,236,F1,78.70,76.56,4.07,3,74.42,0.56,3,-2.14,-4.29
1,236,AUC-PR,76.92,72.81,10.13,3,67.71,1.60,3,-4.11,-9.21
2,236,R-AUC-PR,95.26,94.00,4.18,3,92.87,1.25,3,-1.27,-2.39
3,236,VUS-PR,92.15,90.44,4.44,3,89.13,0.92,3,-1.71,-3.02
4,232,F1,50.13,46.65,2.34,3,49.26,3.85,3,-3.48,-0.86
5,232,AUC-PR,40.58,31.50,2.84,3,38.28,4.06,3,-9.08,-2.30
6,232,R-AUC-PR,43.96,25.57,3.24,3,43.08,8.79,3,-18.39,-0.88
7,232,VUS-PR,43.76,24.53,3.12,3,42.46,8.49,3,-19.23,-1.31
8,240,F1,64.99,76.21,1.48,3,64.34,4.91,3,11.22,-0.65
9,240,AUC-PR,68.75,81.60,0.16,3,65.23,6.53,3,12.85,-3.53


Mean over the 5 series finished in both modes:


,official,paper_mean,code_mean
metric,,,
F1,61.53,62.70,60.02
AUC-PR,56.28,57.52,53.00
R-AUC-PR,59.27,56.66,58.64
VUS-PR,58.07,54.72,56.77


Criterion 1 (code mode, mean VUS-PR within ±5 points of official): -1.30 -> met 


### 8.2. Score plots

Official score and reimplementation scores (seed 46, both modes) on group 236 and on the TODS series, the family
where CoAD is weakest. Saved to `reports/reimpl_scores_<group>.png`.

In [ ]:
def latest(pattern):
    files = sorted(STORE.glob(pattern))
    return files[-1] if files else None

for g in [236, int(REIMPL_MANIFEST.loc[REIMPL_MANIFEST['family'] == 'TODS', 'group'].iloc[0])]:
    series = load_series(g)
    scores = {}
    official_file = latest(f'runs/tsbad64/*_CoAD_TSB-AD_{g}/artifacts/anomaly_score/group_{g:03d}_test_anomaly_score.npy')
    if official_file:
        scores['official'] = np.load(official_file).reshape(-1)
    for mode in ['paper', 'code']:
        score_file = latest(f'runs/reimpl/*_{mode}_{g}_s{SEEDS[0]}/scores.npy')
        if score_file:
            scores[f'reimpl ({mode})'] = np.load(score_file)
    if scores:
        plot_scores(series, scores, REPORTS / f'reimpl_scores_{g}.png')
    else:
        print(f'group {g}: no score file yet')

### 8.3. All 64 series: paper, official run, reimplementation

The TSB-AD column of Table 1 for CoAD, next to the official run (`CoAD_experiments.ipynb`, seed 46) and the two
reimplementation modes. A mode/seed enters the table only when it has all 64 series; the reimplementation rows are
the mean over its complete seeds, with the standard deviation over seeds in the second table. Code mode with 3 seeds
also gives the seed-to-seed spread of the authors' method itself (code mode = authors' code, section 4.3).
Saved to `reports/reimpl_tsbad64_summary.csv` and `reports/reimpl_tsbad64_by_family.csv`.

In [22]:
TABLE1_COAD = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}   # paper, Table 1, TSB-AD
groups64 = manifest64['group'].tolist()
missing = [g for g in groups64 if g not in official.index]
assert not missing, f'no successful official CoAD run for groups {missing}'
grid = ours[ours['group'].isin(groups64)]
coverage = grid.groupby(['mode', 'seed'])['group'].nunique().unstack(fill_value=0)
print('Series finished per mode (rows) and seed (columns), out of 64:')
display(coverage)

means = {'paper (Table 1)': TABLE1_COAD,
         'official run (seed 46)': {name: 100 * official.loc[groups64, key].mean() for key, name in TABLE_KEYS.items()}}
stds = {}
for mode in ['paper', 'code']:
    seeds = [s for s in SEEDS if mode in coverage.index and s in coverage.columns and coverage.loc[mode, s] == 64]
    if not seeds:
        continue
    per_seed = 100 * grid[(grid['mode'] == mode) & grid['seed'].isin(seeds)].groupby('seed')[list(TABLE_KEYS)].mean()
    label = f'reimpl {mode} ({len(seeds)} seed{"s" if len(seeds) > 1 else ""})'
    means[label] = per_seed.mean().rename(TABLE_KEYS).to_dict()
    stds[label] = per_seed.std().rename(TABLE_KEYS).to_dict()
summary = pd.DataFrame(means).T[list(TABLE_KEYS.values())]
summary.to_csv(REPORTS / 'reimpl_tsbad64_summary.csv')
display(summary.round(2))
if stds:
    print('Standard deviation over seeds:')
    display(pd.DataFrame(stds).T[list(TABLE_KEYS.values())].round(2))

family = manifest64.set_index('group')['family']
by_family = (100 * grid.assign(family=grid['group'].map(family)).groupby(['mode', 'family'])['VUS_PR'].mean()).unstack()
by_family.loc['official run'] = 100 * official.loc[groups64, 'VUS_PR'].groupby(family.loc[groups64]).mean()
by_family.to_csv(REPORTS / 'reimpl_tsbad64_by_family.csv')
print('VUS-PR by family (reimplementation rows: all finished runs, all seeds):')
display(by_family.round(2))

Series finished per mode (rows) and seed (columns), out of 64:


seed,46,47,48
mode,,,
code,64,64,64
paper,64,64,64


,F1,AUC-PR,R-AUC-PR,VUS-PR
paper (Table 1),49.13,43.66,40.50,39.83
official run (seed 46),45.99,40.43,42.73,41.70
reimpl paper (3 seeds),45.99,39.81,39.63,38.72
reimpl code (3 seeds),45.62,39.87,42.80,41.69


Standard deviation over seeds:


,F1,AUC-PR,R-AUC-PR,VUS-PR
reimpl paper (3 seeds),0.46,0.62,1.06,1.00
reimpl code (3 seeds),0.69,1.24,0.91,0.96


VUS-PR by family (reimplementation rows: all finished runs, all seeds):


family,IOPS,MGAB,SED,SVDB,TODS
mode,,,,,
code,31.48,44.73,91.97,54.74,23.99
paper,28.62,18.44,88.30,58.97,25.44
official run,32.50,41.71,93.19,56.09,22.62


### 8.4. Table 1 view: 7 baselines, CoAD reproduced, CoAD reimplemented

Same 64 series and evaluator as the reproduction. The baselines are not reimplemented: their rows are read from
`results/tsbad64_results*.csv` (`CoAD_experiments.ipynb`), as is the official CoAD run. The reimplementation rows use
seed 46, like the reproduction runs. A method enters only with all 64 series. Ranks by VUS-PR; the paper columns
repeat the TSB-AD column of Table 1 (the CoAD value for the CoAD rows). Saved to `reports/table1_with_reimpl.csv`.

In [23]:
PAPER_TABLE1 = {   # Table 1, TSB-AD column (%): F1, AUC-PR, R-AUC-PR, VUS-PR
    'CoAD': (49.13, 43.66, 40.50, 39.83), 'Kmeans': (41.42, 37.26, 37.64, 36.81),
    'KshapeAD': (35.51, 32.67, 31.87, 31.27), 'SAND': (34.74, 31.52, 31.85, 31.05),
    'Series2Graph': (33.48, 30.11, 30.13, 29.57), 'MP': (35.10, 27.93, 29.88, 28.94),
    'PCA': (32.49, 27.59, 23.88, 23.85), 'damp': (17.44, 11.59, 12.98, 12.60)}
names = list(TABLE_KEYS.values())

reproduced = (official_rows.query("status == 'success'").sort_values('run_id')
              .drop_duplicates(['method', 'group'], keep='last'))
reproduced = reproduced[reproduced['group'].isin(groups64)]
rows, paper_method = {}, {}
for method, values in reproduced.groupby('method'):
    if values['group'].nunique() == 64:
        label = 'CoAD (reproduce)' if method == 'CoAD' else method
        rows[label] = (100 * values[list(TABLE_KEYS)].mean()).rename(TABLE_KEYS)
        paper_method[label] = method
    else:
        print(f"{method}: only {values['group'].nunique()}/64 series, left out")
for mode in ['paper', 'code']:
    values = grid[(grid['mode'] == mode) & (grid['seed'] == SEEDS[0])]
    if values['group'].nunique() == 64:
        label = f'CoAD reimpl {mode} (seed {SEEDS[0]})'
        rows[label] = (100 * values[list(TABLE_KEYS)].mean()).rename(TABLE_KEYS)
        paper_method[label] = 'CoAD'
    else:
        print(f"CoAD reimpl {mode}: only {values['group'].nunique()}/64 series for seed {SEEDS[0]}, left out")

table1 = pd.DataFrame(rows).T[names]
for i, name in enumerate(names):
    table1[f'{name} (paper)'] = [PAPER_TABLE1[paper_method[label]][i] for label in table1.index]
table1['rank'] = table1['VUS-PR'].rank(ascending=False, method='min').astype(int)
table1 = table1.sort_values('VUS-PR', ascending=False)
table1.to_csv(REPORTS / 'table1_with_reimpl.csv')
display(table1.round(2))

,F1,AUC-PR,R-AUC-PR,VUS-PR,F1 (paper),AUC-PR (paper),R-AUC-PR (paper),VUS-PR (paper),rank
CoAD (reproduce),45.99,40.43,42.73,41.70,49.13,43.66,40.50,39.83,1
Kmeans,45.18,41.06,41.95,41.01,41.42,37.26,37.64,36.81,2
CoAD reimpl code (seed 46),44.84,38.50,41.81,40.61,49.13,43.66,40.50,39.83,3
CoAD reimpl paper (seed 46),45.48,39.22,38.45,37.60,49.13,43.66,40.50,39.83,4
KshapeAD,38.01,36.12,36.65,35.86,35.51,32.67,31.87,31.27,5
Series2Graph,36.10,33.07,32.03,31.44,33.48,30.11,30.13,29.57,6
SAND,34.72,31.54,31.82,31.06,34.74,31.52,31.85,31.05,7
MP,34.94,28.22,30.47,29.48,35.10,27.93,29.88,28.94,8
PCA,32.48,27.72,23.89,23.92,32.49,27.59,23.88,23.85,9
damp,18.44,12.14,13.29,12.99,17.44,11.59,12.98,12.60,10


## 9. M5: statistical comparison with the baselines

No new runs. Every method on the same 64 TSB-AD series, scored by the same evaluator:
the 7 baselines and CoAD from the reproduction (`results/tsbad64_results*.csv`, one run each), plus the
reimplementation (code and paper mode, mean over seeds 46-48). Needs cells 0.1, 0.2, 1.1, 1.2 and 7.1.

* 9.1 builds the series × method matrix of each metric.
* 9.2 average ranks per series and the Friedman test.
* 9.3 CoAD against each baseline, series by series: win/loss, mean difference with a bootstrap 95% interval,
  Wilcoxon signed-rank test with Holm correction for the 7 comparisons.
* 9.4 results by family.

Tables are saved to `reports/m5_*.csv`.

### 9.1. Series × method matrices

In [24]:
from scipy.stats import friedmanchisquare, wilcoxon

METRICS4 = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
BASELINES = ['Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
groups64 = manifest64['group'].tolist()
family_of = manifest64.set_index('group')['family']

reproduced = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))], ignore_index=True)
reproduced = (reproduced.query("status == 'success'").sort_values('run_id')
              .drop_duplicates(['method', 'group'], keep='last'))
reimpl = (read_reimpl_results().query("status == 'success'")
          .drop_duplicates(['mode', 'group', 'seed'], keep='last'))

def metric_matrix(key):
    """64 series x methods, values x100."""
    table = reproduced.pivot(index='group', columns='method', values=key)
    for mode in ['code', 'paper']:
        table[f'CoAD reimpl {mode}'] = reimpl[reimpl['mode'] == mode].groupby('group')[key].mean()
    table = 100 * table.reindex(groups64)[['CoAD', 'CoAD reimpl code', 'CoAD reimpl paper'] + BASELINES]
    assert table.notna().all().all(), f'missing values for {key}: complete the runs first'
    return table

MATRIX = {name: metric_matrix(key) for key, name in METRICS4.items()}
print('64 series x', MATRIX['VUS-PR'].shape[1], 'methods | reimplementation seeds per mode:',
      reimpl.groupby('mode')['seed'].nunique().to_dict())
display(pd.DataFrame({name: m.mean() for name, m in MATRIX.items()}).round(2))

64 series x 10 methods | reimplementation seeds per mode: {'code': 3, 'paper': 3}


,F1,AUC-PR,R-AUC-PR,VUS-PR
method,,,,
CoAD,45.99,40.43,42.73,41.70
CoAD reimpl code,45.62,39.87,42.80,41.69
CoAD reimpl paper,45.99,39.81,39.63,38.72
Kmeans,45.18,41.06,41.95,41.01
KshapeAD,38.01,36.12,36.65,35.86
SAND,34.72,31.54,31.82,31.06
Series2Graph,36.10,33.07,32.03,31.44
MP,34.94,28.22,30.47,29.48
PCA,32.48,27.72,23.89,23.92


### 9.2. Average rank and Friedman test

Rank 1 = best on a series (ties share the average rank). The Friedman test asks whether the 8 methods of Table 1
(CoAD from the reproduction + 7 baselines) differ at all. The reimplementation is left out of the ranking and the
test, since it is the same method as CoAD.

In [25]:
TABLE1_METHODS = ['CoAD'] + BASELINES
ranks = {}
for name, matrix in MATRIX.items():
    ranks[name] = matrix[TABLE1_METHODS].rank(axis=1, ascending=False).mean()
    stat, p = friedmanchisquare(*[matrix[m] for m in TABLE1_METHODS])
    print(f'{name:8} Friedman chi2 = {stat:6.1f}, p = {p:.1e}')
rank_table = pd.DataFrame(ranks).sort_values('VUS-PR')
rank_table.to_csv(REPORTS / 'm5_average_ranks.csv')
print('Average rank among the 8 methods of Table 1 (lower is better):')
display(rank_table.round(2))

ax = rank_table['VUS-PR'].sort_values(ascending=False).plot.barh(figsize=(6, 3.5), color='C0')
ax.set_xlabel('average rank on 64 series, VUS-PR (lower is better)')
ax.figure.savefig(REPORTS / 'm5_average_rank_vus.png', dpi=120, bbox_inches='tight')
plt.show()

F1       Friedman chi2 =   89.1, p = 1.9e-16
AUC-PR   Friedman chi2 =   89.9, p = 1.3e-16
R-AUC-PR Friedman chi2 =  103.7, p = 1.9e-19
VUS-PR   Friedman chi2 =  101.8, p = 4.7e-19
Average rank among the 8 methods of Table 1 (lower is better):


,F1,AUC-PR,R-AUC-PR,VUS-PR
method,,,,
Kmeans,2.82,2.77,2.59,2.61
KshapeAD,3.82,3.78,3.67,3.70
CoAD,3.75,3.81,3.84,3.81
SAND,4.48,4.48,4.44,4.50
Series2Graph,4.70,4.60,4.67,4.66
MP,4.89,4.92,4.78,4.78
PCA,5.03,5.17,5.58,5.50
damp,6.51,6.45,6.42,6.44


### 9.3. CoAD against each baseline, series by series

For each metric and baseline: number of series where CoAD is better / worse, mean difference (CoAD − baseline)
with a bootstrap 95% interval over series (10 000 resamples, `default_rng(2026)`), Wilcoxon signed-rank p-value and
the Holm-adjusted p-value over the 7 baselines. Done for CoAD of the reproduction and, as a robustness check,
for the 3-seed mean of the reimplementation in code mode.

In [26]:
def holm(pvalues):
    """Holm step-down adjustment."""
    pvalues = np.asarray(pvalues)
    adjusted, running = np.empty(len(pvalues)), 0.0
    for i, idx in enumerate(np.argsort(pvalues)):
        running = max(running, (len(pvalues) - i) * pvalues[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def bootstrap_ci(diff, n=10_000, seed=2026):
    """95% interval of the mean difference, resampling series with replacement."""
    rng = np.random.default_rng(seed)
    means = rng.choice(diff, size=(n, len(diff)), replace=True).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

pairwise = []
for coad in ['CoAD', 'CoAD reimpl code']:
    for name, matrix in MATRIX.items():
        rows = []
        for baseline in BASELINES:
            diff = (matrix[coad] - matrix[baseline]).to_numpy()
            low, high = bootstrap_ci(diff)
            rows.append({'CoAD': coad, 'metric': name, 'baseline': baseline,
                         'CoAD better': int((diff > 0).sum()), 'CoAD worse': int((diff < 0).sum()),
                         'mean diff': diff.mean(), 'CI low': low, 'CI high': high,
                         'p': wilcoxon(diff).pvalue})
        for row, p_holm in zip(rows, holm([r['p'] for r in rows])):
            row['p Holm'] = p_holm
            row['significant'] = p_holm < 0.05
        pairwise += rows
pairwise = pd.DataFrame(pairwise)
pairwise.to_csv(REPORTS / 'm5_pairwise_tests.csv', index=False)
display(pairwise[pairwise['metric'] == 'VUS-PR'].round(3))
print('Not significant after Holm (all metrics):')
display(pairwise[~pairwise['significant']][['CoAD', 'metric', 'baseline', 'CoAD better', 'CoAD worse',
                                            'mean diff', 'p Holm']].round(3))

,CoAD,metric,baseline,CoAD better,CoAD worse,mean diff,CI low,CI high,p,p Holm,significant
21,CoAD,VUS-PR,Kmeans,25,39,0.687,-5.305,7.191,0.231,0.463,False
22,CoAD,VUS-PR,KshapeAD,33,31,5.839,-1.663,13.449,0.313,0.463,False
23,CoAD,VUS-PR,SAND,36,28,10.639,3.880,17.713,0.015,0.060,False
24,CoAD,VUS-PR,Series2Graph,37,27,10.262,2.823,18.016,0.039,0.118,False
25,CoAD,VUS-PR,MP,40,24,12.215,5.846,18.818,0.001,0.003,True
26,CoAD,VUS-PR,PCA,48,16,17.773,11.302,24.814,0.000,0.000,True
27,CoAD,VUS-PR,damp,49,15,28.706,22.089,35.607,0.000,0.000,True
49,CoAD reimpl code,VUS-PR,Kmeans,22,42,0.680,-5.453,7.221,0.276,0.551,False
50,CoAD reimpl code,VUS-PR,KshapeAD,31,33,5.832,-1.896,13.675,0.370,0.551,False
51,CoAD reimpl code,VUS-PR,SAND,37,27,10.632,3.897,17.734,0.015,0.059,False


Not significant after Holm (all metrics):


,CoAD,metric,baseline,CoAD better,CoAD worse,mean diff,p Holm
0,CoAD,F1,Kmeans,29,35,0.811,0.698
1,CoAD,F1,KshapeAD,34,30,7.976,0.226
3,CoAD,F1,Series2Graph,37,27,9.886,0.070
7,CoAD,AUC-PR,Kmeans,28,36,-0.630,0.733
8,CoAD,AUC-PR,KshapeAD,33,31,4.312,0.733
9,CoAD,AUC-PR,SAND,37,27,8.891,0.092
10,CoAD,AUC-PR,Series2Graph,35,29,7.363,0.284
14,CoAD,R-AUC-PR,Kmeans,24,40,0.772,0.417
15,CoAD,R-AUC-PR,KshapeAD,32,32,6.080,0.417
16,CoAD,R-AUC-PR,SAND,36,28,10.908,0.070


### 9.4. By family

Mean VUS-PR per family (number of series in brackets) and the best of the 8 Table 1 methods in each family.

In [27]:
families = family_of.reindex(groups64)
counts = families.value_counts()
by_family = MATRIX['VUS-PR'].groupby(families.values).mean().T
by_family.columns = [f'{f} ({counts[f]})' for f in by_family.columns]
by_family.to_csv(REPORTS / 'm5_vus_by_family.csv')
display(by_family.round(2))
best = MATRIX['VUS-PR'][TABLE1_METHODS].groupby(families.values).mean().idxmax(axis=1)
print('Best Table 1 method per family (VUS-PR):', best.to_dict())

,IOPS (17),MGAB (9),SED (3),SVDB (20),TODS (15)
method,,,,,
CoAD,32.50,41.71,93.19,56.09,22.62
CoAD reimpl code,31.48,44.73,91.97,54.74,23.99
CoAD reimpl paper,28.62,18.44,88.30,58.97,25.44
Kmeans,32.52,0.99,92.47,62.52,35.67
KshapeAD,3.39,0.48,87.26,70.88,36.91
SAND,4.04,0.46,90.47,54.45,36.98
Series2Graph,19.66,0.43,14.90,54.53,35.89
MP,4.02,34.50,73.08,37.47,35.97
PCA,21.64,0.40,3.26,44.93,16.75


Best Table 1 method per family (VUS-PR): {'IOPS': 'Kmeans', 'MGAB': 'CoAD', 'SED': 'CoAD', 'SVDB': 'KshapeAD', 'TODS': 'SAND'}


## 10. M6-A: does the score smoothing explain the gap to the paper?

**Question.** Against Table 1, CoAD is lower on the point-wise metrics (F1, AUC-PR) and higher on the range-wise
metrics (R-AUC-PR, VUS-PR), by more than the seed-to-seed spread. Commit `8765811` (9 April 2026) changed exactly
one line for TSB-AD in `train_classify_mask.py`: before it, `average_window = None` (no moving average of the
score); after it, `average_window = subsequence_length`. The rest of that commit (data path, requirements, the UCR
runner) does not touch TSB-AD.

**Prediction, written before running.** Without the moving average, F1 and AUC-PR go up and R-AUC-PR and VUS-PR go
down, all towards the Table 1 values.

**Method.** No retraining. The 192 code-mode models of section 7 (64 series × 3 seeds; code mode = authors' code)
are reloaded from `runs/reimpl/*/model.pt` and their test score recomputed with the moving average switched off,
everything else unchanged. The smoothed score is recomputed as well and compared with the saved `scores.npy`, as a
check that the reload is exact. One row per model goes to `results/m6a_smoothing_<SESSION_TAG>.csv`; re-run 10.1
to continue after a disconnect. Needs cells 0.1-1.2, 3.1, 4.1, 5.1, 6.1 and 7.1, and a GPU runtime.

### 10.1. Rescore without smoothing

In [28]:
M6A_RESULTS = RESULTS_DIR / f'm6a_smoothing_{SESSION_TAG}.csv'
M6A_COLUMNS = ['run_id', 'group', 'seed', 'max_score_diff', 'seconds'] + METRIC_KEYS

def code_mode_scores(model, series):
    """Code-mode score with the moving average (authors' code after commit 8765811) and without it (before)."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    train_prob, train_rec = predict_series(model, series['train'], window, period)
    recon = np.abs(np.abs(series['test'] - test_rec) - np.abs(series['train'] - train_rec).mean())
    classify = np.abs(test_prob - train_prob.mean())
    scores = {}
    for variant, smooth in [('smoothed', True), ('unsmoothed', False)]:
        r = moving_average(recon, period) if smooth else recon.copy()
        c = moving_average(classify, period) if smooth else classify.copy()
        r[:period] = c[:period] = 0
        scores[variant] = (r + c) / 2
    return scores

def read_m6a():
    """Rescored runs of every session, one row per (group, seed)."""
    files = sorted(RESULTS_DIR.glob('m6a_smoothing*.csv'))
    if not files:
        return pd.DataFrame(columns=M6A_COLUMNS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.drop_duplicates(['group', 'seed'], keep='last')

code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))
rescored = read_m6a()
done = set(zip(rescored['group'].astype(int), rescored['seed'].astype(int)))
todo = code_runs[[(int(g), int(s)) not in done for g, s in zip(code_runs['group'], code_runs['seed'])]]
todo = todo.sort_values('seed', kind='stable')
print(len(code_runs), 'code-mode runs,', len(todo), 'still to rescore')

session_start = time.time()
for _, run in todo.iterrows():
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    series = load_series(int(run['group']))
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run['run_id'] / 'model.pt', map_location=DEVICE))
    scores = code_mode_scores(model, series)
    gap = float(np.abs(scores['smoothed'] - np.load(RUNS / run['run_id'] / 'scores.npy')).max())
    row = dict(run_id=run['run_id'], group=int(run['group']), seed=int(run['seed']), max_score_diff=gap,
               **score_metrics(scores['unsmoothed'], series))
    row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame([row]).reindex(columns=M6A_COLUMNS).to_csv(M6A_RESULTS, mode='a', header=not M6A_RESULTS.exists(),
                                                            index=False)
    print(f"group {row['group']} seed {row['seed']}: {row['seconds']:.0f} s | reload gap {gap:.1e} | unsmoothed "
          + ' '.join(f"{n} {100 * row[k]:.2f}" for k, n in METRICS4.items()), flush=True)

192 code-mode runs, 0 still to rescore


### 10.2. Smoothed vs unsmoothed

Smoothed = the code-mode results of section 7; unsmoothed = 10.1. Each seed is averaged over the series rescored so
far, then mean ± SD over seeds. `Δ to paper` is the distance to Table 1. The prediction holds for a metric if
switching the smoothing off moves it in the predicted direction **and** closer to Table 1. The paired test compares
smoothed and unsmoothed series by series (means over seeds).

In [29]:
rescored = read_m6a()
print('reload check: largest |recomputed smoothed score - saved scores.npy| =', f"{rescored['max_score_diff'].max():.1e}")
smoothed = code_runs.astype({'group': int, 'seed': int}).set_index(['group', 'seed'])[list(METRICS4)]
unsmoothed = rescored.astype({'group': int, 'seed': int}).set_index(['group', 'seed'])[list(METRICS4)]
pairs = smoothed.index.intersection(unsmoothed.index)
print(len(pairs), 'of', len(smoothed), 'models rescored')

TABLE1 = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}
EXPECTED = {'F1': 'up', 'AUC-PR': 'up', 'R-AUC-PR': 'down', 'VUS-PR': 'down'}
variants = {'smoothed': smoothed.loc[pairs], 'unsmoothed': unsmoothed.loc[pairs]}
rows = []
for key, name in METRICS4.items():
    per_seed = {v: 100 * f[key].groupby(level='seed').mean() for v, f in variants.items()}
    per_series = {v: 100 * f[key].groupby(level='group').mean() for v, f in variants.items()}
    s, u = per_seed['smoothed'].mean(), per_seed['unsmoothed'].mean()
    moved = 'up' if u > s else 'down'
    rows.append({'metric': name, 'Table 1': TABLE1[name],
                 'smoothed': s, 'smoothed SD': per_seed['smoothed'].std(),
                 'unsmoothed': u, 'unsmoothed SD': per_seed['unsmoothed'].std(),
                 'Δ to paper, smoothed': s - TABLE1[name], 'Δ to paper, unsmoothed': u - TABLE1[name],
                 'unsmoothed better on': f"{int((per_series['unsmoothed'] > per_series['smoothed']).sum())}/{len(per_series['smoothed'])}",
                 'p (Wilcoxon)': wilcoxon(per_series['unsmoothed'] - per_series['smoothed']).pvalue,
                 'prediction': 'holds' if moved == EXPECTED[name] and abs(u - TABLE1[name]) < abs(s - TABLE1[name]) else 'fails'})
m6a_table = pd.DataFrame(rows).set_index('metric')
m6a_table.to_csv(REPORTS / 'm6a_smoothing_summary.csv')
display(m6a_table.round(3))

families_of_pairs = family_of.reindex(pairs.get_level_values('group')).values
fam = pd.DataFrame({v: 100 * f['VUS_PR'].groupby(families_of_pairs).mean() for v, f in variants.items()})
fam['change'] = fam['unsmoothed'] - fam['smoothed']
print('VUS-PR by family:')
display(fam.round(2))

reload check: largest |recomputed smoothed score - saved scores.npy| = 3.8e-06
192 of 192 models rescored


,Table 1,smoothed,smoothed SD,unsmoothed,unsmoothed SD,"Δ to paper, smoothed","Δ to paper, unsmoothed",unsmoothed better on,p (Wilcoxon),prediction
metric,,,,,,,,,,
F1,49.13,45.618,0.694,43.802,2.211,-3.512,-5.328,27/64,0.103,fails
AUC-PR,43.66,39.866,1.241,38.308,2.722,-3.794,-5.352,27/64,0.101,fails
R-AUC-PR,40.50,42.803,0.907,37.726,1.937,2.303,-2.774,23/64,0.004,fails
VUS-PR,39.83,41.691,0.956,36.849,1.922,1.861,-2.981,23/64,0.004,fails


VUS-PR by family:


,smoothed,unsmoothed,change
IOPS,31.48,25.98,-5.50
MGAB,44.73,36.67,-8.06
SED,91.97,94.45,2.48
SVDB,54.74,41.72,-13.01
TODS,23.99,31.25,7.26


## 11. M7: case studies

**Questions.** Why does CoAD win on MGAB and lose on TODS? Why does paper mode collapse on MGAB? Why does the
moving average hurt CoAD on TODS (M6-A)? Which of the two branches, the classifier (`A_c`) or the reconstruction
(|X_r − X|), carries the score in each case?

**Method.** No training.

* 11.1 picks one series per question with a rule fixed in advance, from the results of sections 7, 9 and 10.
* 11.2 reloads the seed-46 models of both modes for these series and splits their score into the two branches
  (each branch alone, as it enters the final score), then scores every variant with the same evaluator. The
  official CoAD score and the baseline scores are read from the reproduction runs (`runs/tsbad64/`).
* 11.3 plots the scores, whole series and zoomed on one anomaly.
* 11.4 (optional, about 40 min on GPU) repeats the branch split on all 64 series, seed 46, both modes, so the
  conclusions do not rest on 4 series only.

Needs cells 0.1-1.2, 3.1, 4.1, 5.1, 6.1, 7.1 and 9.1, and the M6-A results of section 10 for 11.1.
11.2 and 11.3 run on CPU in a few minutes; a GPU is faster.

### 11.1. Choice of the series

In [30]:
vus = MATRIX['VUS-PR']
families = family_of.reindex(groups64)
best_baseline = vus[BASELINES].idxmax(axis=1)
m6a = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('m6a_smoothing*.csv'))], ignore_index=True)
m6a = m6a.drop_duplicates(['group', 'seed'], keep='last')
smoothing_effect = 100 * (m6a.groupby('group')['VUS_PR'].mean()
                          - reimpl[reimpl['mode'] == 'code'].groupby('group')['VUS_PR'].mean())

CASE_RULES = {   # case: (family, value to maximise, rule)
    'mgab_win': ('MGAB', vus['CoAD'] - vus['Kmeans'], 'CoAD - Kmeans'),
    'tods_loss': ('TODS', vus[BASELINES].max(axis=1) - vus['CoAD'], 'best baseline - CoAD'),
    'mgab_paper_drop': ('MGAB', vus['CoAD reimpl code'] - vus['CoAD reimpl paper'], 'code mode - paper mode'),
    'tods_smoothing': ('TODS', smoothing_effect.reindex(groups64), 'unsmoothed - smoothed (code mode)'),
}
CASES, rows = {}, []
for case, (family, value, rule) in CASE_RULES.items():
    group = int(value.loc[families.index[families == family]].idxmax())
    CASES[case] = group
    rows.append({'case': case, 'family': family, 'rule (largest)': rule, 'group': group, 'value': value[group],
                 'CoAD': vus.loc[group, 'CoAD'], 'code': vus.loc[group, 'CoAD reimpl code'],
                 'paper': vus.loc[group, 'CoAD reimpl paper'], 'Kmeans': vus.loc[group, 'Kmeans'],
                 'best baseline': best_baseline[group], 'best baseline VUS-PR': vus.loc[group, best_baseline[group]]})
case_table = pd.DataFrame(rows).set_index('case')
case_table.to_csv(REPORTS / 'm7_cases.csv')
print('VUS-PR x100; code / paper = reimplementation, mean of 3 seeds')
display(case_table.round(2))

VUS-PR x100; code / paper = reimplementation, mean of 3 seeds


,family,rule (largest),group,value,CoAD,code,paper,Kmeans,best baseline,best baseline VUS-PR
case,,,,,,,,,,
mgab_win,MGAB,CoAD - Kmeans,233,76.70,78.05,72.41,26.39,1.36,MP,71.42
tods_loss,TODS,best baseline - CoAD,295,66.45,16.88,19.01,17.01,72.99,Series2Graph,83.34
mgab_paper_drop,MGAB,code mode - paper mode,233,46.02,78.05,72.41,26.39,1.36,MP,71.42
tods_smoothing,TODS,unsmoothed - smoothed (code mode),288,20.65,20.83,22.35,24.32,26.30,Series2Graph,30.32


### 11.2. Branch split and metrics

For each case, scores of: CoAD official (reproduction run), Kmeans, the best baseline of that series, and the
seed-46 models of both modes, each split into `full`, `classifier only` and `reconstruction only`. In code mode the
branches are taken after the train-mean subtraction, the moving average and the zeroed first period, i.e. exactly
the two terms averaged by the authors' code; `unsmoothed` variants skip the moving average (as in M6-A). In paper
mode the two terms of Eq. 11, each smoothed.

Check: the recomputed `full` score of each model must match its saved `scores.npy`.

The first cell only defines the helpers (also used by sections 11.4 and 13); the second runs the 4 cases.

In [ ]:
TSBAD_RUNS = STORE / 'runs' / 'tsbad64'
CASE_SEED = 46
reproduced_runs = reproduced.set_index(['method', 'group'])['run_id']

def reproduced_score(method, group):
    """Test score saved by the reproduction run of a method (official CoAD or a baseline)."""
    artifacts = TSBAD_RUNS / reproduced_runs[(method, group)] / 'artifacts'
    pattern = '*test_anomaly_score.npy' if method == 'CoAD' else f'group_{str(group).zfill(3)}.npy'
    files = sorted(artifacts.rglob(pattern))
    assert files, f'no score file for {method}, group {group}, in {artifacts}'
    return np.load(files[0]).reshape(-1).astype(float)

def load_reimpl_model(mode, group, seed):
    run_id = reimpl.query('mode == @mode and group == @group and seed == @seed')['run_id'].iloc[-1]
    model = CoADReimpl(faithful=(mode == 'code')).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run_id / 'model.pt', map_location=DEVICE))
    return model, run_id

def branch_scores(model, series):
    """Final score of the model and each of its two branches alone."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    test_error = np.abs(series['test'] - test_rec)
    if not model.faithful:   # Eq. 11: moving average of A_c + |X_r - X|
        classify, recon = moving_average(test_prob, period), moving_average(test_error, period)
        return {'full': classify + recon, 'classifier only': classify, 'reconstruction only': recon}
    train_prob, train_rec = predict_series(model, series['train'], window, period)
    recon_raw = np.abs(test_error - np.abs(series['train'] - train_rec).mean())
    classify_raw = np.abs(test_prob - train_prob.mean())
    scores = {}
    for suffix, smooth in [('', True), (' unsmoothed', False)]:
        r = moving_average(recon_raw, period) if smooth else recon_raw.copy()
        c = moving_average(classify_raw, period) if smooth else classify_raw.copy()
        r[:period] = c[:period] = 0
        scores.update({'full' + suffix: (r + c) / 2, 'classifier only' + suffix: c,
                       'reconstruction only' + suffix: r})
    return scores

def case_detail(group):
    """Series, all scores and their metrics for one case series."""
    series = load_series(group)
    scores = {'CoAD official': reproduced_score('CoAD', group)}
    for baseline in dict.fromkeys(['Kmeans', best_baseline[group]]):
        scores[baseline] = reproduced_score(baseline, group)
    for mode in ['code', 'paper']:
        model, run_id = load_reimpl_model(mode, group, CASE_SEED)
        parts = branch_scores(model, series)
        gap = float(np.abs(parts['full'] - np.load(RUNS / run_id / 'scores.npy')).max())
        assert gap < 1e-4, f'{run_id}: recomputed score differs from scores.npy by {gap:.1e}'
        scores.update({f'{mode}: {name}': values for name, values in parts.items()})
    rows = []
    for name in list(scores):
        if len(scores[name]) != len(series['label']):
            print(f'group {group}, {name}: score length {len(scores[name])} != {len(series["label"])}, skipped')
            del scores[name]
            continue
        metrics = score_metrics(scores[name], series)
        rows.append({'score': name, **{label: 100 * metrics[key] for key, label in METRICS4.items()}})
    return series, scores, rows

Run the 4 cases (a few minutes):

In [32]:
details, case_scores, rows = {}, {}, []
for case, group in CASES.items():
    if group not in details:   # two cases may pick the same series
        details[group] = case_detail(group)
    series, scores, metric_rows = details[group]
    case_scores[case] = (series, scores)
    rows += [{'case': case, 'group': group, **row} for row in metric_rows]
    print(f'{case}: group {group} done', flush=True)

case_metrics = pd.DataFrame(rows)
case_metrics.to_csv(REPORTS / 'm7_case_metrics.csv', index=False)
for case, table in case_metrics.groupby('case', sort=False):
    print(f'\n{case} (group {CASES[case]}, {family_of[CASES[case]]})')
    display(table.drop(columns=['case', 'group']).set_index('score').round(2))

mgab_win: group 233 done
tods_loss: group 295 done
mgab_paper_drop: group 233 done
tods_smoothing: group 288 done

mgab_win (group 233, MGAB)


,F1,AUC-PR,R-AUC-PR,VUS-PR
score,,,,
CoAD official,55.30,52.61,80.23,78.05
Kmeans,18.02,10.03,1.38,1.36
MP,64.68,55.05,78.48,71.42
code: full,57.20,44.59,72.12,66.13
code: classifier only,56.97,43.36,71.13,64.15
code: reconstruction only,49.15,35.86,46.34,45.12
code: full unsmoothed,63.87,52.78,62.05,56.27
code: classifier only unsmoothed,64.27,56.31,63.13,61.18
code: reconstruction only unsmoothed,21.25,7.09,18.47,15.53



tods_loss (group 295, TODS)


,F1,AUC-PR,R-AUC-PR,VUS-PR
score,,,,
CoAD official,24.88,9.52,17.37,16.88
Kmeans,59.76,46.95,76.76,72.99
Series2Graph,68.96,70.27,86.42,83.34
code: full,26.09,12.50,23.71,23.00
code: classifier only,25.73,10.26,17.57,17.44
code: reconstruction only,32.74,13.94,26.59,25.93
code: full unsmoothed,24.37,11.11,18.71,18.57
code: classifier only unsmoothed,24.34,10.00,16.74,16.65
code: reconstruction only unsmoothed,20.61,9.44,17.79,17.76



mgab_paper_drop (group 233, MGAB)


,F1,AUC-PR,R-AUC-PR,VUS-PR
score,,,,
CoAD official,55.30,52.61,80.23,78.05
Kmeans,18.02,10.03,1.38,1.36
MP,64.68,55.05,78.48,71.42
code: full,57.20,44.59,72.12,66.13
code: classifier only,56.97,43.36,71.13,64.15
code: reconstruction only,49.15,35.86,46.34,45.12
code: full unsmoothed,63.87,52.78,62.05,56.27
code: classifier only unsmoothed,64.27,56.31,63.13,61.18
code: reconstruction only unsmoothed,21.25,7.09,18.47,15.53



tods_smoothing (group 288, TODS)


,F1,AUC-PR,R-AUC-PR,VUS-PR
score,,,,
CoAD official,73.76,67.47,21.13,20.83
Kmeans,74.17,70.44,26.94,26.30
Series2Graph,73.53,68.60,31.79,30.32
code: full,71.15,65.57,21.05,20.77
code: classifier only,72.43,64.82,17.84,17.43
code: reconstruction only,68.02,61.07,21.80,21.46
code: full unsmoothed,77.48,71.99,34.10,33.87
code: classifier only unsmoothed,72.49,65.27,21.97,21.31
code: reconstruction only unsmoothed,72.23,63.04,32.63,31.82


### 11.3. Figures

Each score is min-max scaled to [0, 1]; labelled anomalies are shaded. First the whole test series, then a zoom of
±3 windows around the labelled anomaly where the code-mode score is weakest (lowest mean score inside the anomaly).
Saved to `reports/m7_<case>_<group>*.png`.

In [33]:
SHOW = ['CoAD official', 'Kmeans', 'code: full', 'code: classifier only', 'code: reconstruction only',
        'paper: full', 'paper: classifier only', 'paper: reconstruction only']

def anomaly_segments(label):
    edges = np.diff(np.concatenate([[0], label.astype(int), [0]]))
    return list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))

for case, (series, scores) in case_scores.items():
    group = series['group']
    show = [n for n in SHOW if n in scores]
    extra = [n for n in scores if n not in SHOW and 'unsmoothed' not in n and ':' not in n]   # best baseline
    if case == 'tods_smoothing':
        show += ['code: full unsmoothed']
    chosen = {n: scores[n] for n in show[:1] + extra + show[1:]}
    print(f'{case}: group {group} ({family_of[group]})')
    plot_scores(series, chosen, REPORTS / f'm7_{case}_{group}.png')

    segments = anomaly_segments(series['label'])
    code_full = scores['code: full']
    start, end = min(segments, key=lambda seg: code_full[seg[0]:seg[1]].mean())
    margin = 3 * series['window']
    a, b = max(0, start - margin), min(len(series['label']), end + margin)
    zoom = dict(series, group=f"{group}, zoom {a}-{b}", test=series['test'][a:b], label=series['label'][a:b])
    plot_scores(zoom, {n: v[a:b] for n, v in chosen.items()}, REPORTS / f'm7_{case}_{group}_zoom.png')

mgab_win: group 233 (MGAB)
tods_loss: group 295 (TODS)
mgab_paper_drop: group 233 (MGAB)
tods_smoothing: group 288 (TODS)


### 11.4. Optional: branch split on all 64 series

Same split as 11.2 for the seed-46 models of both modes on all 64 series: 128 models, about 40 min on a T4. Only the
branch-only variants are scored (the full score is already in section 7). Rows go to
`results/m7_branches_<SESSION_TAG>.csv`; re-run to continue after a disconnect.

In [34]:
M7_RESULTS = RESULTS_DIR / f'm7_branches_{SESSION_TAG}.csv'
BRANCHES = ['classifier only', 'reconstruction only']

def read_m7():
    files = sorted(RESULTS_DIR.glob('m7_branches*.csv'))
    if not files:
        return pd.DataFrame(columns=['run_id', 'mode', 'group', 'seed', 'branch'] + METRIC_KEYS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.drop_duplicates(['mode', 'group', 'seed', 'branch'], keep='last')

done = set(zip(read_m7()['mode'], read_m7()['group'].astype(int)))
todo = [(mode, g) for mode in ['code', 'paper'] for g in groups64 if (mode, g) not in done]
print(len(todo), 'of', 2 * len(groups64), 'models still to split')
session_start = time.time()
for mode, group in todo:
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    series = load_series(group)
    model, run_id = load_reimpl_model(mode, group, CASE_SEED)
    parts = branch_scores(model, series)
    gap = float(np.abs(parts['full'] - np.load(RUNS / run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run_id}: recomputed score differs from scores.npy by {gap:.1e}'
    rows = [dict(run_id=run_id, mode=mode, group=group, seed=CASE_SEED, branch=b, **score_metrics(parts[b], series))
            for b in BRANCHES]
    pd.DataFrame(rows).to_csv(M7_RESULTS, mode='a', header=not M7_RESULTS.exists(), index=False)
    print(f'{mode} group {group}: {time.time() - start:.0f} s | VUS-PR '
          + ' '.join(f"{r['branch']} {100 * r['VUS_PR']:.2f}" for r in rows), flush=True)

0 of 128 models still to split


### 11.5. Optional: which branch carries the score, by family

Mean VUS-PR per family of the full score (seed 46, section 7) and of each branch alone (11.4). A branch close to the
full score carries it; a branch far below adds little or hurts.

In [35]:
branches = read_m7().astype({'group': int})
full = (reimpl[reimpl['seed'] == CASE_SEED].assign(branch='full')
        [['mode', 'group', 'branch', 'VUS_PR']].astype({'group': int}))
both = pd.concat([full, branches[['mode', 'group', 'branch', 'VUS_PR']]], ignore_index=True)
complete = both.groupby(['mode', 'group'])['branch'].nunique().eq(3)
both = both.set_index(['mode', 'group']).loc[complete[complete].index].reset_index()
print(int(complete.sum()), 'of', 2 * len(groups64), 'models with all three scores')
both['family'] = family_of.reindex(both['group']).values
branch_table = 100 * both.pivot_table(index='family', columns=['mode', 'branch'], values='VUS_PR', aggfunc='mean')
branch_table.loc['all'] = 100 * both.groupby(['mode', 'branch'])['VUS_PR'].mean()
branch_table.to_csv(REPORTS / 'm7_branch_by_family.csv')
display(branch_table.round(2))

128 of 128 models with all three scores


mode              code                                      paper         \
branch classifier only   full reconstruction only classifier only   full   
family                                                                     
IOPS             22.50  31.52               33.37           21.62  28.83   
MGAB             41.16  41.29               21.99           12.19  13.08   
SED              92.28  91.49               79.96           87.13  93.40   
SVDB             50.97  53.44               63.06           53.59  57.16   
TODS             21.21  23.22               23.35           24.26  25.00   
all              36.99  40.61               40.88           33.98  37.60   

mode                        
branch reconstruction only  
family                      
IOPS                 36.60  
MGAB                 43.37  
SED                  87.88  
SVDB                 69.22  
TODS                 25.76  
all                  47.61

## 12. M6-C: which code-only details make MGAB work?

**Question.** Paper mode reaches 18.4 VUS-PR on MGAB, code mode 44.7 (section 7). M7 showed that the classifier
branch is what fails in paper mode on MGAB. Which group of code-only details (table D of the plan) is responsible?

**Method.** The differences are switched by group, on top of paper mode:

| Group | Details |
|---|---|
| `architecture` | D1 GRU layers, D2 dropout, D4 MLP heads, D5 hidden size 144, D18 SELU |
| `training` | D6 `pos_weight`, D7 BCE on A and A^r, D8 balanced MSE, D9 clean + distorted batch, D16 StepLR and fp16, D20 MSE scale |
| `distortion` | D12-D15 details of the pseudo-anomalies |
| `scoring` | D10 second reconstruction, D11 score relative to the training part |

Two designs, 4 configurations each, on the 9 MGAB series × 3 seeds:

* **add one** group to paper mode (`paper + X`): does X alone repair MGAB?
* **leave one** group out of code mode (`code - X`): is X needed?

`scoring` does not change training, so `paper + scoring` and `code - scoring` reuse the saved models of section 7
(seconds per run). The 6 other configurations train: about 2-3 min per run on a T4, about 7 h for all 162 runs.
Several notebook copies can share the work: give each its own `SESSION_TAG` (7.1) and its own `RUN_CONFIGS`.

Needs cells 0.1-1.2, 2.1, 3.1, 4.1, 5.1, 5.2, 6.1, 7.1 and 9.1.

### 12.1. `HybridCoAD` and test T8

In [ ]:
CODE_GROUPS = {
    'architecture': 'D1, D2, D4, D5, D18',
    'training': 'D6, D7, D8, D9, D16, D20',
    'distortion': 'D12, D13, D14, D15',
    'scoring': 'D10, D11',
}

class HybridCoAD(CoADReimpl):
    """Paper mode plus the code-only details of the chosen groups. All 4 groups = code mode, none = paper mode.
    The training and distortion groups are applied by train_model(groups=...), not by the model."""

    def __init__(self, groups):
        nn.Module.__init__(self)
        self.groups = frozenset(groups)
        assert self.groups <= set(CODE_GROUPS), f'unknown groups: {self.groups - set(CODE_GROUPS)}'
        self.faithful = 'scoring' in self.groups            # read by predict (D10) and anomaly_score (D11)
        self.classifier = TimeFreqClassifier('architecture' in self.groups)
        self.mae = SoftMaskMAE('architecture' in self.groups)

# T8: the two ends of the hybrid are exactly the two modes
x = torch.rand(4, 496)
s296 = load_series(296)
for groups, faithful in [(set(CODE_GROUPS), True), (set(), False)]:
    set_seed(0)
    hybrid = HybridCoAD(groups).eval()
    set_seed(0)
    reference = CoADReimpl(faithful).eval()
    assert all(torch.equal(a, b) for a, b in zip(hybrid.state_dict().values(), reference.state_dict().values()))
    assert all(torch.equal(a, b) for a, b in zip(hybrid.predict(x), reference.predict(x)))
    loss_hybrid = train_model(s296, seed=46, epochs=2, log_every=0, groups=groups)[1]
    loss_reference = train_model(s296, faithful=faithful, seed=46, epochs=2, log_every=0)[1]
    gap = float((loss_hybrid - loss_reference).abs().max().max())
    print(f"{'all groups vs code' if faithful else 'no group vs paper'} mode: same weights and outputs, "
          f"2-epoch loss difference {gap:.1e}")
    assert gap < 1e-3

# T8, intermediate configurations: each single group switches exactly its own flags
original_batch, original_loss = training_batch, coad_loss
seen = []
def training_batch(clean, period, faithful, code_distortion=None):
    seen.append(('batch', faithful, code_distortion))
    return original_batch(clean, period, faithful, code_distortion)
def coad_loss(model, x, labels, target, code_loss=None):
    seen.append(('loss', code_loss))
    return original_loss(model, x, labels, target, code_loss)
try:
    for group in CODE_GROUPS:
        seen.clear()
        model, _ = train_model(s296, seed=46, epochs=1, log_every=0, groups={group})
        shapes = [tuple(v.shape) for v in model.state_dict().values()]
        expected = [tuple(v.shape) for v in CoADReimpl(group == 'architecture').state_dict().values()]
        assert shapes == expected, f'{group}: architecture not as expected'
        assert model.faithful == (group == 'scoring'), f'{group}: scoring flag wrong'
        assert {c for c in seen if c[0] == 'batch'} == {('batch', group == 'training', group == 'distortion')}, group
        assert {c for c in seen if c[0] == 'loss'} == {('loss', group == 'training')}, group
        print(f'  {group:12s} alone: switches only its own details')
finally:
    training_batch, coad_loss = original_batch, original_loss
print('T8 passed')

### 12.2. Runs (re-run to continue)

One row per run in `results/m6c_results_<SESSION_TAG>.csv`, the score in `runs/m6c/<run_id>/scores.npy`. Reused
models (`scoring`) run first. A failing run is recorded and the loop goes on.

In [37]:
M6C_CONFIGS = {   # name: (code-only groups switched on, saved section-7 models to reuse instead of training)
    'paper + architecture': ({'architecture'}, None),
    'paper + training': ({'training'}, None),
    'paper + distortion': ({'distortion'}, None),
    'paper + scoring': ({'scoring'}, 'paper'),
    'code - architecture': (set(CODE_GROUPS) - {'architecture'}, None),
    'code - training': (set(CODE_GROUPS) - {'training'}, None),
    'code - distortion': (set(CODE_GROUPS) - {'distortion'}, None),
    'code - scoring': (set(CODE_GROUPS) - {'scoring'}, 'code'),
}
RUN_CONFIGS = list(M6C_CONFIGS)   # e.g. the 4 'paper + ...' in one copy, the 4 'code - ...' in another
M6C_GROUPS = manifest64.query("family == 'MGAB'")['group'].tolist()
M6C_SEEDS = SEEDS
M6C_RESULTS = RESULTS_DIR / f'm6c_results_{SESSION_TAG}.csv'
M6C_RUNS = STORE / 'runs' / 'm6c'
M6C_COLUMNS = ['run_id', 'config', 'group', 'seed', 'status', 'seconds', 'error'] + METRIC_KEYS
saved_runs = (read_reimpl_results().query("status == 'success'")
              .drop_duplicates(['mode', 'group', 'seed'], keep='last').set_index(['mode', 'group', 'seed'])['run_id'])

def read_m6c():
    files = sorted(RESULTS_DIR.glob('m6c_results*.csv'))
    if not files:
        return pd.DataFrame(columns=M6C_COLUMNS)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def run_m6c(config, group, seed):
    groups, reuse = M6C_CONFIGS[config]
    run_id = f"{datetime.now().strftime('%Y%m%dT%H%M%S')}_{config.replace(' ', '')}_{group}_s{seed}"
    row = dict(run_id=run_id, config=config, group=group, seed=seed)
    start = time.time()
    try:
        series = load_series(group)
        if reuse:
            model = HybridCoAD(groups).to(DEVICE)
            model.load_state_dict(torch.load(RUNS / saved_runs[(reuse, group, seed)] / 'model.pt', map_location=DEVICE))
        else:
            model, _ = train_model(series, seed=seed, log_every=0, groups=groups)
        score = anomaly_score(model, series)
        (M6C_RUNS / run_id).mkdir(parents=True)
        np.save(M6C_RUNS / run_id / 'scores.npy', score.astype(np.float32))
        row.update(score_metrics(score, series), status='success', error='')
    except Exception as exc:
        row.update(status='failed', error=repr(exc)[-300:])
    row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame([row]).reindex(columns=M6C_COLUMNS).to_csv(M6C_RESULTS, mode='a', header=not M6C_RESULTS.exists(),
                                                            index=False)
    return row

done = read_m6c().query("status == 'success'")
done = set(zip(done['config'], done['group'].astype(int), done['seed'].astype(int)))
todo = [(c, g, s) for c in RUN_CONFIGS for s in M6C_SEEDS for g in M6C_GROUPS if (c, g, s) not in done]
todo.sort(key=lambda run: M6C_CONFIGS[run[0]][1] is None)   # reused models (seconds) before training (minutes)
print(len(todo), 'runs to do | writes to', M6C_RESULTS.name)
session_start = time.time()
for i, (config, group, seed) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    row = run_m6c(config, group, seed)
    print(f"[{i}/{len(todo)}] {config} | group {group} seed {seed}: {row['status']} {row['seconds']:.0f} s | "
          + (f"VUS-PR {100 * row['VUS_PR']:.2f}" if row['status'] == 'success' else row['error']), flush=True)

5 runs to do | writes to m6c_results_gpu-b.csv
[1/5] code - distortion | group 229 seed 48: success 195 s | VUS-PR 56.69
[2/5] code - distortion | group 230 seed 48: success 196 s | VUS-PR 63.19
[3/5] code - distortion | group 231 seed 48: success 197 s | VUS-PR 38.63
[4/5] code - distortion | group 232 seed 48: success 197 s | VUS-PR 44.63
[5/5] code - distortion | group 233 seed 48: success 198 s | VUS-PR 78.85


### 12.3. Results

Mean over the MGAB series of each seed, then mean (and SD of VUS-PR) over the seeds; paper and code mode from
section 7 on the same series. `change vs start`: VUS-PR of `paper + X` minus paper mode, or of `code - X` minus
code mode, series by series (mean over seeds), with the number of series where it is higher and a Wilcoxon test
(only 9 series: read the p-value as weak evidence). Runs of a configuration that are not finished yet are left out of
its row, so check the `runs` column (27 = complete).

In [38]:
m6c = read_m6c().query("status == 'success'").drop_duplicates(['config', 'group', 'seed'], keep='last')
ends = (read_reimpl_results().query("status == 'success'").drop_duplicates(['mode', 'group', 'seed'], keep='last'))
ends = ends[ends['group'].isin(M6C_GROUPS)].assign(config=lambda d: d['mode'] + ' mode (section 7)')
all_runs = pd.concat([ends, m6c], ignore_index=True).astype({'group': int, 'seed': int})
order = (['paper mode (section 7)'] + [c for c in M6C_CONFIGS if c.startswith('paper')]
         + [c for c in M6C_CONFIGS if c.startswith('code')] + ['code mode (section 7)'])
order = [c for c in order if (all_runs['config'] == c).any()]
per_series = {c: 100 * all_runs[all_runs['config'] == c].groupby('group')['VUS_PR'].mean() for c in order}
rows = []
for config in order:
    runs = all_runs[all_runs['config'] == config]
    per_seed = 100 * runs.groupby('seed')[list(METRICS4)].mean()
    row = {'config': config, 'runs': len(runs)}
    row.update({name: per_seed[key].mean() for key, name in METRICS4.items()})
    row['VUS-PR SD (seeds)'] = per_seed['VUS_PR'].std()
    start = ('paper mode (section 7)' if config.startswith('paper +') else
             'code mode (section 7)' if config.startswith('code -') else None)
    if start in per_series:
        common = per_series[config].index.intersection(per_series[start].index)
        diff = per_series[config][common] - per_series[start][common]
        row.update({'change vs start': diff.mean(), 'higher on': f'{int((diff > 0).sum())}/{len(diff)}',
                    'p (Wilcoxon)': wilcoxon(diff).pvalue if len(diff) >= 6 and (diff != 0).any() else np.nan})
    rows.append(row)
m6c_table = pd.DataFrame(rows).set_index('config')
m6c_table.to_csv(REPORTS / 'm6c_summary.csv')
display(m6c_table.round(3))
print('VUS-PR per MGAB series (mean over seeds):')
display(pd.DataFrame(per_series).round(1))

,runs,F1,AUC-PR,R-AUC-PR,VUS-PR,VUS-PR SD (seeds),change vs start,higher on,p (Wilcoxon)
config,,,,,,,,,
paper mode (section 7),27,39.532,25.149,19.223,18.443,8.180,NaN,NaN,NaN
paper + architecture,27,22.332,14.078,10.964,10.599,9.057,-7.843,0/9,0.004
paper + training,27,42.274,29.785,30.708,29.643,10.891,11.200,8/9,0.012
paper + distortion,27,38.557,24.286,17.755,16.830,3.481,-1.612,4/9,0.496
paper + scoring,27,39.728,25.390,19.705,18.974,9.009,0.531,5/9,0.734
code - architecture,27,43.828,33.403,31.480,30.684,11.965,-14.041,0/9,0.004
code - training,27,34.772,22.697,19.787,18.995,9.403,-25.730,0/9,0.004
code - distortion,27,47.373,36.849,39.423,38.763,10.525,-5.962,3/9,0.164
code - scoring,27,48.458,36.963,44.994,43.595,3.694,-1.130,2/9,0.074


VUS-PR per MGAB series (mean over seeds):


,paper mode (section 7),paper + architecture,paper + training,paper + distortion,paper + scoring,code - architecture,code - training,code - distortion,code - scoring,code mode (section 7)
group,,,,,,,,,,
225,13.5,5.5,11.0,6.7,15.3,16.1,6.8,13.5,34.3,34.1
226,10.5,9.3,25.8,18.4,10.4,30.0,30.7,44.1,40.0,41.8
227,28.6,15.4,29.3,21.8,28.6,17.6,15.7,41.5,29.7,31.3
228,20.9,12.8,28.6,10.8,18.6,23.6,6.2,35.8,36.5,36.2
229,11.6,6.0,19.8,15.0,12.1,16.4,16.8,37.7,37.5,37.6
230,17.4,14.0,35.2,19.0,22.5,37.5,30.9,47.2,55.7,58.5
231,12.7,6.7,28.8,16.3,12.6,38.7,22.8,31.2,44.4,48.3
232,24.5,7.2,40.7,21.5,24.6,39.3,14.7,33.7,41.8,42.5
233,26.4,18.4,47.7,21.9,26.1,57.2,26.3,64.1,72.4,72.4


## 13. M8: scoring with the reconstruction branch only

**Hypothesis (from M7, seed 46).** In paper mode, the reconstruction branch alone scores far higher than the full
score (VUS-PR 47.6 vs 37.6 on 64 series), higher than code mode (40.6) and in every family but SED. The paper
reports the opposite for its "Guide w/o Score" variant (Table 2: 27.73 against 39.83 for CoAD).

**One change.** The final score keeps only the reconstruction error (Eq. 11 without `A_c`); the model and its training
are unchanged, so the classifier still guides the soft mask. No training: the saved models of section 7 are rescored.

**Confirmation on seeds that were not looked at.** The hypothesis came from the seed-46 models. Section 13.1
rescores the seed-47 and seed-48 models, never inspected before, and 13.2 tests the prediction written here, before
running: *on seeds 47-48, paper-mode reconstruction-only has a higher VUS-PR than code mode (CoAD as released),
series by series (one-sided Wilcoxon, p < 0.05)*. The test series are the same as in M7, so this does not replace a
test on new data; it rules out a lucky seed.

Needs cells 0.1-1.2, 3.1, 4.1, 5.1, 6.1, 7.1, 9.1, 9.3 and the first cell of 11.2 (helpers). About 1 h on a T4.

### 13.1. Rescore seeds 47 and 48 (re-run to continue)

In [39]:
M8_SEEDS = [47, 48]
M8_RESULTS = RESULTS_DIR / f'm8_recon_{SESSION_TAG}.csv'
M8_COLUMNS = ['run_id', 'mode', 'group', 'seed'] + METRIC_KEYS

def read_m8():
    files = sorted(RESULTS_DIR.glob('m8_recon*.csv'))
    if not files:
        return pd.DataFrame(columns=M8_COLUMNS)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True).drop_duplicates(['mode', 'group', 'seed'],
                                                                                        keep='last')

done = read_m8()
done = set(zip(done['mode'], done['group'].astype(int), done['seed'].astype(int)))
todo = [(m, g, s) for m in ['paper', 'code'] for s in M8_SEEDS for g in groups64 if (m, g, s) not in done]
print(len(todo), 'models to rescore | writes to', M8_RESULTS.name)
session_start = time.time()
for i, (mode, group, seed) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    series = load_series(group)
    model, run_id = load_reimpl_model(mode, group, seed)
    parts = branch_scores(model, series)
    gap = float(np.abs(parts['full'] - np.load(RUNS / run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run_id}: recomputed score differs from scores.npy by {gap:.1e}'
    row = dict(run_id=run_id, mode=mode, group=group, seed=seed, **score_metrics(parts['reconstruction only'], series))
    pd.DataFrame([row]).reindex(columns=M8_COLUMNS).to_csv(M8_RESULTS, mode='a', header=not M8_RESULTS.exists(),
                                                           index=False)
    print(f"[{i}/{len(todo)}] {mode} group {group} seed {seed}: reconstruction only VUS-PR {100 * row['VUS_PR']:.2f}",
          flush=True)

19 models to rescore | writes to m8_recon_gpu-b.csv
[1/19] code group 273 seed 48: reconstruction only VUS-PR 6.31
[2/19] code group 274 seed 48: reconstruction only VUS-PR 64.70
[3/19] code group 275 seed 48: reconstruction only VUS-PR 1.86
[4/19] code group 276 seed 48: reconstruction only VUS-PR 81.34
[5/19] code group 287 seed 48: reconstruction only VUS-PR 12.87
[6/19] code group 288 seed 48: reconstruction only VUS-PR 25.12
[7/19] code group 289 seed 48: reconstruction only VUS-PR 10.75
[8/19] code group 290 seed 48: reconstruction only VUS-PR 28.87
[9/19] code group 291 seed 48: reconstruction only VUS-PR 12.58
[10/19] code group 292 seed 48: reconstruction only VUS-PR 13.15
[11/19] code group 293 seed 48: reconstruction only VUS-PR 22.08
[12/19] code group 294 seed 48: reconstruction only VUS-PR 51.77
[13/19] code group 295 seed 48: reconstruction only VUS-PR 17.11
[14/19] code group 296 seed 48: reconstruction only VUS-PR 10.42
[15/19] code group 297 seed 48: reconstruction on

### 13.2. Results

1. Reconstruction-only against the full score of the same models, per mode, separately for the exploratory seed
   (46, from 11.4) and the confirmatory seeds (47-48).
2. The prediction.
3. Paper-mode reconstruction-only (seeds 47-48) as a new method next to Table 1: average rank and paired tests
   with Holm correction, as in M5.
4. By family.

In [40]:
recon = read_m8()
m7_files = sorted(RESULTS_DIR.glob('m7_branches*.csv'))
if m7_files:
    seed46 = pd.concat([pd.read_csv(f) for f in m7_files], ignore_index=True)
    seed46 = seed46[seed46['branch'] == 'reconstruction only'].drop(columns='branch')
    recon = pd.concat([seed46, recon], ignore_index=True)
recon = recon.astype({'group': int, 'seed': int}).drop_duplicates(['mode', 'group', 'seed'], keep='last')
full = reimpl.astype({'group': int, 'seed': int})

def series_means(frame, mode, seeds, key):
    """Per-series mean (x100) over the given seeds, only for series that have all of them."""
    part = frame[(frame['mode'] == mode) & frame['seed'].isin(seeds)]
    counts = part.groupby('group')['seed'].nunique()
    part = part[part['group'].isin(counts[counts == len(seeds)].index)]
    return 100 * part.groupby('group')[key].mean()

# 1. reconstruction only vs full score of the same models
rows = []
for label, seeds in {'exploratory (seed 46)': [46], 'confirmatory (seeds 47-48)': [47, 48]}.items():
    for mode in ['paper', 'code']:
        for key, name in METRICS4.items():
            r, f = series_means(recon, mode, seeds, key), series_means(full, mode, seeds, key)
            common = r.index.intersection(f.index)
            if len(common) < 6:
                continue
            diff = r[common] - f[common]
            rows.append({'seeds': label, 'mode': mode, 'metric': name, 'series': len(common),
                         'full': f[common].mean(), 'reconstruction only': r[common].mean(), 'difference': diff.mean(),
                         'higher on': f'{int((diff > 0).sum())}/{len(diff)}', 'p (Wilcoxon)': wilcoxon(diff).pvalue})
m8_table = pd.DataFrame(rows)
m8_table.to_csv(REPORTS / 'm8_reconstruction_only.csv', index=False)
display(m8_table.round(3))

# 2. the prediction written before 13.1
r, c = series_means(recon, 'paper', [47, 48], 'VUS_PR'), series_means(full, 'code', [47, 48], 'VUS_PR')
common = r.index.intersection(c.index)
if len(common) == len(groups64):
    diff = r[common] - c[common]
    p = wilcoxon(diff, alternative='greater').pvalue
    print(f'Prediction (seeds 47-48, VUS-PR): paper-mode reconstruction-only {r.mean():.2f} vs code mode {c.mean():.2f}, '
          f'higher on {int((diff > 0).sum())}/{len(diff)} series, one-sided p = {p:.1e} -> '
          + ('HOLDS' if diff.mean() > 0 and p < 0.05 else 'FAILS'))
else:
    print(f'Prediction not tested yet: {len(common)} of {len(groups64)} series rescored for both seeds')

# 3. next to Table 1
NEW = 'CoAD recon-only (paper mode)'
new_vus = series_means(recon, 'paper', [47, 48], 'VUS_PR').reindex(groups64)
if new_vus.notna().all():
    vus = MATRIX['VUS-PR'][TABLE1_METHODS].assign(**{NEW: new_vus})
    print('Average rank among 9 methods (VUS-PR, lower is better):')
    display(vus.rank(axis=1, ascending=False).mean().sort_values().round(2).to_frame('average rank'))
    rows = []
    for other in TABLE1_METHODS:
        diff = (vus[NEW] - vus[other]).to_numpy()
        low, high = bootstrap_ci(diff)
        rows.append({'against': other, 'better': int((diff > 0).sum()), 'worse': int((diff < 0).sum()),
                     'mean diff': diff.mean(), 'CI low': low, 'CI high': high, 'p': wilcoxon(diff).pvalue})
    tests = pd.DataFrame(rows)
    tests['p Holm'] = holm(tests['p'])
    tests['significant'] = tests['p Holm'] < 0.05
    tests.to_csv(REPORTS / 'm8_against_table1.csv', index=False)
    display(tests.round(3))

    # 4. by family
    families = family_of.reindex(groups64).values
    per_series_vus = pd.DataFrame({
        'CoAD (reproduction)': vus['CoAD'], 'Kmeans': vus['Kmeans'],
        'code mode, full (47-48)': series_means(full, 'code', [47, 48], 'VUS_PR').reindex(groups64),
        'paper mode, full (47-48)': series_means(full, 'paper', [47, 48], 'VUS_PR').reindex(groups64),
        NEW + ' (47-48)': new_vus})
    by_family = per_series_vus.groupby(families).mean()
    by_family.loc['all'] = per_series_vus.mean()
    by_family.to_csv(REPORTS / 'm8_by_family.csv')
    display(by_family.round(2))

,seeds,mode,metric,series,full,reconstruction only,difference,higher on,p (Wilcoxon)
0,exploratory (seed 46),paper,F1,64,45.483,50.725,5.242,45/64,0.000
1,exploratory (seed 46),paper,AUC-PR,64,39.221,46.737,7.516,47/64,0.000
2,exploratory (seed 46),paper,R-AUC-PR,64,38.448,48.499,10.051,49/64,0.000
3,exploratory (seed 46),paper,VUS-PR,64,37.597,47.608,10.011,50/64,0.000
4,exploratory (seed 46),code,F1,64,44.842,48.643,3.801,37/64,0.027
5,exploratory (seed 46),code,AUC-PR,64,38.501,42.272,3.771,35/64,0.138
6,exploratory (seed 46),code,R-AUC-PR,64,41.807,41.479,-0.328,28/64,0.766
7,exploratory (seed 46),code,VUS-PR,64,40.610,40.884,0.273,32/64,0.800
8,confirmatory (seeds 47-48),paper,F1,64,46.245,49.973,3.728,44/64,0.002
9,confirmatory (seeds 47-48),paper,AUC-PR,64,40.112,44.933,4.821,45/64,0.000


Prediction (seeds 47-48, VUS-PR): paper-mode reconstruction-only 44.76 vs code mode 42.23, higher on 41/64 series, one-sided p = 1.9e-02 -> HOLDS
Average rank among 9 methods (VUS-PR, lower is better):


,average rank
method,
Kmeans,3.08
CoAD recon-only (paper mode),3.59
KshapeAD,4.17
CoAD,4.48
SAND,5.12
Series2Graph,5.38
MP,5.42
PCA,6.42
damp,7.33


,against,better,worse,mean diff,CI low,CI high,p,p Holm,significant
0,CoAD,43,21,3.058,-0.839,6.772,0.029,0.086,False
1,Kmeans,30,34,3.745,-1.640,9.239,0.570,0.570,False
2,KshapeAD,30,34,8.897,1.597,16.342,0.188,0.375,False
3,SAND,40,24,13.696,6.548,21.148,0.002,0.007,True
4,Series2Graph,46,18,13.320,6.908,19.683,0.000,0.001,True
5,MP,41,23,15.273,7.995,22.860,0.001,0.003,True
6,PCA,59,5,20.831,16.223,25.730,0.000,0.000,True
7,damp,57,7,31.764,25.290,38.132,0.000,0.000,True


,CoAD (reproduction),Kmeans,"code mode, full (47-48)","paper mode, full (47-48)",CoAD recon-only (paper mode) (47-48)
IOPS,32.50,32.52,31.47,28.52,36.27
MGAB,41.71,0.99,46.44,21.12,36.35
SED,93.19,92.47,92.21,85.74,59.42
SVDB,56.09,62.52,55.38,59.87,67.51
TODS,22.62,35.67,24.38,25.66,26.14
all,41.70,41.01,42.23,39.29,44.76


## 14. M9: where CoAD predicts wrong, why, and the two problems

All on the **code-mode models** (= authors' CoAD), 64 series × 3 seeds, saved in section 7; no training.
Comparison axis: CoAD reimplementation (code mode) vs Table 1.

* 14.1 extracts the intermediate signals of every model: OE probability `A` (time/frequency classifier alone),
  residual probability `A^r`, `A_c = (A + A^r) / 2`, reconstruction error, the two processed branches and the final
  score, plus **backup reconstruction** (problem 1, module A): every patch is reconstructed once with a full mask that
  does not depend on OE (4 passes, patches `i % 4 == j` masked in pass `j`); its error is counted only on the masked
  patches.
* 14.2 runs it on the 192 models and writes one row per labelled anomaly event and one row per model.
* 14.3 error analysis: which events are missed, which normal points get high scores, and at which step of the
  pipeline the information is lost.
* 14.4 problem 1 (cooperation when OE misses) and module A.
* 14.5 problem 2 (long anomalies), observational.
* 14.6-14.7 problem 2, controlled stress test: anomalies of fixed type injected into normal stretches with
  increasing length and shifted patch offset.

**Oracle diagnostics.** 14.3-14.5 use the test labels to locate events: they explain errors, they are not methods.
An event counts as *found* when its peak score is above 99% of the normal points of the same series
(`peak percentile >= 0.99`); this is a diagnostic threshold, not a deployable one. Module A's scores (14.4) use no
labels; its weights come from the training part only.

**Predictions written before running.**

* H1 (problem 1): among events that OE misses (`A` peak percentile < 0.99), reconstruction rescues fewer than half;
  OE and reconstruction-error percentiles are positively correlated across events (the branches fail together);
  backup masking raises the reconstruction percentile of OE-missed events.
* Module A: `calibrated + backup` beats CoAD (VUS-PR, 64 series, mean over 3 seeds, one-sided Wilcoxon p < 0.05).
* H2 (problem 2): in the stress test with the uniform-replacement type (seen in training), the mean percentile of
  the final score at 4 periods is at least 0.05 below that at 1 period, and at >= 2 periods the middle third of the
  event scores lower than its edges.

Needs cells 0.1-1.2, 3.1, 4.1, 5.1, 6.1, 7.1 and 9.1; a GPU (about 2-3 h for 14.2, 1 h for 14.6).

### 14.1. Intermediate signals and backup reconstruction

In [ ]:
@torch.no_grad()
def predict_components(model, values, window, stride):
    """Per-point A (OE), A^r (residual), A_c and the final reconstruction, averaged over overlapping windows.
    Same computation as CoADReimpl.predict (section 4.1), keeping the intermediate tensors."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    parts = {k: [] for k in ['oe', 'residual', 'combined', 'reconstruction']}
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        oe = torch.sigmoid(model.classifier(x))
        first = model.mae(x, oe)
        residual = torch.sigmoid(model.classifier.residual_logit(x, first))
        combined = (oe + residual) / 2
        final = model.mae(x, combined) if model.faithful else first      # D10
        for key, value in [('oe', oe), ('residual', residual), ('combined', combined)]:
            parts[key].append(value.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        parts['reconstruction'].append(final.cpu())
    return {k: windows_to_series(torch.cat(v).numpy(), len(values), stride, pad) for k, v in parts.items()}

@torch.no_grad()
def backup_error(model, values, window, stride, n_groups=4):
    """Module A: |x - reconstruction| when the patch itself is fully masked, independently of OE."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    out = []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        B, T = x.shape
        index = torch.arange(T // PATCH, device=x.device)
        error = torch.zeros_like(x)
        for j in range(n_groups):
            masked = (index % n_groups == j).float()
            reconstruction = model.mae(x, masked.view(1, -1, 1).expand(B, -1, 1))
            error += (x - reconstruction).abs() * masked.repeat_interleave(PATCH).view(1, T)
        out.append(error.cpu())
    return windows_to_series(torch.cat(out).numpy(), len(values), stride, pad)

def processed(raw_test, raw_train, period, smooth=True, zero_start=True):
    """Authors' code-mode processing of one source: |x - train mean|, moving average, first period zeroed.
    Returns the processed test score and its train counterpart (for calibration)."""
    mean = raw_train.mean()
    out = []
    for raw in (raw_test, raw_train):
        value = np.abs(raw - mean)
        value = moving_average(value, period) if smooth else value.copy()
        if zero_start:
            value[:period] = 0
        out.append(value)
    return out

def model_signals(model, series, values=None, train_parts=None):
    """All point signals of one model on the test series (or on `values`, e.g. a slice with an injected anomaly).
    train_parts: precomputed training-part components, reused by the stress test."""
    period, window = series['period'], series['window']
    values = series['test'] if values is None else values
    test = predict_components(model, values, window, period)
    test['backup'] = backup_error(model, values, window, period)
    if train_parts is None:
        train_parts = predict_components(model, series['train'], window, period)
        train_parts['backup'] = backup_error(model, series['train'], window, period)
    zero = values is series['test']
    error_test, error_train = np.abs(values - test['reconstruction']), np.abs(series['train'] - train_parts['reconstruction'])
    r, r_train = processed(error_test, error_train, period, zero_start=zero)
    c, c_train = processed(test['combined'], train_parts['combined'], period, zero_start=zero)
    b, b_train = processed(test['backup'], train_parts['backup'], period, zero_start=zero)
    r_raw, _ = processed(error_test, error_train, period, smooth=False, zero_start=zero)
    c_raw, _ = processed(test['combined'], train_parts['combined'], period, smooth=False, zero_start=zero)
    s = {name: np.std(t) + 1e-12 for name, t in [('r', r_train), ('c', c_train), ('b', b_train)]}
    signals = {
        'full': (r + c) / 2, 'classify': c, 'reconstruct': r, 'backup': b, 'full_unsmoothed': (r_raw + c_raw) / 2,
        'oe': test['oe'], 'residual': test['residual'], 'error': error_test,
        'calibrated': (r / s['r'] + c / s['c']) / 2,
        'calibrated + backup': (r / s['r'] + c / s['c'] + b / s['b']) / 3,
    }
    return signals, train_parts

EVENT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup', 'full_unsmoothed', 'oe', 'residual', 'error']
SPLIT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup']
VARIANTS = ['calibrated', 'backup', 'calibrated + backup']
# Fixed column orders: rows are appended model by model, so every model must write the same columns in the same order
EVENT_COLUMNS = (['run_id', 'group', 'seed', 'family', 'event', 'start', 'length', 'length_over_period', 'window_share',
                  'amplitude_z'] + [f'{s}_{k}' for s in EVENT_SIGNALS
                                    for k in (['peak', 'mean', 'middle', 'edges'] if s in SPLIT_SIGNALS else ['peak', 'mean'])])
STRESS_COLUMNS = (['group', 'seed', 'family', 'type', 'length_periods', 'offset', 'position']
                  + [f'{s}_{k}' for s in SPLIT_SIGNALS + ['oe'] for k in ['peak', 'mean', 'middle', 'edges']] + ['spill'])

def segments(label):
    edges = np.diff(np.concatenate([[0], np.asarray(label).astype(int), [0]]))
    return list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))

def percentile_against(reference):
    """Fraction of the reference (normal) values strictly below each value."""
    ordered = np.sort(reference)
    return lambda v: np.searchsorted(ordered, v, side='left') / len(ordered)

def event_stats(signal, start, end, pct):
    """Peak and mean percentile over an event; middle third vs both edge thirds for events of 3 patches or more."""
    p = pct(signal[start:end])
    row = {'peak': float(pct(np.array([signal[start:end].max()]))[0]), 'mean': float(p.mean())}
    if end - start >= 3 * PATCH:
        third = (end - start) // 3
        row.update(middle=float(p[third:2 * third].mean()), edges=float(np.r_[p[:third], p[2 * third:]].mean()))
    return row

def event_rows(series, signals, base):
    """One row per labelled event. Normal reference: label 0, after the first period."""
    label, test, period = np.asarray(series['label']), series['test'], series['period']
    normal = (label == 0) & (np.arange(len(label)) >= period)
    mu, sd = test[normal].mean(), test[normal].std() + 1e-12
    pcts = {name: percentile_against(signals[name][normal]) for name in EVENT_SIGNALS}
    rows = []
    for k, (a, b) in enumerate(segments(label)):
        row = dict(base, event=k, start=int(a), length=int(b - a), length_over_period=(b - a) / period,
                   window_share=min(1.0, (b - a) / series['window']),
                   amplitude_z=float(np.abs(test[a:b] - mu).max() / sd))
        for name in EVENT_SIGNALS:
            stats = event_stats(signals[name], a, b, pcts[name])
            if name not in SPLIT_SIGNALS:
                stats = {k2: v for k2, v in stats.items() if k2 in ('peak', 'mean')}
            row.update({f'{name}_{k2}': v for k2, v in stats.items()})
        rows.append(row)
    return rows

def alarm_stats(series, score):
    """Top-k points by score, k = number of anomalous points: share that are normal, and of those the share
    within one period of an event (spill-over) rather than far from any event."""
    label, period = np.asarray(series['label']), series['period']
    valid = np.arange(len(label)) >= period
    k = int(label[valid].sum())
    top = np.where(valid)[0][np.argsort(-score[valid])[:k]]
    normal_top = top[label[top] == 0]
    near = np.zeros(len(label), bool)
    for a, b in segments(label):
        near[max(0, a - period):b + period] = True
    return {'top_k_normal_share': len(normal_top) / max(k, 1),
            'normal_alarms_near_share': float(near[normal_top].mean()) if len(normal_top) else np.nan}

### 14.2. Run on the 192 code-mode models (re-run to continue)

Per model: the check that the recomputed final score equals the saved `scores.npy`, one row per event
(`results/m9_events_<SESSION_TAG>.csv`) and one row per model with the metrics of the module-A variants and the
alarm statistics (`results/m9_models_<SESSION_TAG>.csv`). Scoring three extra variants with the evaluator is the slow
part.

In [ ]:
M9_EVENTS = RESULTS_DIR / f'm9_events_{SESSION_TAG}.csv'
M9_MODELS = RESULTS_DIR / f'm9_models_{SESSION_TAG}.csv'
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))

def read_m9(kind):
    files = sorted(RESULTS_DIR.glob(f'm9_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_m9('models')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if (int(r.group), int(r.seed)) not in done]
todo.sort(key=lambda r: r.seed)
print(len(code_runs), 'code-mode models,', len(todo), 'to analyse')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    signals, _ = model_signals(model, series)
    gap = float(np.abs(signals['full'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: recomputed final score differs from scores.npy by {gap:.1e}'
    base = dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group])
    rows = event_rows(series, signals, base)
    model_row = dict(base, n_events=len(rows), max_score_diff=gap, **alarm_stats(series, signals['full']))
    for variant in VARIANTS:
        metrics = score_metrics(signals[variant], series)
        model_row.update({f'{variant}: {k}': v for k, v in metrics.items()})
        model_row.update({f'{variant}: {k}': v for k, v in alarm_stats(series, signals[variant]).items()})
    pd.DataFrame(rows).reindex(columns=EVENT_COLUMNS).to_csv(M9_EVENTS, mode='a', header=not M9_EVENTS.exists(), index=False)
    model_row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame([model_row]).to_csv(M9_MODELS, mode='a', header=not M9_MODELS.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} events, {model_row['seconds']:.0f} s | VUS-PR "
          + ' '.join(f"{v} {100 * model_row[f'{v}: VUS_PR']:.2f}" for v in VARIANTS), flush=True)

### 14.3. Error analysis: what is predicted wrong, and where the information is lost

* **Missed events** (`full` peak percentile < 0.99), by family and by event length.
* **Normal points with high scores**: of the top-k points (k = number of anomalous points), the share that are
  normal, and how many of those lie within one period of an event (spill-over / localisation) rather than far away
  (false alarm on normal behaviour).
* **Step of the pipeline where a missed event is lost**, checked in this order:
  1. `smoothing`: found by the unsmoothed final score;
  2. `fusion`: found by one processed branch (`classify` or `reconstruct`) but not by their average;
  3. `processing`: found by a raw source (`oe`, `residual` or `error`) but by neither processed branch;
  4. `all sources weak`: no signal finds it.

In [ ]:
events = read_m9('events').drop_duplicates(['group', 'seed', 'event'], keep='last')
models = read_m9('models').drop_duplicates(['group', 'seed'], keep='last')
print(len(models), 'models,', len(events), 'events')
FOUND = 0.99
events['found'] = events['full_peak'] >= FOUND
events['length bin'] = pd.cut(events['length_over_period'], [0, 0.1, 0.5, 1, 2, np.inf],
                              labels=['< 0.1 period', '0.1-0.5', '0.5-1', '1-2', '> 2 periods'])

def step_lost(row):
    if row['found']:
        return 'found'
    if row['full_unsmoothed_peak'] >= FOUND:
        return 'smoothing'
    if max(row['classify_peak'], row['reconstruct_peak']) >= FOUND:
        return 'fusion'
    if max(row['oe_peak'], row['residual_peak'], row['error_peak']) >= FOUND:
        return 'processing'
    return 'all sources weak'
events['step'] = events.apply(step_lost, axis=1)

print('Share of events found (peak percentile >= 0.99), by family:')
display(events.groupby('family')['found'].agg(['mean', 'size']).round(3))
print('Share found by event length (relative to the period):')
display(events.groupby(['family', 'length bin'], observed=True)['found'].agg(['mean', 'size']).round(3).unstack(0))
print('Where missed events are lost (share of missed events):')
step_table = pd.crosstab(events.loc[~events['found'], 'family'], events.loc[~events['found'], 'step'], normalize='index')
display(step_table.round(3))
print('Normal points among the top-k scores, and share of those near an event (spill-over), mean over models:')
display(models.groupby('family')[['top_k_normal_share', 'normal_alarms_near_share']].mean().round(3))
events.to_csv(REPORTS / 'm9_events_all.csv', index=False)
step_table.to_csv(REPORTS / 'm9_step_lost.csv')

### 14.4. Problem 1: does reconstruction rescue the events OE misses? Module A

* Event level (oracle diagnostic): among events with `oe` peak < 0.99, the share rescued by the reconstruction error,
  the residual classifier, the final score and the backup reconstruction; the same for events OE finds. Spearman
  correlation of OE and reconstruction-error mean percentiles across events. Backup vs guided reconstruction on
  OE-missed events (paired Wilcoxon on mean percentile): does changing only the mask change the reconstruction
  evidence?
* Series level (no labels used by the method): VUS-PR etc. of `calibrated` (CoAD's two branches each divided by
  their standard deviation on the training part), `backup` alone and `calibrated + backup`, against CoAD (section 7),
  mean over 3 seeds, with Table 1. `calibrated` isolates the effect of rescaling from the effect of backup masking.

In [ ]:
from scipy.stats import spearmanr
events['oe_found'] = events['oe_peak'] >= FOUND
rescue = events.groupby('oe_found')[[f'{s}_peak' for s in ['error', 'residual', 'reconstruct', 'full', 'backup']]] \
    .agg(lambda v: (v >= FOUND).mean())
rescue.index = rescue.index.map({False: 'OE misses', True: 'OE finds'})
rescue['events'] = events.groupby('oe_found').size().values
print('Share of events found by each signal, split by whether OE finds the event:')
display(rescue.round(3))
print('Same, OE-missed events only, by family:')
display(events[~events['oe_found']].groupby('family')[['error_peak', 'reconstruct_peak', 'full_peak', 'backup_peak']]
        .agg(lambda v: (v >= FOUND).mean()).round(3))
rho = spearmanr(events['oe_mean'], events['error_mean'], nan_policy='omit')
print(f'Spearman(OE, reconstruction error) over events: rho = {rho.statistic:.2f}, p = {rho.pvalue:.1e}')
missed = events[~events['oe_found']]
diff = missed['backup_mean'] - missed['reconstruct_mean']
print(f'OE-missed events: backup minus guided reconstruction, mean percentile {diff.mean():+.3f}, '
      f'higher on {int((diff > 0).sum())}/{len(diff)}, Wilcoxon p = {wilcoxon(diff).pvalue:.1e}')

rows = []
for key, name in METRICS4.items():
    coad = 100 * code_runs.astype({'group': int, 'seed': int}).set_index(['group', 'seed'])[key]
    row = {'metric': name, 'Table 1': {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}[name],
           'CoAD (code mode)': coad.groupby(level='seed').mean().mean()}
    for variant in VARIANTS:
        v = 100 * models.set_index(['group', 'seed'])[f'{variant}: {key}']
        common = v.index.intersection(coad.index)
        per_series = (v[common] - coad[common]).groupby(level='group').mean()
        row[variant] = v[common].groupby(level='seed').mean().mean()
        row[f'{variant} - CoAD'] = per_series.mean()
        row[f'{variant} higher on'] = f'{int((per_series > 0).sum())}/{len(per_series)}'
        row[f'{variant} p (one-sided)'] = wilcoxon(per_series, alternative='greater').pvalue
    rows.append(row)
module_a = pd.DataFrame(rows).set_index('metric')
module_a.to_csv(REPORTS / 'm9_module_a.csv')
display(module_a.T.round(3))

### 14.5. Problem 2, observational: long events

Within each family (lengths are confounded with family: 19 of the 22 series with anomalies longer than a period are
SVDB), the share found by event length; for events of 3 patches or more, the middle third against the edges
(paired, per signal). Middle < edges = the model reacts to the transitions but not to a sustained anomaly.

In [ ]:
long_events = events.dropna(subset=['full_middle']) if 'full_middle' in events else events.iloc[:0]
print(len(long_events), 'events of 3 patches or more')
rows = []
for (family, length_bin), d in long_events.groupby(['family', 'length bin'], observed=True):
    if len(d) < 5:
        continue
    row = {'family': family, 'length': length_bin, 'events': len(d)}
    for s in SPLIT_SIGNALS:
        gap = d[f'{s}_middle'] - d[f'{s}_edges']
        row[f'{s}: middle - edges'] = gap.mean()
        row[f'{s}: p'] = wilcoxon(gap).pvalue if (gap != 0).any() else np.nan
    rows.append(row)
middle_edges = pd.DataFrame(rows)
middle_edges.to_csv(REPORTS / 'm9_middle_edges.csv', index=False)
display(middle_edges.round(3))
print('Spearman(length / period, full mean percentile) within family:')
for family, d in events.groupby('family'):
    r = spearmanr(d['length_over_period'], d['full_mean'])
    print(f'  {family}: rho = {r.statistic:+.2f} (p = {r.pvalue:.1e}, {len(d)} events)')

### 14.6. Problem 2, controlled stress test (re-run to continue)

Series: per family, the first two in manifest order that have room for the injections (a family with dense anomalies, e.g. TODS, may have none), the 3 seeds. Injected into normal stretches
(at least one window away from any labelled anomaly; 4 positions drawn once with `default_rng(2026 + group)`):

* types: `uniform` (segment replaced by its first value, a distortion seen in training) and `level`
  (segment shifted by +3 standard deviations of the training part);
* lengths: 0.25, 0.5, 1, 2 and 4 periods (4 periods = the whole window); offset 0;
* patch offset: length 1 period, start shifted by 0-7 points (first 2 positions), `uniform` type.

Only a slice of two windows around the injection is rescored (the rest of the score is unchanged); percentiles are
against the normal points of the untouched series. `spill` = mean percentile of the final score within one period
after the injected segment.

In [ ]:
M9_STRESS = RESULTS_DIR / f'm9_stress_{SESSION_TAG}.csv'
STRESS_LENGTHS = [0.25, 0.5, 1, 2, 4]
STRESS_TYPES = ['uniform', 'level']
N_POSITIONS, OFFSETS = 4, range(PATCH)

def inject(values, start, length, kind, train_sd):
    values = values.copy()
    if kind == 'uniform':
        values[start:start + length] = values[start]
    else:
        values[start:start + length] += 3 * train_sd
    return values

def stress_cases(series):
    period, window, n = series['period'], series['window'], len(series['test'])
    longest = int(max(STRESS_LENGTHS) * period)
    blocked = np.zeros(n, bool)
    for a, b in segments(series['label']):
        blocked[max(0, a - window - longest):b + window] = True
    candidates = [s for s in range(max(2 * window, period), n - longest - 2 * window) if not blocked[s]]
    if not candidates:
        return []
    rng = np.random.default_rng(2026 + series['group'])
    positions = sorted(rng.choice(candidates, size=min(N_POSITIONS, len(candidates)), replace=False))
    cases = [(kind, L, 0, p) for kind in STRESS_TYPES for L in STRESS_LENGTHS for p in positions]
    cases += [('uniform', 1, o, p) for o in OFFSETS for p in positions[:2]]
    return cases

STRESS_GROUPS = []   # per family, the first two series (manifest order) with room for the injections
for family, d in manifest64.groupby('family', sort=False):
    chosen = []
    for g in d['group']:
        if len(chosen) == 2:
            break
        if stress_cases(load_series(int(g))):
            chosen.append(int(g))
    print(f'{family}: stress series {chosen}' + ('' if chosen else ' (no normal stretch long enough)'))
    STRESS_GROUPS += chosen

done = read_m9('stress')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in STRESS_GROUPS and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'models for the stress test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    period, window = series['period'], series['window']
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    base_signals, train_parts = model_signals(model, series)
    label = np.asarray(series['label'])
    keep = np.ones(len(label), bool)
    for a, b in segments(label):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    pcts = {s: percentile_against(base_signals[s][keep]) for s in SPLIT_SIGNALS + ['oe']}
    train_sd = series['train'].std()
    rows = []
    for kind, L, offset, position in stress_cases(series):
        length = max(1, int(round(L * period)))
        start = position + offset
        a, b = start - 2 * window, start + length + 2 * window
        values = inject(series['test'], start, length, kind, train_sd)[a:b]
        signals, _ = model_signals(model, series, values=values, train_parts=train_parts)
        s0 = start - a
        row = dict(group=group, seed=seed, family=family_of[group], type=kind, length_periods=L, offset=offset,
                   position=int(position))
        for s in SPLIT_SIGNALS + ['oe']:
            stats = event_stats(signals[s], s0, s0 + length, pcts[s])
            row.update({f'{s}_{k}': v for k, v in stats.items()})
        row['spill'] = float(pcts['full'](signals['full'][s0 + length:s0 + length + period]).mean())
        rows.append(row)
    if rows:
        pd.DataFrame(rows).reindex(columns=STRESS_COLUMNS).to_csv(M9_STRESS, mode='a', header=not M9_STRESS.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} injections', flush=True)

### 14.7. Stress test results and the H2 prediction

In [ ]:
stress = read_m9('stress')
lengths = stress[stress['offset'] == 0]
cols = ['full_mean', 'full_peak', 'classify_mean', 'reconstruct_mean', 'backup_mean', 'oe_mean', 'spill']
print('Mean percentile of the injected segment, by type and length (mean over series, positions, seeds):')
by_length = lengths.groupby(['type', 'length_periods'])[cols].mean()
display(by_length.round(3))
print('Middle third minus edges (events of 3 patches or more):')
display(lengths.dropna(subset=['full_middle']).assign(
    **{f'{s}: middle - edges': lambda d, s=s: d[f'{s}_middle'] - d[f'{s}_edges'] for s in SPLIT_SIGNALS})
    .groupby(['type', 'length_periods'])[[f'{s}: middle - edges' for s in SPLIT_SIGNALS]].mean().round(3))
offsets = stress[(stress['length_periods'] == 1) & (stress['type'] == 'uniform')]
spread = offsets.groupby(['group', 'seed', 'position'])['full_mean'].agg(lambda v: v.max() - v.min())
print(f'Patch offset: range of the full-score mean percentile over 8 offsets, median {spread.median():.3f}, '
      f'90th percentile {spread.quantile(0.9):.3f}')

u = lengths[lengths['type'] == 'uniform'].groupby(['group', 'seed', 'position', 'length_periods'])['full_mean'].mean().unstack()
drop = (u[1.0] - u[4.0]).dropna()
mid = lengths[(lengths['type'] == 'uniform') & (lengths['length_periods'] >= 2)].dropna(subset=['full_middle'])
gap = mid['full_middle'] - mid['full_edges']
holds = drop.mean() >= 0.05 and gap.mean() < 0 and wilcoxon(gap).pvalue < 0.05
print(f'H2: 1 period minus 4 periods = {drop.mean():+.3f} (needs >= 0.05); middle - edges at >= 2 periods = '
      f'{gap.mean():+.3f} (p = {wilcoxon(gap).pvalue:.1e}) -> ' + ('HOLDS' if holds else 'FAILS'))
by_length.to_csv(REPORTS / 'm9_stress_by_length.csv')

## 15. M10: three mechanisms of CoAD under test (stage D1 of `04.md`)

The questions of `04.md` target CoAD's core mechanisms. Code-mode models (= authors' CoAD), no change to the method.

**D0, implementation audit (done by reading the code, section 4.1):**

| Question of 04.md | Implementation | Consequence |
|---|---|---|
| Is the mask applied before or after contextualisation? | Mixed into the **per-patch** embedding (`proj` + SELU), **before** the Bi-GRU | Path "embedding already carries context" does not exist: H2-C dropped |
| Does soft mixing keep information? | `mixed = p E_mask + (1 - p) e`: for `p < 1` part of `e` remains | Path 1 exists: tested by H2-A |
| Can neighbouring patches carry the anomaly? | The Bi-GRU runs **after** masking and reads all patches | Path 2 exists: tested by H2-B |
| Is the mask probability detached? | **No**: the reconstruction loss reaches the classifier through the mask | Gradient path exists: tested in 15.4 |

**Experiments:**

* 15.2 (192 models, inference): **H1-A** OE probability in the middle vs the edges of real long anomalies;
  **H2-A** on the same patches, guided soft mask vs hard mask (p = 1), anomalous vs normal patches;
  **H2-B** at the central patch of 3 anomalous patches, hard-masking the anomalous neighbours (span) vs the same
  number of far normal patches (scattered), against the same contrast at normal centres.
* 15.3 (stress, inference): **H1-B** same inner change (level shift or noise burst, 1 or 3 training SDs) with
  abrupt or smooth (raised-cosine) boundaries; the plateau, where both versions are identical, is compared.
* 15.4 (training, 9 MGAB series): cosine between the gradients of the classification and reconstruction losses on
  the classifier parameters during training, and an intervention: the mask probability detached.

**Predictions written before running** (a hypothesis is *supported* only if its prediction holds):

* H1-A: on real events of 3 patches or more, OE middle minus edges percentile <= -0.05 (Wilcoxon p < 0.05).
* H1-B: OE plateau percentile, abrupt minus smooth >= 0.05 (paired, p < 0.05).
* H2-A: patch-level AUC (anomalous vs normal patches, reconstruction error) hard minus soft >= 0.02 (p < 0.05).
* H2-B: (span - scattered) error gain at anomalous centres exceeds that at normal centres (p < 0.05).
* Gradient conflict: mean cosine < 0 **and** the detached variant has a higher VUS-PR than joint training
  (9 series, one-sided p < 0.05).

H1-A, H2-A and H2-B use the test labels to locate anomalies (**oracle diagnostics**, not methods).
Needs cells 0.1-1.2, 2.1, 3.1, 4.1, 5.1, 5.2, 6.1, 7.1, 9.1 and 14.1. About 1.5 h (15.2), 0.5 h (15.3),
1.5 h (15.4) on a T4.

### 15.1. Helpers

In [ ]:
M10_SIGNALS = ['oe', 'residual', 'combined', 'error']

def window_patch_labels(label, starts, window):
    """Per window and patch: 1 if the patch holds an anomalous point; valid = patch outside the front padding."""
    n = len(label)
    idx = starts[:, None] + np.arange(window)[None, :]
    valid = idx >= 0
    lab = np.where(valid, np.asarray(label)[np.clip(idx, 0, n - 1)], 0)
    shape = (len(starts), window // PATCH, PATCH)
    return lab.reshape(shape).max(-1), valid.reshape(shape).all(-1)

def patch_error(x, reconstruction):
    """Mean |x - reconstruction| per patch: (B, T) -> (B, N)."""
    B, T = x.shape
    return (x - reconstruction).abs().reshape(B, T // PATCH, PATCH).mean(-1)

def auc(positive, negative):
    """Probability that a random positive value exceeds a random negative one (ties count 1/2)."""
    positive, negative = np.asarray(positive), np.asarray(negative)
    if len(positive) == 0 or len(negative) == 0:
        return np.nan
    ranks = pd.Series(np.r_[positive, negative]).rank().to_numpy()
    return (ranks[:len(positive)].sum() - len(positive) * (len(positive) + 1) / 2) / (len(positive) * len(negative))

@torch.no_grad()
def mask_experiments(model, series, rng, max_windows=300, centres=2):
    """H2-A and H2-B on test windows that contain anomalies. Returns patch-level rows (H2-A) and centre rows (H2-B)."""
    model.eval()
    window, stride = series['window'], series['period']
    windows, pad = make_windows(series['test'], window, stride)
    starts = np.arange(len(windows)) * stride - pad
    lab, valid = window_patch_labels(series['label'], starts, window)
    N = window // PATCH
    candidates = np.where((lab * valid).any(1))[0]
    if len(candidates) > max_windows:
        candidates = np.sort(rng.choice(candidates, max_windows, replace=False))
    rows_a, rows_b = [], []
    for w in candidates:
        x = torch.from_numpy(windows[w:w + 1]).to(DEVICE)
        oe = torch.sigmoid(model.classifier(x))
        combined = (oe + torch.sigmoid(model.classifier.residual_logit(x, model.mae(x, oe)))) / 2
        anomalous = np.where((lab[w] == 1) & valid[w])[0]
        normal = np.where((lab[w] == 0) & valid[w])[0]
        if len(normal) == 0:
            continue
        masks, tags = [combined], ['soft']
        # H2-A: the anomalous patches and as many random normal patches, hard-masked together
        chosen = rng.choice(normal, min(len(anomalous), len(normal)), replace=False)
        hard = combined.clone()
        hard[0, np.r_[anomalous, chosen], 0] = 1.0
        masks.append(hard)
        tags.append('hard')
        # H2-B: centres with two neighbours of the same kind; span vs scattered, 3 hard-masked patches each
        plans = []
        for kind, pool in [('anomalous', anomalous), ('normal', normal)]:
            kind_set = set(pool)
            centre_ok = [j for j in pool if j - 1 in kind_set and j + 1 in kind_set]
            for j in rng.permutation(centre_ok)[:centres]:
                far = [k for k in normal if abs(k - j) >= 3]
                if len(far) < 2:
                    continue
                for config, patches in [('span', [j - 1, j, j + 1]), ('scattered', [j, *rng.choice(far, 2, replace=False)])]:
                    m = combined.clone()
                    m[0, patches, 0] = 1.0
                    masks.append(m)
                    tags.append((kind, int(j), config))
        prob = torch.cat(masks)
        errors = patch_error(x.expand(len(masks), -1), model.mae(x.expand(len(masks), -1), prob)).cpu().numpy()
        soft, hard_err = errors[0], errors[1]
        level = combined[0, :, 0].cpu().numpy()
        for j in anomalous:
            rows_a.append(('anomalous', soft[j], hard_err[j], level[j]))
        for j in chosen:
            rows_a.append(('normal', soft[j], hard_err[j], level[j]))
        by_centre = {}
        for tag, err in zip(tags[2:], errors[2:]):
            kind, j, config = tag
            by_centre.setdefault((kind, j), {})[config] = err[j]
        for (kind, j), e in by_centre.items():
            if len(e) == 2:
                rows_b.append((kind, soft[j], e['span'], e['scattered']))
    return rows_a, rows_b

### 15.2. H1-A, H2-A, H2-B on the 192 code-mode models (re-run to continue)

One row per long real event (OE, residual, `A_c` and reconstruction-error percentiles in the middle third and the
edge thirds; `results/m10_events_<SESSION_TAG>.csv`) and one row per model with the H2 summaries
(`results/m10_models_<SESSION_TAG>.csv`). Columns are fixed so that every model writes the same layout.

In [ ]:
M10_EVENTS = RESULTS_DIR / f'm10_events_{SESSION_TAG}.csv'
M10_MODELS = RESULTS_DIR / f'm10_models_{SESSION_TAG}.csv'
M10_EVENT_COLUMNS = (['run_id', 'group', 'seed', 'family', 'event', 'length', 'length_over_period']
                     + [f'{s}_{k}' for s in M10_SIGNALS for k in ['peak', 'mean', 'middle', 'edges']])
M10_MODEL_COLUMNS = ['run_id', 'group', 'seed', 'family', 'h2a_anomalous', 'h2a_normal', 'auc_soft', 'auc_hard',
                     'mask_level_anomalous', 'mask_level_normal', 'err_anomalous_soft', 'err_anomalous_hard',
                     'err_normal_soft', 'err_normal_hard', 'h2b_anomalous', 'h2b_normal',
                     'gain_anomalous', 'gain_normal', 'seconds']
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))

def read_m10(kind):
    files = sorted(RESULTS_DIR.glob(f'm10_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_m10('models')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if (int(r.group), int(r.seed)) not in done]
print(len(code_runs), 'code-mode models,', len(todo), 'to analyse')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    parts = predict_components(model, series['test'], series['window'], series['period'])
    signals = {'oe': parts['oe'], 'residual': parts['residual'], 'combined': parts['combined'],
               'error': np.abs(series['test'] - parts['reconstruction'])}
    label, period = np.asarray(series['label']), series['period']
    normal = (label == 0) & (np.arange(len(label)) >= period)
    pcts = {s: percentile_against(v[normal]) for s, v in signals.items()}
    base = dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group])
    rows = []
    for k, (a, b) in enumerate(segments(label)):
        if b - a < 3 * PATCH:
            continue
        row = dict(base, event=k, length=int(b - a), length_over_period=(b - a) / period)
        for s in M10_SIGNALS:
            row.update({f'{s}_{key}': v for key, v in event_stats(signals[s], a, b, pcts[s]).items()})
        rows.append(row)
    if rows:
        pd.DataFrame(rows).reindex(columns=M10_EVENT_COLUMNS).to_csv(M10_EVENTS, mode='a', header=not M10_EVENTS.exists(), index=False)

    rows_a, rows_b = mask_experiments(model, series, np.random.default_rng(2026 + group + seed))
    a = pd.DataFrame(rows_a, columns=['kind', 'soft', 'hard', 'level'])
    b = pd.DataFrame(rows_b, columns=['kind', 'soft', 'span', 'scattered'])
    is_anom, is_norm = a['kind'] == 'anomalous', a['kind'] == 'normal'
    scale = a.loc[is_norm, 'soft'].mean() if is_norm.any() else np.nan     # normal error level of this model
    gain = (b['span'] - b['scattered']) / scale
    summary = dict(base, h2a_anomalous=int(is_anom.sum()), h2a_normal=int(is_norm.sum()),
                   auc_soft=auc(a.loc[is_anom, 'soft'], a.loc[is_norm, 'soft']),
                   auc_hard=auc(a.loc[is_anom, 'hard'], a.loc[is_norm, 'hard']),
                   mask_level_anomalous=a.loc[is_anom, 'level'].mean(), mask_level_normal=a.loc[is_norm, 'level'].mean(),
                   err_anomalous_soft=a.loc[is_anom, 'soft'].mean(), err_anomalous_hard=a.loc[is_anom, 'hard'].mean(),
                   err_normal_soft=a.loc[is_norm, 'soft'].mean(), err_normal_hard=a.loc[is_norm, 'hard'].mean(),
                   h2b_anomalous=int((b['kind'] == 'anomalous').sum()), h2b_normal=int((b['kind'] == 'normal').sum()),
                   gain_anomalous=gain[b['kind'] == 'anomalous'].mean(), gain_normal=gain[b['kind'] == 'normal'].mean(),
                   seconds=round(time.time() - start, 1))
    pd.DataFrame([summary]).reindex(columns=M10_MODEL_COLUMNS).to_csv(M10_MODELS, mode='a', header=not M10_MODELS.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} long events | AUC soft {summary['auc_soft']:.3f} "
          f"hard {summary['auc_hard']:.3f} | {summary['seconds']:.0f} s", flush=True)

### 15.3. H1-B: abrupt vs smooth boundaries (stress test, re-run to continue)

Series: per family, the first two in manifest order with a normal stretch long enough (TODS may have none); the 3
seeds; 4 positions drawn once with `default_rng(2026 + group)`. Segment of 2 periods; the change is multiplied by an
envelope that is 1 everywhere (abrupt) or rises and falls with a raised cosine over half a period at each end
(smooth). The middle period (**plateau**) is identical in both versions; the energy ratio smooth / abrupt is
reported. Changes: `level` (+a training SD) and `noise` (a training SD times a fixed Gaussian draw), a = 1 or 3.

In [ ]:
M10_STRESS = RESULTS_DIR / f'm10_stress_{SESSION_TAG}.csv'
H1B_TYPES, H1B_AMPLITUDES, H1B_BOUNDARIES = ['level', 'noise'], [1, 3], ['abrupt', 'smooth']
H1B_SIGNALS = ['oe', 'classify', 'reconstruct', 'full']
M10_STRESS_COLUMNS = (['group', 'seed', 'family', 'type', 'amplitude', 'boundary', 'position', 'energy_ratio']
                      + [f'{s}_{k}' for s in H1B_SIGNALS for k in ['plateau', 'ramps', 'peak']])

def envelope(length, ramp, boundary):
    w = np.ones(length)
    if boundary == 'smooth':
        rise = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, ramp))
        w[:ramp], w[-ramp:] = rise, rise[::-1]
    return w

def h1b_positions(series):
    period, window, n = series['period'], series['window'], len(series['test'])
    length = 2 * period
    blocked = np.zeros(n, bool)
    for a, b in segments(series['label']):
        blocked[max(0, a - window - length):b + window] = True
    candidates = [s for s in range(max(2 * window, period), n - length - 2 * window) if not blocked[s]]
    if not candidates:
        return []
    rng = np.random.default_rng(2026 + series['group'])
    return sorted(int(p) for p in rng.choice(candidates, size=min(4, len(candidates)), replace=False))

H1B_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    chosen = [int(g) for g in d['group'] if h1b_positions(load_series(int(g)))][:2]
    print(f'{family}: H1-B series {chosen}' + ('' if chosen else ' (no normal stretch long enough)'))
    H1B_GROUPS += chosen

done = read_m10('stress')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in H1B_GROUPS and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'models for H1-B')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    period, window = series['period'], series['window']
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    base_signals, train_parts = model_signals(model, series)
    label = np.asarray(series['label'])
    keep = np.ones(len(label), bool)
    for a, b in segments(label):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    pcts = {s: percentile_against(base_signals[s][keep]) for s in H1B_SIGNALS}
    sd = series['train'].std()
    length, ramp = 2 * period, max(2, period // 2)
    rows = []
    for position in h1b_positions(series):
        noise = np.random.default_rng(position).standard_normal(length)
        for kind in H1B_TYPES:
            for amp in H1B_AMPLITUDES:
                change = amp * sd * (np.ones(length) if kind == 'level' else noise)
                energies = {}
                for boundary in H1B_BOUNDARIES:
                    w = envelope(length, ramp, boundary)
                    energies[boundary] = float(((w * change) ** 2).sum())
                    values = series['test'].copy()
                    values[position:position + length] += w * change
                    a, b = position - 2 * window, position + length + 2 * window
                    signals, _ = model_signals(model, series, values=values[a:b], train_parts=train_parts)
                    s0 = position - a
                    row = dict(group=group, seed=seed, family=family_of[group], type=kind, amplitude=amp,
                               boundary=boundary, position=position)
                    for s in H1B_SIGNALS:
                        p = pcts[s](signals[s][s0:s0 + length])
                        row[f'{s}_plateau'] = float(p[ramp:length - ramp].mean())
                        row[f'{s}_ramps'] = float(np.r_[p[:ramp], p[length - ramp:]].mean())
                        row[f'{s}_peak'] = float(pcts[s](np.array([signals[s][s0:s0 + length].max()]))[0])
                    rows.append(row)
                for row in rows[-2:]:
                    row['energy_ratio'] = energies['smooth'] / energies['abrupt']
    if rows:
        pd.DataFrame(rows).reindex(columns=M10_STRESS_COLUMNS).to_csv(M10_STRESS, mode='a', header=not M10_STRESS.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} injections', flush=True)

### 15.4. Gradient audit: does the reconstruction loss pull the classifier against classification? (re-run to continue)

Code mode, 9 MGAB series (where the classifier carries the score, M7/M6-C), seed 46 (`M10_SEEDS`). Two variants:

* `joint`: the authors' training (identical to `train_model(faithful=True)`; checked below);
* `detached`: the mask probability is detached, so the reconstruction loss no longer reaches the classifier through
  the mask (the residual-classifier path is unchanged).

Every 10 epochs, on one fixed distorted training batch (drawn with a separate random state, so training itself is
not changed), the gradients of the classification term and of `lambda x` reconstruction term with respect to the
classifier parameters: cosine and norms. Then both models are scored as in section 7.

In [ ]:
M10_GRADIENT = RESULTS_DIR / f'm10_gradient_{SESSION_TAG}.csv'
M10_PROBES = RESULTS_DIR / f'm10_probes_{SESSION_TAG}.csv'
M10_GROUPS = manifest64.query("family == 'MGAB'")['group'].tolist()
M10_SEEDS = [46]
M10_GRADIENT_COLUMNS = ['group', 'seed', 'variant', 'seconds', 'cosine_mean', 'cosine_negative_share',
                        'norm_classify_mean', 'norm_reconstruct_mean'] + METRIC_KEYS

class DetachedMaskCoAD(CoADReimpl):
    """CoAD with the mask probability detached (same parameters and initialisation as CoADReimpl)."""

    def forward(self, x):
        logit = self.classifier(x)
        x_rec = self.mae(x, torch.sigmoid(logit).detach())
        return logit, self.classifier.residual_logit(x, x_rec), x_rec

def rng_state():
    return (random.getstate(), np.random.get_state(), torch.get_rng_state(),
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(state):
    random.setstate(state[0]); np.random.set_state(state[1]); torch.set_rng_state(state[2])
    if state[3] is not None:
        torch.cuda.set_rng_state_all(state[3])

def gradient_probe(model, batch):
    """Cosine and norms of the classification and weighted reconstruction gradients on the classifier parameters."""
    state = rng_state()
    model.eval()                                    # no dropout during the probe
    shared = [p for p in model.classifier.parameters() if p.requires_grad]
    x, labels, target = batch
    flat = []
    with torch.backends.cudnn.flags(enabled=False):  # cuDNN allows the GRU backward pass only in training mode
        _, classify, recon = coad_loss(model, x, labels, target, True)
        for loss in [classify, LAMBDA * recon]:
            grads = torch.autograd.grad(loss, shared, retain_graph=True, allow_unused=True)
            flat.append(torch.cat([(g if g is not None else torch.zeros_like(p)).reshape(-1) for g, p in zip(grads, shared)]))
    model.train()
    restore_rng(state)
    nc, nr = flat[0].norm().item(), flat[1].norm().item()
    cosine = (flat[0] @ flat[1]).item() / (nc * nr) if nc > 0 and nr > 0 else np.nan
    return {'cosine': cosine, 'norm_classify': nc, 'norm_reconstruct': nr}

def train_probe(series, seed, detached=False, epochs=None, probe_every=10):
    """Same loop as train_model(faithful=True), plus gradient probes that do not touch the training random stream."""
    epochs = EPOCHS if epochs is None else epochs
    set_seed(seed)
    model = (DetachedMaskCoAD if detached else CoADReimpl)(faithful=True).to(DEVICE)
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99)
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    state = rng_state()
    set_seed(12345)
    probe_batch = training_batch(torch.from_numpy(windows[:BATCH_SIZE]).to(DEVICE), series['period'], True)
    restore_rng(state)
    probes = []
    for epoch in range(epochs):
        model.train()
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                loss = coad_loss(model, x, labels, target, True)[0]
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        if probe_every and (epoch % probe_every == 0 or epoch == epochs - 1):
            probes.append(dict(epoch=epoch, **gradient_probe(model, probe_batch)))
    return model, pd.DataFrame(probes)

# Check: without detaching, train_probe trains exactly like train_model (2 epochs, group 296)
s296 = load_series(296)
reference, _ = train_model(s296, faithful=True, seed=46, epochs=2, log_every=0)
probed, _ = train_probe(s296, seed=46, epochs=2, probe_every=1)
gap = max((a - b).abs().max().item() for a, b in zip(reference.state_dict().values(), probed.state_dict().values()))
print(f'train_probe vs train_model, max parameter difference after 2 epochs: {gap:.1e}')
assert gap < 1e-3

done = read_m10('gradient')
done = set(zip(done['group'].astype(int), done['seed'].astype(int), done['variant'])) if len(done) else set()
todo = [(g, s, v) for s in M10_SEEDS for g in M10_GROUPS for v in ['joint', 'detached'] if (g, s, v) not in done]
print(len(todo), 'training runs to do')
session_start = time.time()
for i, (group, seed, variant) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    series = load_series(group)
    model, probes = train_probe(series, seed, detached=(variant == 'detached'))
    metrics = score_metrics(anomaly_score(model, series), series)
    row = dict(group=group, seed=seed, variant=variant, seconds=round(time.time() - start, 1),
               cosine_mean=probes['cosine'].mean(), cosine_negative_share=(probes['cosine'] < 0).mean(),
               norm_classify_mean=probes['norm_classify'].mean(), norm_reconstruct_mean=probes['norm_reconstruct'].mean(),
               **metrics)
    pd.DataFrame([row]).reindex(columns=M10_GRADIENT_COLUMNS).to_csv(M10_GRADIENT, mode='a', header=not M10_GRADIENT.exists(), index=False)
    probes.assign(group=group, seed=seed, variant=variant).to_csv(M10_PROBES, mode='a', header=not M10_PROBES.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} {variant}: VUS-PR {100 * row['VUS_PR']:.2f} | cosine {row['cosine_mean']:+.3f} "
          f"| {row['seconds']:.0f} s", flush=True)

### 15.5. Results and decisions

Each hypothesis is reported against its prediction, then the decision table of `04.md` (section 9) is applied.

In [ ]:
verdicts = {}

def paired(diff, alternative='two-sided'):
    diff = pd.Series(diff).dropna()
    return diff.mean(), len(diff), (wilcoxon(diff, alternative=alternative).pvalue if (diff != 0).any() else np.nan)

# H1-A: OE in the middle vs the edges of real long events
ev = read_m10('events')
if len(ev):
    ev = ev.drop_duplicates(['group', 'seed', 'event'], keep='last')
    rows = []
    for s in M10_SIGNALS:
        m, n, p = paired(ev[f'{s}_middle'] - ev[f'{s}_edges'])
        rows.append({'signal': s, 'middle - edges': m, 'events': n, 'p': p})
    display(pd.DataFrame(rows).set_index('signal').round(4))
    print('OE middle - edges by family:')
    display(ev.assign(d=ev['oe_middle'] - ev['oe_edges']).groupby('family')['d'].agg(['mean', 'size']).round(3))
    m, n, p = paired(ev['oe_middle'] - ev['oe_edges'], 'less')
    verdicts['H1-A'] = m <= -0.05 and p < 0.05
    print(f'H1-A: OE middle - edges = {m:+.3f} over {n} events (one-sided p = {p:.1e}) -> '
          + ('SUPPORTED' if verdicts['H1-A'] else 'NOT SUPPORTED'))

# H1-B: abrupt vs smooth boundaries, plateau
st = read_m10('stress')
if len(st):
    print('Plateau percentile by type, amplitude and boundary:')
    display(st.groupby(['type', 'amplitude', 'boundary'])[[f'{s}_plateau' for s in H1B_SIGNALS] + ['energy_ratio']].mean().round(3))
    wide = st.pivot_table(index=['group', 'seed', 'position', 'type', 'amplitude'], columns='boundary',
                          values=[f'{s}_plateau' for s in H1B_SIGNALS])
    rows = []
    for s in H1B_SIGNALS:
        m, n, p = paired(wide[(f'{s}_plateau', 'abrupt')] - wide[(f'{s}_plateau', 'smooth')])
        rows.append({'signal': s, 'abrupt - smooth (plateau)': m, 'pairs': n, 'p': p})
    display(pd.DataFrame(rows).set_index('signal').round(4))
    m, n, p = paired(wide[('oe_plateau', 'abrupt')] - wide[('oe_plateau', 'smooth')], 'greater')
    verdicts['H1-B'] = m >= 0.05 and p < 0.05
    print(f'H1-B: OE plateau abrupt - smooth = {m:+.3f} over {n} pairs (one-sided p = {p:.1e}) -> '
          + ('SUPPORTED' if verdicts['H1-B'] else 'NOT SUPPORTED'))

# H2-A and H2-B
mo = read_m10('models')
if len(mo):
    mo = mo.drop_duplicates(['group', 'seed'], keep='last')
    print('Per family, mean over models:')
    display(mo.groupby('family')[['auc_soft', 'auc_hard', 'mask_level_anomalous', 'mask_level_normal',
                                  'err_anomalous_soft', 'err_anomalous_hard', 'err_normal_soft', 'err_normal_hard',
                                  'gain_anomalous', 'gain_normal']].mean().round(3))
    m, n, p = paired(mo['auc_hard'] - mo['auc_soft'], 'greater')
    verdicts['H2-A'] = m >= 0.02 and p < 0.05
    print(f'H2-A: AUC hard - soft = {m:+.3f} over {n} models (one-sided p = {p:.1e}) -> '
          + ('SUPPORTED' if verdicts['H2-A'] else 'NOT SUPPORTED'))
    m, n, p = paired(mo['gain_anomalous'] - mo['gain_normal'], 'greater')
    verdicts['H2-B'] = m > 0 and p < 0.05
    print(f'H2-B: span gain at anomalous minus normal centres = {m:+.3f} (in units of the normal error) over {n} models '
          f'(one-sided p = {p:.1e}) -> ' + ('SUPPORTED' if verdicts['H2-B'] else 'NOT SUPPORTED'))

# Gradient audit
gr = read_m10('gradient')
if len(gr):
    gr = gr.drop_duplicates(['group', 'seed', 'variant'], keep='last')
    display(gr.groupby('variant')[['cosine_mean', 'cosine_negative_share', 'norm_classify_mean', 'norm_reconstruct_mean']
                                  + list(METRICS4)].mean().round(4))
    wide = gr.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
    if {'joint', 'detached'} <= set(wide.columns):
        m, n, p = paired(100 * (wide['detached'] - wide['joint']), 'greater')
        cos = gr.loc[gr['variant'] == 'joint', 'cosine_mean'].mean()
        verdicts['Gradient conflict'] = cos < 0 and p < 0.05 and m > 0
        print(f'Gradient: mean cosine (joint) = {cos:+.3f}; VUS-PR detached - joint = {m:+.2f} over {n} series '
              f'(one-sided p = {p:.2f}) -> ' + ('SUPPORTED' if verdicts['Gradient conflict'] else 'NOT SUPPORTED'))

print('\nVerdicts:', verdicts)
pd.Series(verdicts, name='supported').to_csv(REPORTS / 'm10_verdicts.csv')

## 16. M11: normal-referenced evidence fusion (`05-coad-cai-tien.md`)

**Question.** M9 showed that CoAD's intermediate signals carry evidence that the final score loses (the raw
reconstruction error finds 53% of the events OE misses, the final score 7%), and that rescaling the two branches
(`calibrated`) gains 2.4 VUS-PR. M10 showed the core mechanisms are sound. Can a better **fusion of the evidence**
do more, and does it hold on series **not used to design it**?

**Code facts behind the variants** (commit `3286813`, checked): `get_error_score` turns each source into
`|x - mean_train(x)|`, a **two-sided** deviation, for the reconstruction error **and** for the classifier
probability (`if_normalize=True` by default, used by the TSB-AD pipeline); the classifier is trained with
`pos_weight = 2`, so its sigmoid is not a calibrated probability.

**Protocol.**

1. **Split before any score is computed** (16.1): the 64 series are split, per family, into a *development* half
   and a *confirmation* half with `default_rng(2026)`; written once to `eda/m11_split.csv`.
2. **Development** (16.2-16.3): every variant is computed on the development models (3 seeds) and ranked by AUC-PR
   (fast, rank-based); the weights of B3 are chosen there and **locked** (`reports/m11_locked.json`).
3. **Confirmation** (16.4-16.5): only the locked variants, full evaluator, compared with CoAD (B0) and `calibrated`
   (B1) on the confirmation series, next to Table 1.

All variants use the same smoothing (moving average over one period, first period zeroed) except B5, whose temporal
processing is the point. Calibration uses the **training part** (the models were trained on all of it, so no unused
normal data exists without retraining; training scores are slightly lower than unseen normal ones, which makes
test values look somewhat rarer than they are).

| Code | Variant | Change |
|---|---|---|
| B0 | CoAD | authors' score (`(|e - mu| + |c - mu|) / 2`, smoothed) |
| B1 | `calibrated` | each branch divided by its SD on the training part (part 14) |
| B1b | one-sided | as B1 with `max(0, x - mu)` instead of `|x - mu|` |
| B2 | tail | per source (OE `A`, residual `A^r`, reconstruction error), rarity under the training part: `z = -log((1 + #{train >= v}) / (n + 1))` up to the 95th training percentile `u`, exponential tail above it (`z(u) + (v - u) / beta`, beta = mean excess over `u`; avoids saturation); mean of the three |
| B3 | weighted tail | as B2 with non-negative weights in {0, 1, 2} per source, chosen on development |
| B4 | context tail | as B2, but the error is ranked against training errors **of the same context** (4 bins of the local SD of the input over the previous period) |
| B5 | multi-length | from the B2 evidence: moving means over 1, period / 4 and period points, each calibrated on the training part; `0.5 x` instantaneous + `0.5 x` mean over lengths |

**Prediction written before running (primary):** on the confirmation series, the locked B3 has a higher VUS-PR than
B1 (`calibrated`), one-sided Wilcoxon p < 0.05 over series (mean over seeds). Secondary comparisons are reported
with Holm correction. If no variant beats B1, the conclusion is that rescaling already removes most of the fusion
bottleneck (stopping rule of `05-coad-cai-tien.md`).

**How to run.** Cell **16.0** runs every definition cell needed (0.1-1.2, 2.1, 3.1, 4.1, 5.1, 5.2, 6.1, 7.1, 9.1, 14.1,
16.1) in one go. One session: 16.0, then 16.2 -> 16.7 in order (about 3.5 h on a T4). Three sessions at the same
time (three notebook copies, three GPU runtimes), A and B share the main part, C runs the spike test:

| Step | Notebook A (`SETUP_TAG = 'gpu-a'`) | Notebook B (`SETUP_TAG = 'gpu-b'`) | Notebook C (`SETUP_TAG = 'gpu-c'`) |
|---|---|---|---|
| 1 | 16.0, 16.2 with `M11_DEV_SEEDS = [46, 47]` | 16.0, 16.2 with `M11_DEV_SEEDS = [48]` | 16.0, 16.6, 16.7 (spike test) |
| 2 | 16.3 (lock), in whichever of A / B finishes 16.2 last | | |
| 3 | 16.4 with `M11_CONFIRM_SEEDS = [46]` | 16.4 with `M11_CONFIRM_SEEDS = [47]` | 16.4 with `M11_CONFIRM_SEEDS = [48]` |
| 4 | 16.5 when all three are done (any notebook) | | |

16.3 refuses to lock before every development model is done; 16.4 refuses to start before the lock exists.

### 16.1. Split and variants

### 16.0. One-click setup

Runs, in one go, the definition cells part 16 needs: 0.1, 0.2, 1.1, 1.2, 2.1, 3.1, 4.1, 5.1, 5.2, 6.1, 7.1, 9.1, 14.1 and 16.1
(their code is copied here automatically when the notebook is built, so it always matches them). No training, no
loop of earlier parts. Set `SETUP_TAG` first: a different value in each notebook running at the same time.
After it, run only the cells of your session (see the table at the top of part 16).

In [ ]:
# ==== One-click setup for part 16 (generated by build.py from the cells it lists; do not edit)
SETUP_TAG = 'gpu-a'          # 'gpu-a' in notebook A, 'gpu-b' in notebook B

# ---- cell 0.1
from google.colab import drive
drive.mount('/content/drive')

import json, os, random, shutil, subprocess, sys, time, zipfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

STORE = Path('/content/drive/MyDrive/CoAD_project')
REPO = Path('/content/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)

CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_reimpl.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    assert install.returncode == 0, 'Installation failed; inspect env/install_reimpl.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_reimpl.log'
print('Commit', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip(),
      '| packages OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

# ---- cell 0.2
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
assert len(list(RAW.iterdir())) == 870, 'raw/ must contain exactly 870 CSVs'

os.chdir(REPO)                       # load_dataset reads the relative path dataset/TSB-AD/raw
sys.path.insert(0, str(REPO))
from toolkit.utils import load_dataset
from evaluation.evaluator import evaluate
from model.frequency import time_to_timefreq       # reference for test T2 only
from toolkit import distort as reference_distort   # reference for test T1 only
from model.patch_soft_mask import PatchFrequencyMask  # reference for test T4 only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORTS = STORE / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Data: 870 files | device:', DEVICE)

# ---- cell 1.1
FAITHFUL_TO_CODE = False   # False: follow the paper (main version); True: follow the authors' code (D1-D21)
SEEDS = [46, 47, 48]
PATCH = 8                  # P
WINDOW_MULTIPLE = 4        # T = 4 x dominant period
N_FFT = 4                  # K, STFT size
LAMBDA = 10                # weight of the reconstruction loss
LR = 0.002
EPOCHS = 300
BATCH_SIZE = 128

def use_code_details(faithful=None):
    """Per-call override of FAITHFUL_TO_CODE (None = use the global switch)."""
    return FAITHFUL_TO_CODE if faithful is None else faithful

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_series(group):
    """One TSB-AD series (min-max scaled on train) plus the window settings of the authors' code."""
    train, test, label, period = [values[0] for values in load_dataset('TSB-AD', [str(group)])]
    multiple = WINDOW_MULTIPLE if len(train) // period > WINDOW_MULTIPLE else len(train) // period
    window = (period * multiple) // PATCH * PATCH
    period = min(period, window)
    n = len(train)
    train_stride = 2 if n < 1000 else 6 if n < 10000 else 10 if n < 60000 else PATCH if n < 100000 else period
    return dict(group=group, train=train, test=test, label=label,
                period=period, window=window, train_stride=train_stride)

print('FAITHFUL_TO_CODE =', FAITHFUL_TO_CODE, '| seeds', SEEDS)

# ---- cell 1.2
manifest64 = pd.read_csv(STORE / 'eda' / 'manifest_tsbad64.csv')
reimpl_manifest_file = STORE / 'eda' / 'manifest_reimpl.csv'
if not reimpl_manifest_file.exists():
    rng = np.random.default_rng(2026)
    chosen = [236]
    for family in ['MGAB', 'SVDB', 'IOPS', 'TODS']:
        chosen.append(int(rng.choice(sorted(manifest64.loc[manifest64['family'] == family, 'group']))))
    rows = manifest64.set_index('group').loc[chosen].reset_index()
    rows['selection_reason'] = ['Pilot series of the reproduction'] + ['default_rng(2026), one per family'] * 4
    rows.to_csv(reimpl_manifest_file, index=False)
REIMPL_MANIFEST = pd.read_csv(reimpl_manifest_file)
REIMPL_GROUPS = REIMPL_MANIFEST['group'].tolist()

overview = []
for g, family in zip(REIMPL_GROUPS, REIMPL_MANIFEST['family']):
    s = load_series(g)
    overview.append({'group': g, 'family': family, 'n_train': len(s['train']), 'n_test': len(s['test']),
                     'anomaly_%': round(100 * s['label'].mean(), 2), 'period': s['period'],
                     'window': s['window'], 'train_stride': s['train_stride']})
display(pd.DataFrame(overview))

# ---- cell 2.1
DISTORTIONS = ['uniform', 'scale', 'jitter', 'mirror']

def distort(x, period, kind, faithful=None):
    """x: clean windows (B, T, 1). Returns (distorted copy, point labels (B, T))."""
    faithful = use_code_details(faithful)
    x = x.clone()
    B, T, _ = x.shape
    n = int(period * np.random.uniform(0.1, 1))           # segment length, one draw per batch
    if n == 0:
        n = 2 if kind == 'uniform' else 1
    if n >= T:
        n = T - 2
    start = torch.randint(0, T - n, (B, 1))               # segment start, one per window
    if faithful and kind == 'mirror':
        torch.randint(0, T - n, (B, 1))                   # unused draw of the authors' mirror_flip: keeps the RNG stream aligned
    rows = torch.arange(B)[:, None]

    if kind == 'scale':
        factor = np.random.randint(2, 5)
        if faithful or np.random.rand() < 0.5:            # lengthen: first n / factor points, each repeated
            n = n // factor * factor
            source = start + torch.arange(n) // factor
        else:                                             # shorten: every factor-th point of a longer span
            source = (start + torch.arange(n) * factor).clamp(max=T - 1)
    cols = start + torch.arange(n)
    segment = x[rows, cols]                               # (B, n, 1)

    if kind == 'uniform':
        if faithful:
            x[rows, cols] = x[rows, start]
        else:
            low, high = segment.min(1, keepdim=True).values, segment.max(1, keepdim=True).values
            x[rows, cols] = low + (high - low) * torch.rand(B, 1, 1, device=x.device)
    elif kind == 'scale':
        x[rows, cols] = x[rows, source]
    elif kind == 'jitter':
        if faithful:
            std = 0.05 * (x.max(1, keepdim=True).values - x.min(1, keepdim=True).values)
        else:
            std = 0.1 ** 0.5
        x[rows, cols] = segment + torch.randn(B, n, 1, device=x.device) * std
    elif kind == 'mirror':
        if faithful or np.random.rand() < 0.5:            # across the y-axis: reverse time
            x[rows, cols] = torch.flip(segment, [1])
        else:                                             # across the x-axis: mirror values inside the segment range
            x[rows, cols] = segment.min(1, keepdim=True).values + segment.max(1, keepdim=True).values - segment
    else:
        raise ValueError(f'unknown distortion {kind}')

    labels = torch.zeros(B, T, device=x.device)
    labels[rows, cols] = 1
    return x, labels

# ---- cell 3.1
def stft_features(x):
    """x: (B, T) -> (B, T, 2 * bins): real and imaginary part of every bin, one frame per time point."""
    spectrum = torch.stft(x, N_FFT, hop_length=N_FFT // 4, window=torch.hann_window(N_FFT, device=x.device),
                          normalized=True, onesided=True, return_complex=True)
    spectrum = torch.view_as_real(spectrum)            # (B, bins, frames, 2)
    spectrum = spectrum.permute(0, 2, 1, 3).flatten(2)  # (B, frames, bins * 2)
    return spectrum[:, :x.shape[1]]                     # frames = T + 1 (centred STFT); keep T

def make_patches(x, x_freq):
    """x: (B, T), x_freq: (B, T, F) -> time patches (B, N, P) and frequency patches (B, N, P * F)."""
    B, T = x.shape
    return x.reshape(B, T // PATCH, PATCH), x_freq.reshape(B, T // PATCH, -1)

# ---- cell 4.1
def head(in_size, hidden, faithful):
    """Patch classifier: linear in the paper (Eq. 2, 3, 8), two-layer MLP in the code (D4)."""
    if faithful:
        return nn.Sequential(nn.Linear(in_size, hidden), nn.SELU(), nn.Linear(hidden, 1, bias=False))
    return nn.Sequential(nn.Linear(in_size, 1))

def gru(size, faithful):
    """Bi-GRU encoder (Section 3.2): 2 layers and dropout 0.1 in the code, 3 layers in the paper (D1, D2)."""
    return nn.GRU(size, size, num_layers=2 if faithful else 3, batch_first=True, bidirectional=True,
                  dropout=0.1 if faithful else 0.0)

def init_linear(module):
    """Xavier-uniform weights and zero biases for every linear layer, as in the authors' code."""
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


class TimeFreqClassifier(nn.Module):
    """Module 3 (Eq. 1-4) and the residual classifier of module 5 (Eq. 7-8)."""

    def __init__(self, faithful):
        super().__init__()
        self.faithful = faithful
        freq_size = PATCH * 2 * (N_FFT // 2 + 1)                  # 8 points x 3 bins x (real, imaginary) = 48
        hidden = 3 * freq_size if faithful else 24                # D5
        activation = [nn.SELU()] if faithful else []              # D18
        self.proj_freq = nn.Sequential(nn.Linear(freq_size, hidden), *activation)   # W_f^p
        self.proj_time = nn.Sequential(nn.Linear(PATCH, hidden), *activation)       # W_t^p
        self.encoder = gru(hidden, faithful)                      # shared by both branches (D3)
        self.classifier_time = head(2 * hidden, hidden, faithful)       # W_t (created first, as in the authors' code,
        self.classifier_freq = head(2 * hidden, hidden, faithful)       # W_f  so a seed gives the same initial weights)
        self.classifier_residual = head(2 * hidden, hidden, faithful)   # W_r, shared by both residual branches
        init_linear(self)

    def features(self, x):
        """x: (B, T) -> encoder outputs of the frequency and of the time patches, each (B, N, 2H)."""
        time_patches, freq_patches = make_patches(x, stft_features(x))
        projected = torch.cat([self.proj_freq(freq_patches), self.proj_time(time_patches)])
        encoded, _ = self.encoder(projected)
        return encoded.chunk(2)

    def forward(self, x):
        """Eq. 1-4: logit of A = max(A_f, A_t), (B, N, 1). Max of logits = max of probabilities (sigmoid is increasing)."""
        h_freq, h_time = self.features(x)
        return torch.maximum(self.classifier_freq(h_freq), self.classifier_time(h_time))

    def residual_logit(self, x, x_rec):
        """Eq. 7-8: logit of A^r from the feature difference between the input and its reconstruction."""
        h_freq, h_time = self.features(torch.cat([x, x_rec]))
        (freq_in, freq_rec), (time_in, time_rec) = h_freq.chunk(2), h_time.chunk(2)
        return torch.maximum(self.classifier_residual(freq_in - freq_rec),
                             self.classifier_residual(time_in - time_rec))


class SoftMaskMAE(nn.Module):
    """Module 4 (Eq. 5-6): patch embeddings softly replaced by a learned mask embedding, then reconstructed."""

    def __init__(self, faithful):
        super().__init__()
        hidden = 3 * PATCH                                        # 24 in both modes
        self.proj = nn.Linear(PATCH, hidden)                      # W_m
        self.act = nn.SELU() if faithful else nn.Identity()       # D18
        self.encoder = gru(hidden, faithful)
        self.head = nn.Linear(2 * hidden, PATCH)                  # W_o
        self.mask_embedding = nn.Parameter(torch.randn(hidden))   # E_mask, one vector repeated over patches (D19)
        init_linear(self)

    def forward(self, x, prob):
        """x: (B, T), prob: patch anomaly probability (B, N, 1) -> reconstruction X_r (B, T)."""
        B, T = x.shape
        embedding = self.act(self.proj(x.reshape(B, T // PATCH, PATCH)))
        mixed = prob * self.mask_embedding + (1 - prob) * embedding    # Eq. 5
        encoded, _ = self.encoder(mixed)
        return self.head(encoded).reshape(B, T)                         # Eq. 6


class CoADReimpl(nn.Module):
    def __init__(self, faithful=None):
        super().__init__()
        self.faithful = use_code_details(faithful)
        self.classifier = TimeFreqClassifier(self.faithful)
        self.mae = SoftMaskMAE(self.faithful)

    def forward(self, x):
        """Training pass. x: (B, T) -> logits of A and A^r (B, N, 1), reconstruction X_r (B, T)."""
        logit = self.classifier(x)
        x_rec = self.mae(x, torch.sigmoid(logit))
        return logit, self.classifier.residual_logit(x, x_rec), x_rec

    @torch.no_grad()
    def predict(self, x):
        """Inference (call .eval() first). x: (B, T) -> A_c (B, N, 1) (Eq. 9) and the reconstruction (B, T)."""
        prob = torch.sigmoid(self.classifier(x))
        x_rec = self.mae(x, prob)
        prob = (prob + torch.sigmoid(self.classifier.residual_logit(x, x_rec))) / 2
        if self.faithful:                   # D10: reconstruct again, masked with A_c
            x_rec = self.mae(x, prob)
        return prob, x_rec

    def load_official(self, official):
        """Copy the weights of an authors' PatchFrequencyMask (code mode; test T4)."""
        renamed = {k.replace('detection_net.', 'classifier.').replace('reconstruction_net.', 'mae.'): v
                   for k, v in official.state_dict().items()
                   if k.startswith(('detection_net.', 'reconstruction_net.'))}
        self.load_state_dict(renamed)

# ---- cell 5.1
def make_windows(values, window, stride):
    """values (L,) -> windows (M, window) float32 and the front padding length."""
    L = len(values)
    pad = 0 if stride == 1 or (L - window) % stride == 0 else stride - (L - window) % stride
    padded = np.concatenate([np.zeros(pad), values])
    starts = np.arange(0, len(padded) - window + 1, stride)
    return np.stack([padded[s:s + window] for s in starts]).astype(np.float32), pad

def windows_to_series(window_values, length, stride, pad):
    """window_values (M, window) -> (length,): mean over the overlapping windows, padding removed."""
    total, count = np.zeros(pad + length), np.zeros(pad + length)
    for i, values in enumerate(window_values):
        total[i * stride:i * stride + len(values)] += values
        count[i * stride:i * stride + len(values)] += 1
    return total[pad:] / count[pad:]

def patch_labels(labels):
    """Point labels (B, T) -> patch labels (B, N, 1)."""
    B, T = labels.shape
    return labels.reshape(B, T // PATCH, PATCH).amax(-1, keepdim=True)

def training_batch(clean, period, faithful, code_distortion=None):
    """Clean windows (B, T) -> model input, point labels, reconstruction target.
    code_distortion (M6-C only): distortion details D12-D15 chosen apart from the batch layout D9."""
    code_distortion = faithful if code_distortion is None else code_distortion
    distorted, labels = distort(clean.unsqueeze(-1), period, random.choice(DISTORTIONS), code_distortion)
    distorted = distorted.squeeze(-1)
    if faithful:                                    # D9: clean copy + distorted copy
        return torch.cat([clean, distorted]), torch.cat([torch.zeros_like(labels), labels]), torch.cat([clean, clean])
    return distorted, labels, clean

def coad_loss(model, x, labels, target, code_loss=None):
    """Eq. 10. Returns (total, classification term, reconstruction term).
    code_loss (M6-C only): loss details D6-D8, D20 chosen apart from the model."""
    logit, residual_logit, x_rec = model(x)
    y = patch_labels(labels)
    if (model.faithful if code_loss is None else code_loss):
        pos_weight = torch.tensor([2.0], device=x.device)                                    # D6
        classify = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight) +
                    F.binary_cross_entropy_with_logits(residual_logit, y, pos_weight=pos_weight)) / 2   # D7
        error = (x_rec - target).pow(2).reshape(y.shape[0], y.shape[1], PATCH)
        anomalous = (error * y).sum() / (y.sum() * PATCH + 1e-7)
        normal = (error * (1 - y)).sum() / ((1 - y).sum() * PATCH + 1e-7)
        recon = 0.5 * anomalous + 0.5 * normal                                               # D8
    else:
        prob = (torch.sigmoid(logit) + torch.sigmoid(residual_logit)) / 2                    # A_c, Eq. 9
        classify = F.binary_cross_entropy(prob, y)
        recon = PATCH * F.mse_loss(x_rec, target)      # Eq. 10: squared norm summed over a patch, averaged over patches (D20)
    return classify + LAMBDA * recon, classify, recon

# ---- cell 5.2
def train_model(series, faithful=None, seed=46, epochs=None, log_every=50, groups=None):
    """Train CoADReimpl on the training part of one series. Returns the model and the mean losses per epoch.
    groups (M6-C only): set of code-only detail groups (section 12) applied on top of paper mode."""
    epochs = EPOCHS if epochs is None else epochs
    set_seed(seed)
    if groups is None:
        faithful = use_code_details(faithful)
        code_distortion = faithful
        model = CoADReimpl(faithful).to(DEVICE)
    else:
        faithful, code_distortion = 'training' in groups, 'distortion' in groups
        model = HybridCoAD(groups).to(DEVICE)
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99) if faithful else None   # D16
    use_amp = faithful and DEVICE.type == 'cuda'                                                            # D16
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = []
    for epoch in range(epochs):
        model.train()
        sums = np.zeros(3)
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], faithful, code_distortion)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                losses = coad_loss(model, x, labels, target, faithful)
            scaler.scale(losses[0]).backward()
            scaler.step(optimizer)
            scaler.update()
            sums += [value.item() for value in losses]
        if scheduler:
            scheduler.step()
        history.append(dict(zip(['loss', 'classify', 'recon'], sums / len(loader))))
        if log_every and (epoch + 1) % log_every == 0:
            print(f'  epoch {epoch + 1}/{epochs}: ' + ', '.join(f'{k} {v:.4f}' for k, v in history[-1].items()),
                  flush=True)
    return model, pd.DataFrame(history)

# ---- cell 6.1
@torch.no_grad()
def predict_series(model, values, window, stride):
    """Whole series -> per-point A_c and reconstruction, averaged over overlapping windows."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    probs, recs = [], []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        prob, rec = model.predict(batch.to(DEVICE))
        probs.append(prob.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        recs.append(rec.cpu())
    return (windows_to_series(torch.cat(probs).numpy(), len(values), stride, pad),
            windows_to_series(torch.cat(recs).numpy(), len(values), stride, pad))

def moving_average(score, width):
    """Centred moving average keeping the length (np.convolve 'same', as in the authors' code)."""
    return np.convolve(score, np.ones(width) / width, mode='same')

def anomaly_score(model, series):
    """Per-point anomaly score of the test series."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    if not model.faithful:
        return moving_average(test_prob + np.abs(series['test'] - test_rec), period)    # Eq. 11
    train_prob, train_rec = predict_series(model, series['train'], window, period)      # D11
    recon = np.abs(np.abs(series['test'] - test_rec) - np.abs(series['train'] - train_rec).mean())
    classify = np.abs(test_prob - train_prob.mean())
    recon, classify = moving_average(recon, period), moving_average(classify, period)
    recon[:period] = classify[:period] = 0
    return (recon + classify) / 2

METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']

def score_metrics(score, series):
    """Checks of test T7, then the repository evaluator (values in 0-1)."""
    assert len(score) == len(series['label']), 'score and label lengths differ'
    assert np.isfinite(score).all(), 'score contains NaN or Inf'
    result = evaluate(ground_truth=series['label'], anomaly_scores=score, subsequence=series['period'], mode='test')
    row = {k: float(result.point_wise[k]) for k in METRIC_KEYS[:3]}
    row.update({k: float(result.vus[k]) for k in METRIC_KEYS[3:]})
    return row

def plot_scores(series, scores, path=None):
    """Test series on top, one min-max scaled score per row below; labelled anomalies shaded."""
    fig, axes = plt.subplots(len(scores) + 1, 1, figsize=(14, 1.6 * (len(scores) + 1)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    rows = [('test data', series['test'])] + list(scores.items())
    for ax, (name, values) in zip(axes, rows):
        values = np.asarray(values, dtype=float)
        ax.plot((values - values.min()) / (values.max() - values.min() + 1e-12), lw=0.6)
        ax.fill_between(np.arange(len(values)), 0, 1, where=series['label'] == 1, color='C3', alpha=0.25,
                        transform=ax.get_xaxis_transform())
        ax.set_ylabel(name, rotation=0, ha='right', fontsize=8)
        ax.set_yticks([])
    fig.suptitle(f"group {series['group']}")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    plt.show()

# ---- cell 7.1
RESULTS_DIR = STORE / 'results'
RUNS = STORE / 'runs' / 'reimpl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

SESSION_TAG = SETUP_TAG            # unique per notebook copy running at the same time
RESULTS = RESULTS_DIR / f'reimpl_results_{SESSION_TAG}.csv'
RUN_MODES = ['paper', 'code']
RUN_GROUPS = REIMPL_GROUPS + [g for g in manifest64['group'] if g not in REIMPL_GROUPS]   # 5 chosen series first, then the other 59
RUN_SEEDS = SEEDS
TIME_BUDGET_MIN = 150            # no new run after this many minutes; re-run 7.2 to continue
COLUMNS = ['run_id', 'mode', 'group', 'seed', 'status', 'seconds', 'error'] + METRIC_KEYS

def read_reimpl_results():
    """Rows of every session, oldest run first."""
    files = sorted(RESULTS_DIR.glob('reimpl_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.sort_values('run_id', kind='stable').reset_index(drop=True)

def run_reimpl(mode, group, seed):
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{mode}_{group}_s{seed}"
    out = RUNS / run_id
    out.mkdir(parents=True)
    row = dict(run_id=run_id, mode=mode, group=group, seed=seed)
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=(mode == 'code'), seed=seed, log_every=0)
        score = anomaly_score(model, series)
        metrics = score_metrics(score, series)
        torch.save(model.state_dict(), out / 'model.pt')
        np.save(out / 'scores.npy', score.astype(np.float32))
        history.to_csv(out / 'loss.csv', index=False)
        (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
        row.update(status='success', **metrics)
    except Exception as error:
        row.update(status='failed', error=f'{type(error).__name__}: {error}'[:300])
    row['seconds'] = round(time.time() - start, 1)
    config = dict(row, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patch=PATCH, n_fft=N_FFT, lam=LAMBDA,
                  commit=commit, device=str(DEVICE))
    (out / 'config.json').write_text(json.dumps(config, indent=2, default=str))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

print('Session', SESSION_TAG, '| writes to', RESULTS.name, '|', len(RUN_MODES) * len(RUN_GROUPS) * len(RUN_SEEDS), 'runs planned')

# ---- cell 9.1
from scipy.stats import friedmanchisquare, wilcoxon

METRICS4 = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
BASELINES = ['Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
groups64 = manifest64['group'].tolist()
family_of = manifest64.set_index('group')['family']

reproduced = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))], ignore_index=True)
reproduced = (reproduced.query("status == 'success'").sort_values('run_id')
              .drop_duplicates(['method', 'group'], keep='last'))
reimpl = (read_reimpl_results().query("status == 'success'")
          .drop_duplicates(['mode', 'group', 'seed'], keep='last'))

def metric_matrix(key):
    """64 series x methods, values x100."""
    table = reproduced.pivot(index='group', columns='method', values=key)
    for mode in ['code', 'paper']:
        table[f'CoAD reimpl {mode}'] = reimpl[reimpl['mode'] == mode].groupby('group')[key].mean()
    table = 100 * table.reindex(groups64)[['CoAD', 'CoAD reimpl code', 'CoAD reimpl paper'] + BASELINES]
    assert table.notna().all().all(), f'missing values for {key}: complete the runs first'
    return table

MATRIX = {name: metric_matrix(key) for key, name in METRICS4.items()}
print('64 series x', MATRIX['VUS-PR'].shape[1], 'methods | reimplementation seeds per mode:',
      reimpl.groupby('mode')['seed'].nunique().to_dict())
display(pd.DataFrame({name: m.mean() for name, m in MATRIX.items()}).round(2))

# ---- cell 14.1
@torch.no_grad()
def predict_components(model, values, window, stride):
    """Per-point A (OE), A^r (residual), A_c and the final reconstruction, averaged over overlapping windows.
    Same computation as CoADReimpl.predict (section 4.1), keeping the intermediate tensors."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    parts = {k: [] for k in ['oe', 'residual', 'combined', 'reconstruction']}
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        oe = torch.sigmoid(model.classifier(x))
        first = model.mae(x, oe)
        residual = torch.sigmoid(model.classifier.residual_logit(x, first))
        combined = (oe + residual) / 2
        final = model.mae(x, combined) if model.faithful else first      # D10
        for key, value in [('oe', oe), ('residual', residual), ('combined', combined)]:
            parts[key].append(value.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        parts['reconstruction'].append(final.cpu())
    return {k: windows_to_series(torch.cat(v).numpy(), len(values), stride, pad) for k, v in parts.items()}

@torch.no_grad()
def backup_error(model, values, window, stride, n_groups=4):
    """Module A: |x - reconstruction| when the patch itself is fully masked, independently of OE."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    out = []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        B, T = x.shape
        index = torch.arange(T // PATCH, device=x.device)
        error = torch.zeros_like(x)
        for j in range(n_groups):
            masked = (index % n_groups == j).float()
            reconstruction = model.mae(x, masked.view(1, -1, 1).expand(B, -1, 1))
            error += (x - reconstruction).abs() * masked.repeat_interleave(PATCH).view(1, T)
        out.append(error.cpu())
    return windows_to_series(torch.cat(out).numpy(), len(values), stride, pad)

def processed(raw_test, raw_train, period, smooth=True, zero_start=True):
    """Authors' code-mode processing of one source: |x - train mean|, moving average, first period zeroed.
    Returns the processed test score and its train counterpart (for calibration)."""
    mean = raw_train.mean()
    out = []
    for raw in (raw_test, raw_train):
        value = np.abs(raw - mean)
        value = moving_average(value, period) if smooth else value.copy()
        if zero_start:
            value[:period] = 0
        out.append(value)
    return out

def model_signals(model, series, values=None, train_parts=None):
    """All point signals of one model on the test series (or on `values`, e.g. a slice with an injected anomaly).
    train_parts: precomputed training-part components, reused by the stress test."""
    period, window = series['period'], series['window']
    values = series['test'] if values is None else values
    test = predict_components(model, values, window, period)
    test['backup'] = backup_error(model, values, window, period)
    if train_parts is None:
        train_parts = predict_components(model, series['train'], window, period)
        train_parts['backup'] = backup_error(model, series['train'], window, period)
    zero = values is series['test']
    error_test, error_train = np.abs(values - test['reconstruction']), np.abs(series['train'] - train_parts['reconstruction'])
    r, r_train = processed(error_test, error_train, period, zero_start=zero)
    c, c_train = processed(test['combined'], train_parts['combined'], period, zero_start=zero)
    b, b_train = processed(test['backup'], train_parts['backup'], period, zero_start=zero)
    r_raw, _ = processed(error_test, error_train, period, smooth=False, zero_start=zero)
    c_raw, _ = processed(test['combined'], train_parts['combined'], period, smooth=False, zero_start=zero)
    s = {name: np.std(t) + 1e-12 for name, t in [('r', r_train), ('c', c_train), ('b', b_train)]}
    signals = {
        'full': (r + c) / 2, 'classify': c, 'reconstruct': r, 'backup': b, 'full_unsmoothed': (r_raw + c_raw) / 2,
        'oe': test['oe'], 'residual': test['residual'], 'error': error_test,
        'calibrated': (r / s['r'] + c / s['c']) / 2,
        'calibrated + backup': (r / s['r'] + c / s['c'] + b / s['b']) / 3,
    }
    return signals, train_parts

EVENT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup', 'full_unsmoothed', 'oe', 'residual', 'error']
SPLIT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup']
VARIANTS = ['calibrated', 'backup', 'calibrated + backup']
# Fixed column orders: rows are appended model by model, so every model must write the same columns in the same order
EVENT_COLUMNS = (['run_id', 'group', 'seed', 'family', 'event', 'start', 'length', 'length_over_period', 'window_share',
                  'amplitude_z'] + [f'{s}_{k}' for s in EVENT_SIGNALS
                                    for k in (['peak', 'mean', 'middle', 'edges'] if s in SPLIT_SIGNALS else ['peak', 'mean'])])
STRESS_COLUMNS = (['group', 'seed', 'family', 'type', 'length_periods', 'offset', 'position']
                  + [f'{s}_{k}' for s in SPLIT_SIGNALS + ['oe'] for k in ['peak', 'mean', 'middle', 'edges']] + ['spill'])

def segments(label):
    edges = np.diff(np.concatenate([[0], np.asarray(label).astype(int), [0]]))
    return list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))

def percentile_against(reference):
    """Fraction of the reference (normal) values strictly below each value."""
    ordered = np.sort(reference)
    return lambda v: np.searchsorted(ordered, v, side='left') / len(ordered)

def event_stats(signal, start, end, pct):
    """Peak and mean percentile over an event; middle third vs both edge thirds for events of 3 patches or more."""
    p = pct(signal[start:end])
    row = {'peak': float(pct(np.array([signal[start:end].max()]))[0]), 'mean': float(p.mean())}
    if end - start >= 3 * PATCH:
        third = (end - start) // 3
        row.update(middle=float(p[third:2 * third].mean()), edges=float(np.r_[p[:third], p[2 * third:]].mean()))
    return row

def event_rows(series, signals, base):
    """One row per labelled event. Normal reference: label 0, after the first period."""
    label, test, period = np.asarray(series['label']), series['test'], series['period']
    normal = (label == 0) & (np.arange(len(label)) >= period)
    mu, sd = test[normal].mean(), test[normal].std() + 1e-12
    pcts = {name: percentile_against(signals[name][normal]) for name in EVENT_SIGNALS}
    rows = []
    for k, (a, b) in enumerate(segments(label)):
        row = dict(base, event=k, start=int(a), length=int(b - a), length_over_period=(b - a) / period,
                   window_share=min(1.0, (b - a) / series['window']),
                   amplitude_z=float(np.abs(test[a:b] - mu).max() / sd))
        for name in EVENT_SIGNALS:
            stats = event_stats(signals[name], a, b, pcts[name])
            if name not in SPLIT_SIGNALS:
                stats = {k2: v for k2, v in stats.items() if k2 in ('peak', 'mean')}
            row.update({f'{name}_{k2}': v for k2, v in stats.items()})
        rows.append(row)
    return rows

def alarm_stats(series, score):
    """Top-k points by score, k = number of anomalous points: share that are normal, and of those the share
    within one period of an event (spill-over) rather than far from any event."""
    label, period = np.asarray(series['label']), series['period']
    valid = np.arange(len(label)) >= period
    k = int(label[valid].sum())
    top = np.where(valid)[0][np.argsort(-score[valid])[:k]]
    normal_top = top[label[top] == 0]
    near = np.zeros(len(label), bool)
    for a, b in segments(label):
        near[max(0, a - period):b + period] = True
    return {'top_k_normal_share': len(normal_top) / max(k, 1),
            'normal_alarms_near_share': float(near[normal_top].mean()) if len(normal_top) else np.nan}

# ---- cell 16.1
from sklearn.metrics import average_precision_score

M11_SPLIT = STORE / 'eda' / 'm11_split.csv'
if not M11_SPLIT.exists():
    rng = np.random.default_rng(2026)
    rows = []
    for family, d in manifest64.groupby('family', sort=True):
        groups = sorted(int(g) for g in d['group'])
        development = set(int(g) for g in rng.choice(groups, size=len(groups) // 2, replace=False))
        rows += [dict(group=g, family=family, role='development' if g in development else 'confirmation') for g in groups]
    pd.DataFrame(rows).to_csv(M11_SPLIT, index=False)
m11_split = pd.read_csv(M11_SPLIT)
print(m11_split.groupby(['family', 'role']).size().unstack())

# Shared by 16.2-16.7, defined here so that every notebook (via 16.0) has them without running 16.2
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))
role = m11_split.set_index('group')['role']

def read_m11(kind):
    files = sorted(RESULTS_DIR.glob(f'm11_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

SOURCES = ['oe', 'residual', 'error']
COMBOS = [w for w in np.ndindex(3, 3, 3) if sum(w) > 0]          # weights in {0, 1, 2} for (oe, residual, error)
EPS = 1e-12

def m11_sources(model, series):
    """Raw per-point sources on the training part and on the test part."""
    period, window = series['period'], series['window']
    out = {}
    for part in ['train', 'test']:
        c = predict_components(model, series[part], window, period)
        out[part] = dict(oe=c['oe'], residual=c['residual'], combined=c['combined'],
                         error=np.abs(series[part] - c['reconstruction']), x=series[part])
    return out

def smooth(v, period):
    v = moving_average(v, period)
    v[:period] = 0
    return v

def tail_z(values, reference, tail=0.95):
    """Rarity of each value under the reference (training) values: -log of its upper-tail rank (empirical) up to the
    95th percentile u of the reference; above u, an exponential tail (peaks over threshold): z(u) + (v - u) / beta,
    beta = mean excess of the reference over u. Without the tail, every value above the reference maximum would get
    the same score (saturation) and the ranking of strong anomalies would be lost."""
    ref = np.sort(reference)
    n = len(ref)
    at_least = n - np.searchsorted(ref, values, side='left')
    z = -np.log((1 + at_least) / (n + 1))
    u = np.quantile(ref, tail)
    excess = ref[ref > u] - u
    beta = excess.mean() if len(excess) and excess.mean() > 0 else (ref.std() + EPS)
    z_u = -np.log((1 + n - np.searchsorted(ref, u, side='left')) / (n + 1))
    above = values > u
    z[above] = z_u + (values[above] - u) / beta
    return z

def trailing_sd(x, width):
    return pd.Series(x).shift(1).rolling(width, min_periods=2).std().bfill().fillna(0).to_numpy()

def context_tail(test, train, period, bins=4):
    """Reconstruction error ranked against training errors of the same context (local SD over the previous period)."""
    c_train, c_test = trailing_sd(train['x'], period), trailing_sd(test['x'], period)
    edges = np.quantile(c_train, np.linspace(0, 1, bins + 1)[1:-1])
    b_train, b_test = np.searchsorted(edges, c_train), np.searchsorted(edges, c_test)
    z = np.empty(len(c_test))
    for b in range(bins):
        ref = train['error'][b_train == b]
        if len(ref) < 20:
            ref = train['error']
        mask = b_test == b
        z[mask] = tail_z(test['error'][mask], ref)
    return z

def m11_variants(src, period, combos=COMBOS, b3_weights=None):
    """Scores of all variants. combos: B3 weight sets to compute; b3_weights: the locked set (named 'B3')."""
    tr, te = src['train'], src['test']
    out = {}
    def branch(kind):
        f = (lambda v, ref: np.abs(v - ref.mean())) if kind == 'two' else (lambda v, ref: np.maximum(0, v - ref.mean()))
        r, c = smooth(f(te['error'], tr['error']), period), smooth(f(te['combined'], tr['combined']), period)
        rt, ct = smooth(f(tr['error'], tr['error']), period), smooth(f(tr['combined'], tr['combined']), period)
        return r, c, rt.std() + EPS, ct.std() + EPS
    r, c, sr, sc = branch('two')
    out['B0 CoAD'] = (r + c) / 2
    out['B1 calibrated'] = (r / sr + c / sc) / 2
    r1, c1, sr1, sc1 = branch('one')
    out['B1b one-sided'] = (r1 / sr1 + c1 / sc1) / 2
    z = {s: tail_z(te[s], tr[s]) for s in SOURCES}
    out['B2 tail'] = smooth((z['oe'] + z['residual'] + z['error']) / 3, period)
    for w in combos:
        out[f'B3 w={w}'] = smooth(sum(wi * z[s] for wi, s in zip(w, SOURCES)) / sum(w), period)
    if b3_weights is not None:
        out['B3 weighted tail'] = smooth(sum(wi * z[s] for wi, s in zip(b3_weights, SOURCES)) / sum(b3_weights), period)
    out['B4 context tail'] = smooth((z['oe'] + z['residual'] + context_tail(te, tr, period)) / 3, period)
    z_train = {s: tail_z(tr[s], tr[s]) for s in SOURCES}
    f_test, f_train = sum(z.values()) / 3, sum(z_train.values()) / 3
    parts = []
    for length in [1, max(2, period // 4), period]:
        g_test = f_test if length == 1 else moving_average(f_test, length)
        g_train = f_train if length == 1 else moving_average(f_train, length)
        parts.append(tail_z(g_test, g_train))
    b5 = 0.5 * parts[0] + 0.5 * np.mean(parts, axis=0)
    b5[:period] = 0
    out['B5 multi-length'] = b5
    saturation = float((te['error'] > tr['error'].max()).mean())     # share of test errors above every training error
    return out, saturation

def code_model(run):
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    return model
print('Setup done | SESSION_TAG =', SESSION_TAG)

In [ ]:
from sklearn.metrics import average_precision_score

M11_SPLIT = STORE / 'eda' / 'm11_split.csv'
if not M11_SPLIT.exists():
    rng = np.random.default_rng(2026)
    rows = []
    for family, d in manifest64.groupby('family', sort=True):
        groups = sorted(int(g) for g in d['group'])
        development = set(int(g) for g in rng.choice(groups, size=len(groups) // 2, replace=False))
        rows += [dict(group=g, family=family, role='development' if g in development else 'confirmation') for g in groups]
    pd.DataFrame(rows).to_csv(M11_SPLIT, index=False)
m11_split = pd.read_csv(M11_SPLIT)
print(m11_split.groupby(['family', 'role']).size().unstack())

# Shared by 16.2-16.7, defined here so that every notebook (via 16.0) has them without running 16.2
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))
role = m11_split.set_index('group')['role']

def read_m11(kind):
    files = sorted(RESULTS_DIR.glob(f'm11_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

SOURCES = ['oe', 'residual', 'error']
COMBOS = [w for w in np.ndindex(3, 3, 3) if sum(w) > 0]          # weights in {0, 1, 2} for (oe, residual, error)
EPS = 1e-12

def m11_sources(model, series):
    """Raw per-point sources on the training part and on the test part."""
    period, window = series['period'], series['window']
    out = {}
    for part in ['train', 'test']:
        c = predict_components(model, series[part], window, period)
        out[part] = dict(oe=c['oe'], residual=c['residual'], combined=c['combined'],
                         error=np.abs(series[part] - c['reconstruction']), x=series[part])
    return out

def smooth(v, period):
    v = moving_average(v, period)
    v[:period] = 0
    return v

def tail_z(values, reference, tail=0.95):
    """Rarity of each value under the reference (training) values: -log of its upper-tail rank (empirical) up to the
    95th percentile u of the reference; above u, an exponential tail (peaks over threshold): z(u) + (v - u) / beta,
    beta = mean excess of the reference over u. Without the tail, every value above the reference maximum would get
    the same score (saturation) and the ranking of strong anomalies would be lost."""
    ref = np.sort(reference)
    n = len(ref)
    at_least = n - np.searchsorted(ref, values, side='left')
    z = -np.log((1 + at_least) / (n + 1))
    u = np.quantile(ref, tail)
    excess = ref[ref > u] - u
    beta = excess.mean() if len(excess) and excess.mean() > 0 else (ref.std() + EPS)
    z_u = -np.log((1 + n - np.searchsorted(ref, u, side='left')) / (n + 1))
    above = values > u
    z[above] = z_u + (values[above] - u) / beta
    return z

def trailing_sd(x, width):
    return pd.Series(x).shift(1).rolling(width, min_periods=2).std().bfill().fillna(0).to_numpy()

def context_tail(test, train, period, bins=4):
    """Reconstruction error ranked against training errors of the same context (local SD over the previous period)."""
    c_train, c_test = trailing_sd(train['x'], period), trailing_sd(test['x'], period)
    edges = np.quantile(c_train, np.linspace(0, 1, bins + 1)[1:-1])
    b_train, b_test = np.searchsorted(edges, c_train), np.searchsorted(edges, c_test)
    z = np.empty(len(c_test))
    for b in range(bins):
        ref = train['error'][b_train == b]
        if len(ref) < 20:
            ref = train['error']
        mask = b_test == b
        z[mask] = tail_z(test['error'][mask], ref)
    return z

def m11_variants(src, period, combos=COMBOS, b3_weights=None):
    """Scores of all variants. combos: B3 weight sets to compute; b3_weights: the locked set (named 'B3')."""
    tr, te = src['train'], src['test']
    out = {}
    def branch(kind):
        f = (lambda v, ref: np.abs(v - ref.mean())) if kind == 'two' else (lambda v, ref: np.maximum(0, v - ref.mean()))
        r, c = smooth(f(te['error'], tr['error']), period), smooth(f(te['combined'], tr['combined']), period)
        rt, ct = smooth(f(tr['error'], tr['error']), period), smooth(f(tr['combined'], tr['combined']), period)
        return r, c, rt.std() + EPS, ct.std() + EPS
    r, c, sr, sc = branch('two')
    out['B0 CoAD'] = (r + c) / 2
    out['B1 calibrated'] = (r / sr + c / sc) / 2
    r1, c1, sr1, sc1 = branch('one')
    out['B1b one-sided'] = (r1 / sr1 + c1 / sc1) / 2
    z = {s: tail_z(te[s], tr[s]) for s in SOURCES}
    out['B2 tail'] = smooth((z['oe'] + z['residual'] + z['error']) / 3, period)
    for w in combos:
        out[f'B3 w={w}'] = smooth(sum(wi * z[s] for wi, s in zip(w, SOURCES)) / sum(w), period)
    if b3_weights is not None:
        out['B3 weighted tail'] = smooth(sum(wi * z[s] for wi, s in zip(b3_weights, SOURCES)) / sum(b3_weights), period)
    out['B4 context tail'] = smooth((z['oe'] + z['residual'] + context_tail(te, tr, period)) / 3, period)
    z_train = {s: tail_z(tr[s], tr[s]) for s in SOURCES}
    f_test, f_train = sum(z.values()) / 3, sum(z_train.values()) / 3
    parts = []
    for length in [1, max(2, period // 4), period]:
        g_test = f_test if length == 1 else moving_average(f_test, length)
        g_train = f_train if length == 1 else moving_average(f_train, length)
        parts.append(tail_z(g_test, g_train))
    b5 = 0.5 * parts[0] + 0.5 * np.mean(parts, axis=0)
    b5[:period] = 0
    out['B5 multi-length'] = b5
    saturation = float((te['error'] > tr['error'].max()).mean())     # share of test errors above every training error
    return out, saturation

def code_model(run):
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    return model

### 16.2. Development: every variant on the development models (re-run to continue)

AUC-PR (scikit-learn average precision, rank-based) of every variant, including the 26 weight sets of B3, on the
development series × 3 seeds. Check: B0 must equal the saved `scores.npy`. Rows in
`results/m11_dev_<SESSION_TAG>.csv` (fixed columns).

In [ ]:
M11_DEV = RESULTS_DIR / f'm11_dev_{SESSION_TAG}.csv'
M11_DEV_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant', 'auc_pr', 'saturation']
M11_DEV_SEEDS = SEEDS                    # e.g. [46] in one session and [47, 48] in another

done = read_m11('dev')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'development' and int(r.seed) in M11_DEV_SEEDS
        and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'development models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    variants, saturation = m11_variants(m11_sources(code_model(run), series), series['period'])
    gap = float(np.abs(variants['B0 CoAD'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: B0 differs from scores.npy by {gap:.1e}'
    label = np.asarray(series['label'])
    rows = [dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                 auc_pr=average_precision_score(label, score), saturation=saturation) for name, score in variants.items()]
    pd.DataFrame(rows).reindex(columns=M11_DEV_COLUMNS).to_csv(M11_DEV, mode='a', header=not M11_DEV.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: AUC-PR B0 {rows[0]["auc_pr"]:.3f} B1 {rows[1]["auc_pr"]:.3f}', flush=True)

### 16.3. Choose and lock on development

Mean AUC-PR over the development models (per series first, then over series). The best B3 weight set is locked
together with the other variants in `reports/m11_locked.json`; 16.4 reads only this file. Nothing here looks at the
confirmation series.

In [ ]:
dev = read_m11('dev').drop_duplicates(['group', 'seed', 'variant'], keep='last')
n_expected = int(sum(role.get(int(r.group)) == 'development' for r in code_runs.itertuples()))
n_done = dev.drop_duplicates(['group', 'seed']).shape[0]
assert n_done == n_expected, f'{n_done} of {n_expected} development models done: finish 16.2 (every session) before locking'
per_series = dev.groupby(['variant', 'group'])['auc_pr'].mean().unstack(0)
summary = (100 * per_series).mean().sort_values(ascending=False)
named = ['B0 CoAD', 'B1 calibrated', 'B1b one-sided', 'B2 tail', 'B4 context tail', 'B5 multi-length']
print(f'{per_series.shape[0]} development series, {dev["seed"].nunique()} seeds')
print('Development AUC-PR (x100), named variants:')
display(summary[named].round(2).to_frame('AUC-PR'))
b3 = summary[[v for v in summary.index if v.startswith('B3 w=')]]
print('Best B3 weight sets (oe, residual, error):')
display(b3.head(5).round(2).to_frame('AUC-PR'))
best = tuple(int(c) for c in b3.index[0].split('=')[1].strip('()').split(','))
locked = {'B3_weights': best, 'variants': ['B1 calibrated', 'B1b one-sided', 'B2 tail', 'B3 weighted tail',
                                          'B4 context tail', 'B5 multi-length'],
          'development_series': sorted(int(g) for g in per_series.index),
          'saturation_mean': float(dev['saturation'].mean())}
(REPORTS / 'm11_locked.json').write_text(json.dumps(locked, indent=2))
print('Locked:', locked['B3_weights'], '| share of test errors above every training error (mean):',
      round(locked['saturation_mean'], 4))

### 16.4. Confirmation: locked variants on the confirmation models (re-run to continue)

Full evaluator for the locked variants on the confirmation series × 3 seeds, plus the share of normal points among
the top-k scores (k = number of anomalous points, `alarm_stats` of part 14). B0 comes from section 7.
Rows in `results/m11_confirm_<SESSION_TAG>.csv` (fixed columns).

In [ ]:
M11_CONFIRM = RESULTS_DIR / f'm11_confirm_{SESSION_TAG}.csv'
M11_CONFIRM_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share', 'seconds']
M11_CONFIRM_SEEDS = SEEDS                # split between sessions, e.g. [46, 47] in notebook A and [48] in notebook B
assert (REPORTS / 'm11_locked.json').exists(), 'Run 16.3 first (in one session): the variants must be locked before confirmation'
locked = json.loads((REPORTS / 'm11_locked.json').read_text())
done = read_m11('confirm')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'confirmation' and int(r.seed) in M11_CONFIRM_SEEDS
        and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'confirmation models to do | locked B3 weights', locked['B3_weights'])
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    variants, _ = m11_variants(m11_sources(code_model(run), series), series['period'], combos=[],
                               b3_weights=tuple(locked['B3_weights']))
    gap = float(np.abs(variants['B0 CoAD'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: B0 differs from scores.npy by {gap:.1e}'
    rows = []
    for name in locked['variants']:
        row = dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                   **score_metrics(variants[name], series),
                   top_k_normal_share=alarm_stats(series, variants[name])['top_k_normal_share'])
        rows.append(row)
    for row in rows:
        row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame(rows).reindex(columns=M11_CONFIRM_COLUMNS).to_csv(M11_CONFIRM, mode='a', header=not M11_CONFIRM.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: VUS-PR " + ' | '.join(
        f"{r['variant'].split()[0]} {100 * r['VUS_PR']:.2f}" for r in rows) + f" | {rows[0]['seconds']:.0f} s", flush=True)

### 16.5. Results on the confirmation series

In [ ]:
conf = read_m11('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last')
b0 = code_runs[code_runs['group'].isin(conf['group'].unique())].assign(variant='B0 CoAD')
b0['top_k_normal_share'] = np.nan
allv = pd.concat([b0, conf], ignore_index=True).astype({'group': int, 'seed': int})
order = ['B0 CoAD'] + locked['variants']
TABLE1 = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}
print(f"{allv['group'].nunique()} confirmation series, seeds {sorted(allv['seed'].unique())}")
rows = []
for v in order:
    d = allv[allv['variant'] == v]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
table = pd.DataFrame(rows).set_index('variant')
table.loc['Table 1 (64 series, paper)'] = TABLE1
display(table.round(3))

if 'holm' not in globals():                       # defined in 9.3; repeated here so 16.5 runs without it
    def holm(pvalues):
        pvalues = np.asarray(pvalues); adjusted, running = np.empty(len(pvalues)), 0.0
        for i, idx in enumerate(np.argsort(pvalues)):
            running = max(running, (len(pvalues) - i) * pvalues[idx]); adjusted[idx] = min(1.0, running)
        return adjusted
per = 100 * allv.pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
tests = []
for reference in ['B0 CoAD', 'B1 calibrated']:
    for v in order:
        if v in ('B0 CoAD', reference):
            continue
        d = per[v] - per[reference]
        tests.append({'vs': reference, 'variant': v, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                      'p (one-sided)': wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else np.nan})
tests = pd.DataFrame(tests)
for reference in ['B0 CoAD', 'B1 calibrated']:
    m = tests['vs'] == reference
    tests.loc[m, 'p Holm'] = holm(tests.loc[m, 'p (one-sided)'].fillna(1).to_numpy())
display(tests.round(4))
print('VUS-PR by family (confirmation):')
display(per.groupby(family_of.reindex(per.index).values).mean()[order].round(2))
print('VUS-PR gain over B1 per seed:')
ps = 100 * allv.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
display(ps[order[2:]].sub(ps['B1 calibrated'], axis=0).groupby(level='seed').mean().round(2))

primary = per['B3 weighted tail'] - per['B1 calibrated']
p = wilcoxon(primary, alternative='greater').pvalue
print(f'Primary prediction (locked B3 > B1 on confirmation, VUS-PR): {primary.mean():+.2f}, higher on '
      f'{int((primary > 0).sum())}/{len(primary)}, one-sided p = {p:.3f} -> ' + ('HOLDS' if p < 0.05 and primary.mean() > 0 else 'FAILS'))
table.to_csv(REPORTS / 'm11_confirmation.csv'); tests.to_csv(REPORTS / 'm11_tests.csv', index=False)

### 16.6. Short anomalies: spike stress test (re-run to continue)

**Why.** M9 (8 262 real events): the final score finds 1.3% of 1-point events and ~28% of 2-8-point events, OE
finds 1.2% / 12-20%, but the **raw reconstruction error** finds 44% / 86% and the unsmoothed final score 33% / ~80%.
So short anomalies are partly lost by OE and largely by the smoothing. Causes not yet tested: the patch size
(8 points), pseudo-anomalies that are never shorter than 10% of a period (D21), the moving average over one period.

**Design** (one factor at a time, inference on the saved code-mode models):

* rectangular spike of `SPIKE_AMPLITUDES` x the training SD (fixed), lengths 1, 2, 4, 8, 16 points;
* start offset 0, 3, 5, 7 within the patch grid of the last test window starting before the spike (an 8-point spike
  at offset 0 fills exactly one patch of that window, at offset 5 it is split over two);
* 4 positions per series in normal stretches (at least one window away from any labelled anomaly),
  `default_rng(2026 + group)`; per family the first two series (manifest order) with room; 3 seeds;
* percentiles against the normal points of the untouched series; a spike is **found** by a signal when its peak
  percentile over the spike is >= 0.99.

**Predictions written before running:** (S1) the OE found-rate rises with the spike length (1 -> 16 points) by at
least 0.20; (S2) for spikes of 1-4 points the unsmoothed final score finds at least 0.20 more spikes than the
smoothed one (smoothing loss); (S3) for 8-point spikes, offset 0 (one patch) is found more often by OE than offset 5
(two patches), i.e. a patch-grid effect. Rows in `results/m11_spike_<SESSION_TAG>.csv` (fixed columns).
Needs 16.1 (definitions only) and 14.1; **not** 16.2-16.5, so it can run in a second notebook at the same time.

In [ ]:
# Self-contained after 16.1: 16.2 is not needed, so this part can run in a second notebook in parallel
family_of = manifest64.set_index('group')['family']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
M11_SPIKE = RESULTS_DIR / f'm11_spike_{SESSION_TAG}.csv'
SPIKE_LENGTHS, SPIKE_OFFSETS, SPIKE_AMPLITUDES = [1, 2, 4, 8, 16], [0, 3, 5, 7], [3, 1]   # x training SD; finished amplitudes are skipped
SPIKE_SIGNALS = ['oe', 'residual', 'error', 'reconstruct', 'classify', 'full_unsmoothed', 'full']
M11_SPIKE_COLUMNS = (['group', 'seed', 'family', 'amplitude', 'length', 'offset', 'position']
                     + [f'{s}_peak' for s in SPIKE_SIGNALS])

def spike_positions(series):
    window, period, n = series['window'], series['period'], len(series['test'])
    blocked = np.zeros(n, bool)
    for a, b in segments(series['label']):
        blocked[max(0, a - window - max(SPIKE_LENGTHS) - PATCH):b + window] = True
    lo, hi = max(2 * window, period) + PATCH, n - 2 * window - max(SPIKE_LENGTHS) - PATCH
    candidates = [s for s in range(lo, hi) if not blocked[s]]
    if not candidates:
        return []
    rng = np.random.default_rng(2026 + series['group'])
    return sorted(int(p) for p in rng.choice(candidates, size=min(4, len(candidates)), replace=False))

def grid_start(series, position, offset):
    """Start of a spike at `offset` within the patch grid of the last test window starting at or before `position`."""
    window, stride, n = series['window'], series['period'], len(series['test'])
    pad = 0 if stride == 1 or (n - window) % stride == 0 else stride - (n - window) % stride
    starts = np.arange(0, pad + n - window + 1, stride) - pad
    w = starts[starts <= position].max()
    return int(position - (position - w) % PATCH + offset)

SPIKE_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    chosen = [int(g) for g in d['group'] if spike_positions(load_series(int(g)))][:2]
    print(f'{family}: spike series {chosen}' + ('' if chosen else ' (no normal stretch long enough)'))
    SPIKE_GROUPS += chosen

def read_spike():
    files = sorted(RESULTS_DIR.glob('m11_spike_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_spike()
done = set(zip(done['group'].astype(int), done['seed'].astype(int), done['amplitude'].astype(float))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in SPIKE_GROUPS
        and any((int(r.group), int(r.seed), float(a)) not in done for a in SPIKE_AMPLITUDES)]
print(len(todo), 'models for the spike test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    window, period = series['window'], series['period']
    model = code_model(run)
    base, train_parts = model_signals(model, series)
    label = np.asarray(series['label'])
    keep = np.ones(len(label), bool)
    for a, b in segments(label):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    pcts = {s: percentile_against(base[s][keep]) for s in SPIKE_SIGNALS}
    sd = series['train'].std()
    rows = []
    for position in spike_positions(series):
        for amp in [a for a in SPIKE_AMPLITUDES if (group, seed, float(a)) not in done]:
            for length in SPIKE_LENGTHS:
                for offset in SPIKE_OFFSETS:
                    start = grid_start(series, position, offset)
                    values = series['test'].copy()
                    values[start:start + length] += amp * sd
                    a, b = start - 2 * window, start + length + 2 * window
                    signals, _ = model_signals(model, series, values=values[a:b], train_parts=train_parts)
                    s0 = start - a
                    row = dict(group=group, seed=seed, family=family_of[group], amplitude=amp, length=length,
                               offset=offset, position=position)
                    for s in SPIKE_SIGNALS:
                        row[f'{s}_peak'] = float(pcts[s](np.array([signals[s][s0:s0 + length].max()]))[0])
                    rows.append(row)
    if rows:
        pd.DataFrame(rows).reindex(columns=M11_SPIKE_COLUMNS).to_csv(M11_SPIKE, mode='a', header=not M11_SPIKE.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} spikes', flush=True)

### 16.7. Spike results and predictions S1-S3

In [ ]:
sp = read_spike().drop_duplicates(['group', 'seed', 'amplitude', 'length', 'offset', 'position'], keep='last')
found = sp.assign(**{s: sp[f'{s}_peak'] >= 0.99 for s in SPIKE_SIGNALS})
print(f"{sp['group'].nunique()} series, {len(sp)} spikes, amplitudes {sorted(sp['amplitude'].unique())}")
for amp, f in found.groupby('amplitude'):
    print()
    print(f'===== amplitude {amp} x training SD')
    by_len = f.groupby('length')[SPIKE_SIGNALS].mean()
    print('Share of spikes found, by length:')
    display(by_len.round(3))
    print('Share found by OE, by length and offset:')
    display(f.pivot_table(index='length', columns='offset', values='oe').round(3))
    print('By family:')
    display(f.groupby(['family', 'length'])[['oe', 'error', 'full_unsmoothed', 'full']].mean().round(3))
    s1 = bool(by_len['oe'].is_monotonic_increasing and by_len['oe'].iloc[-1] - by_len['oe'].iloc[0] >= 0.2)
    short = f[f['length'] <= 4]
    s2 = short['full_unsmoothed'].mean() - short['full'].mean()
    eight = f[f['length'] == 8]
    s3 = eight.loc[eight['offset'] == 0, 'oe'].mean() - eight.loc[eight['offset'] == 5, 'oe'].mean()
    print(f"S1 (OE found-rate rises with length by >= 0.20): {by_len['oe'].round(3).tolist()} -> " + ('HOLDS' if s1 else 'FAILS'))
    print(f'S2 (smoothing loss on 1-4-point spikes >= 0.20): {s2:+.3f} -> ' + ('HOLDS' if s2 >= 0.2 else 'FAILS'))
    print(f'S3 (8-point spike: OE offset 0 - offset 5 > 0): {s3:+.3f} -> ' + ('HOLDS' if s3 > 0 else 'FAILS'))
found.groupby(['amplitude', 'length'])[SPIKE_SIGNALS].mean().to_csv(REPORTS / 'm11_spike_by_length.csv')

### 16.9. Dense 1-point anomalies: density test (re-run to continue)

**Why.** 16.7: isolated 1-point spikes of 1x the training SD are found by the final score 37% of the time (raw error
73%), but the real 1-point events (all 3 834 in TODS, 9 series) only 1.3%. At the same amplitude (1-2 SD) real events
are still found far less (final 1.4%, raw error 53%) than isolated spikes (TODS 1 SD: 40%, 92%). The real events are
dense: about 142 per series, median gap 21 points, shorter than one period (100). The spike test placed spikes at least
one window away from any anomaly, so it never saw this.

**Design** (inference on the saved code-mode models, no retraining):

* 1-point spikes of `DENSITY_AMPLITUDES` x training SD (+0.5 and +1), one every `gap` points over the **whole** normal
  part of the test series (at least one window away from labelled anomalies, after the first period), random start
  phase `default_rng(2026 + group)`; gaps of 4 periods (isolated: one window apart, reference), 0.4, 0.2 and 0.1
  period, at least 5 points (for TODS, period 100: 400, 40, 20, 10 points; the real TODS events are 0.21 period
  apart). The gap is set in periods because the model window and the smoothing both scale with the period;
* per family the first two series (manifest order) with at least 3 spikes at the 4-period gap; 3 seeds (series whose anomalies
  leave no normal stretch are skipped; for IOPS, period 8, the three dense gaps are all 5 points);
* the whole modified series goes through the model, as in a real evaluation;
* three numbers per signal: **found_own**, peak percentile >= 0.99 against the normal points of the *modified* series
  (what an evaluation sees); **found_clean**, against the normal points of the *untouched* series (the spike's own
  score, threshold fixed); **normal_above**, share of the modified normal points above the untouched 99th percentile
  (how much the normal points next to the spikes are lifted).

**Predictions written before running:**

* (D1) density hurts: at 1 SD, the final-score found_own at 0.2 period is at least 0.20 below 4 periods;
* (D2) the loss comes mainly from lifted normal points (threshold), not from a lower spike score: D1 holds and the
  drop of found_clean (4 -> 0.2 period) is less than half the drop of found_own;
* (D3) consistency with the real events: at 0.5 SD and 0.2 period the final-score found_own is <= 0.10 (real: 1.3%).

Signals: raw error, OE, unsmoothed final, final, `calibrated`. Rows in `results/m11_density_<SESSION_TAG>.csv`
(fixed columns). Needs 14.1 and 16.1 (for `code_model`); not 16.2-16.8.

In [ ]:
# Self-contained after 16.1 (16.6 is not needed)
family_of = manifest64.set_index('group')['family']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
M11_DENSITY = RESULTS_DIR / f'm11_density_{SESSION_TAG}.csv'
DENSITY_AMPLITUDES, DENSITY_GAPS = [0.5, 1], [4, 0.4, 0.2, 0.1]     # x training SD; gap between spikes in periods
DENSITY_SIGNALS = ['error', 'oe', 'full_unsmoothed', 'full', 'calibrated']
M11_DENSITY_COLUMNS = (['group', 'seed', 'family', 'amplitude', 'gap_periods', 'gap', 'n_spikes']
                       + [f'{s}_{k}' for s in DENSITY_SIGNALS for k in ['found_own', 'found_clean', 'normal_above']])

def normal_mask(series):
    """Normal test points used as the percentile reference: away from labelled anomalies, after the first period."""
    window, period = series['window'], series['period']
    keep = np.ones(len(series['test']), bool)
    for a, b in segments(series['label']):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    return keep

def gap_points(series, gap_periods):
    return max(5, int(round(gap_periods * series['period'])))

def density_positions(series, gap_periods):
    gap = gap_points(series, gap_periods)
    window, period, n = series['window'], series['period'], len(series['test'])
    allowed = normal_mask(series)
    phase = int(np.random.default_rng(2026 + series['group']).integers(gap))
    return np.array([p for p in range(period + window + phase, n - window, gap) if allowed[p]], int)

DENSITY_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    chosen = [int(g) for g in d['group'] if len(density_positions(load_series(int(g)), max(DENSITY_GAPS))) >= 3][:2]
    print(f'{family}: density series {chosen}')
    DENSITY_GROUPS += chosen

def read_density():
    files = sorted(RESULTS_DIR.glob('m11_density_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_density()
done = (set(zip(done['group'].astype(int), done['seed'].astype(int), done['amplitude'].astype(float),
                done['gap_periods'].astype(float))) if len(done) else set())
keys = lambda g, s: [(g, s, float(a), float(q)) for a in DENSITY_AMPLITUDES for q in DENSITY_GAPS]
todo = [r for r in code_runs.itertuples() if int(r.group) in DENSITY_GROUPS
        and any(k not in done for k in keys(int(r.group), int(r.seed)))]
print(len(todo), 'models for the density test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = code_model(run)
    # a copy, so the first period is processed as in the injected runs (same train SD for `calibrated`)
    base, train_parts = model_signals(model, series, values=series['test'].copy())
    keep = normal_mask(series)
    clean = {s: percentile_against(base[s][keep]) for s in DENSITY_SIGNALS}
    clean99 = {s: np.quantile(base[s][keep], 0.99) for s in DENSITY_SIGNALS}
    sd = series['train'].std()
    rows = []
    for _, _, amp, gap in [k for k in keys(group, seed) if k not in done]:
        pos = density_positions(series, gap)
        values = series['test'].copy()
        values[pos] += amp * sd
        signals, _ = model_signals(model, series, values=values, train_parts=train_parts)
        own_mask = keep.copy()
        own_mask[pos] = False
        row = dict(group=group, seed=seed, family=family_of[group], amplitude=amp, gap_periods=gap,
                   gap=gap_points(series, gap), n_spikes=len(pos))
        for s in DENSITY_SIGNALS:
            sig = signals[s]
            own = percentile_against(sig[own_mask])
            row[f'{s}_found_own'] = float((own(sig[pos]) >= 0.99).mean())
            row[f'{s}_found_clean'] = float((clean[s](sig[pos]) >= 0.99).mean())
            row[f'{s}_normal_above'] = float((sig[own_mask] > clean99[s]).mean())
        rows.append(row)
    if rows:
        (pd.DataFrame(rows).reindex(columns=M11_DENSITY_COLUMNS)
         .to_csv(M11_DENSITY, mode='a', header=not M11_DENSITY.exists(), index=False))
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} conditions', flush=True)

### 16.10. Density results and predictions D1-D3

In [ ]:
dn = read_density().drop_duplicates(['group', 'seed', 'amplitude', 'gap_periods'], keep='last')
print(f"{dn['group'].nunique()} series, {len(dn)} conditions; spikes per series (median by gap in periods):",
      dn.groupby('gap_periods')['n_spikes'].median().to_dict())
for kind in ['found_own', 'found_clean', 'normal_above']:
    print()
    print(kind, '(mean over series and seeds); rows: amplitude, gap in periods')
    t = dn.groupby(['amplitude', 'gap_periods'])[[f'{s}_{kind}' for s in DENSITY_SIGNALS]].mean()
    t.columns = DENSITY_SIGNALS
    display(t.sort_index(ascending=[True, False]).round(3))
print('Final score found_own by family:')
display(dn.pivot_table(index=['amplitude', 'gap_periods'], columns='family', values='full_found_own')
        .sort_index(ascending=[True, False]).round(3))
m = dn.groupby(['amplitude', 'gap_periods']).mean(numeric_only=True)
d_own = m.loc[(1, 4), 'full_found_own'] - m.loc[(1, 0.2), 'full_found_own']
d_clean = m.loc[(1, 4), 'full_found_clean'] - m.loc[(1, 0.2), 'full_found_clean']
d3 = m.loc[(0.5, 0.2), 'full_found_own']
print()
print(f'D1 (1 SD, final found_own 4 periods - 0.2 period >= 0.20): {d_own:+.3f} -> ' + ('HOLDS' if d_own >= 0.2 else 'FAILS'))
print(f'D2 (D1 holds and drop of found_clean < half the drop of found_own): {d_clean:+.3f} vs {d_own:+.3f} -> '
      + ('HOLDS' if d_own >= 0.2 and d_clean < d_own / 2 else 'FAILS'))
print(f'D3 (0.5 SD, 0.2 period, final found_own <= 0.10; real 1-point events 0.013): {d3:.3f} -> '
      + ('HOLDS' if d3 <= 0.1 else 'FAILS'))
m.to_csv(REPORTS / 'm11_density.csv')

### 16.12. M12: replacing the moving average (definitions)

**Why.** 16.11: the moving average over one period is the main reason 1-point anomalies are lost (1 SD, isolated:
the unsmoothed score finds 0.66, the final score 0.03; dilution by 1/P costs 0.36, spreading to the P neighbours
0.29). But M11's B5 (adds an instantaneous channel, with the rarity transform) helped TODS (+4.0) and SED (+2.6) and
hurt SVDB (-7.6) and IOPS (-6.9): long anomalies need smoothing. So the question is **which temporal operator, per
branch**, keeps short peaks without adding false alarms.

**Variant.** Everything as `calibrated` (B1, the best so far): per branch `|x - mean_train(x)|`, then a temporal
operator `T`, first period zeroed, divided by the SD of `T` on the training part, mean of the two branches. Only `T`
changes, separately for the error branch and the classification branch:

| Operator | Meaning |
|---|---|
| `none` | no smoothing |
| `mean w` | centred moving average, width w (authors: `mean P`) |
| `max w` | centred moving maximum, width w: keeps the peak height (no dilution), still spreads it over w points |
| `mix` | `(T_mean P / SD + T_none / SD) / 2`: half smoothed, half instantaneous, without B5's rarity transform |

Widths: P/8, P/4, P/2, P (at least 2 points). 9 operators x 9 operators = 81 pairs; `mean P | mean P` is exactly B1.

**Protocol** (as M11, same split `eda/m11_split.csv`): AUC-PR of the 81 pairs on the development models (16.13),
best pair locked (16.14, `reports/m12_locked.json`), full evaluator on the confirmation models (16.15), compared with
B0 and B1 (16.16). One pair is fixed **before running**, from 16.11 alone: `error = max P/4 | classify = mean P`
(keep the short peaks in the reconstruction error, the branch that sees them; keep the classifier smoothed).

**Predictions written before running:**

* (primary) the locked pair has a higher confirmation VUS-PR than B1, one-sided Wilcoxon p < 0.05 over series;
* (secondary) the a-priori pair has a higher confirmation VUS-PR than B1; its gain is largest on TODS.

**Caveat.** The confirmation series were already used to evaluate M11, and the per-family M11 results motivated this
design. The lock uses development series only, but the confirmation series are no longer fully unseen; an external
check (KDD21) stays the clean test.

**How to run.** After 16.0: 16.12, 16.13, 16.14 once all development models are done, 16.15, 16.16. Two notebooks at
the same time: `SETUP_TAG = 'gpu-a'` in one and `'gpu-b'` in the other (cell 16.0); each takes half of the models
(fixed split by `(group + seed) % 2`), nothing else to change. 16.14 runs in one notebook, after both finish 16.13. 16.15 reuses B1 from M11
(`m11_confirm_*.csv`), so it evaluates at most two variants per model.

In [ ]:
from scipy.ndimage import maximum_filter1d

def m12_operators(period):
    widths = {'P/8': max(2, period // 8), 'P/4': max(2, period // 4), 'P/2': max(2, period // 2), 'P': period}
    ops = {'none': lambda v: v.copy()}
    for name in ['P/4', 'P/2', 'P']:
        ops[f'mean {name}'] = (lambda w: lambda v: moving_average(v, w))(widths[name])
    for name in ['P/8', 'P/4', 'P/2', 'P']:
        ops[f'max {name}'] = (lambda w: lambda v: maximum_filter1d(v, size=w, mode='nearest'))(widths[name])
    return ops
M12_OPS = list(m12_operators(100)) + ['mix']
M12_A_PRIORI = ('max P/4', 'mean P')                # (error, classify), fixed from 16.11 before running

def m12_branches(src, period):
    """Calibrated branch scores for every operator: {op: (test score / train SD)} for the error and classify branches."""
    tr, te = src['train'], src['test']
    out = {}
    for branch, key in [('error', 'error'), ('classify', 'combined')]:
        mu = tr[key].mean()
        dev_test, dev_train = np.abs(te[key] - mu), np.abs(tr[key] - mu)
        scaled = {}
        for name, op in m12_operators(period).items():
            a, b = op(dev_test), op(dev_train)
            a[:period] = 0
            b[:period] = 0
            scaled[name] = a / (b.std() + EPS)
        scaled['mix'] = (scaled['mean P'] + scaled['none']) / 2
        out[branch] = scaled
    return out

def m12_name(pair):
    return f'error={pair[0]} | classify={pair[1]}'

def m12_scores(branches, pairs):
    return {m12_name(p): (branches['error'][p[0]] + branches['classify'][p[1]]) / 2 for p in pairs}

def read_m12(kind):
    files = sorted(RESULTS_DIR.glob(f'm12_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

M12_PAIRS = [(a, b) for a in M12_OPS for b in M12_OPS]
# Two notebooks at the same time: 'gpu-a' takes the models with (group + seed) even, 'gpu-b' the odd ones (a fixed
# split, so the two never take the same model); any other SESSION_TAG runs every model
M12_PARTS = {'gpu-a': 0, 'gpu-b': 1}
def m12_mine(group, seed):
    part = M12_PARTS.get(SESSION_TAG)
    return part is None or (int(group) + int(seed)) % 2 == part
print(len(M12_OPS), 'operators per branch,', len(M12_PAIRS), 'pairs | a priori:', m12_name(M12_A_PRIORI),
      '| SESSION_TAG', SESSION_TAG, '->', 'half', M12_PARTS[SESSION_TAG] if SESSION_TAG in M12_PARTS else 'all models')

### 16.13. M12 development: the 81 pairs on the development models (re-run to continue)

AUC-PR of every pair on the development series x 3 seeds. Check: `mean P | mean P` must equal M11's B1.
Rows in `results/m12_dev_<SESSION_TAG>.csv` (fixed columns).

In [ ]:
M12_DEV = RESULTS_DIR / f'm12_dev_{SESSION_TAG}.csv'
M12_DEV_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant', 'auc_pr']
M12_DEV_SEEDS = SEEDS                    # the split between notebooks is by SESSION_TAG (m12_mine, 16.12)

done = read_m12('dev')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'development' and int(r.seed) in M12_DEV_SEEDS
        and m12_mine(r.group, r.seed) and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'development models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    src = m11_sources(code_model(run), series)
    scores = m12_scores(m12_branches(src, series['period']), M12_PAIRS)
    b1 = m11_variants(src, series['period'], combos=[])[0]['B1 calibrated']
    gap = float(np.abs(scores[m12_name(('mean P', 'mean P'))] - b1).max())
    assert gap < 1e-6 * (1 + np.abs(b1).max()), f'{run.run_id}: mean P | mean P differs from B1 by {gap:.1e}'
    label = np.asarray(series['label'])
    rows = [dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                 auc_pr=average_precision_score(label, score)) for name, score in scores.items()]
    pd.DataFrame(rows).reindex(columns=M12_DEV_COLUMNS).to_csv(M12_DEV, mode='a', header=not M12_DEV.exists(), index=False)
    by = {r['variant']: r['auc_pr'] for r in rows}
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: AUC-PR B1 {by[m12_name(('mean P', 'mean P'))]:.3f} "
          f"a priori {by[m12_name(M12_A_PRIORI)]:.3f} best {max(by.values()):.3f}", flush=True)

### 16.14. M12: choose and lock on development

In [ ]:
dev12 = read_m12('dev').drop_duplicates(['group', 'seed', 'variant'], keep='last')
n_expected = int(sum(role.get(int(r.group)) == 'development' for r in code_runs.itertuples()))
n_done = dev12.drop_duplicates(['group', 'seed']).shape[0]
assert n_done == n_expected, f'{n_done} of {n_expected} development models done: finish 16.13 (every session) before locking'
per12 = dev12.groupby(['variant', 'group'])['auc_pr'].mean().unstack(0)
summary12 = (100 * per12).mean().sort_values(ascending=False)
B1_NAME, AP_NAME = m12_name(('mean P', 'mean P')), m12_name(M12_A_PRIORI)
print(f'{per12.shape[0]} development series, {dev12["seed"].nunique()} seeds')
print('Top 10 pairs (development AUC-PR x 100):')
display(summary12.head(10).round(2).to_frame('AUC-PR'))
print(f'B1 (mean P | mean P): {summary12[B1_NAME]:.2f} | a priori: {summary12[AP_NAME]:.2f}')
print('Development AUC-PR by error operator (classify = mean P) and by classify operator (error = mean P):')
display(pd.DataFrame({'error op (classify=mean P)': [summary12[m12_name((o, 'mean P'))] for o in M12_OPS],
                      'classify op (error=mean P)': [summary12[m12_name(('mean P', o))] for o in M12_OPS]},
                     index=M12_OPS).round(2))
fam12 = (100 * per12[[summary12.index[0], B1_NAME, AP_NAME]]).groupby(family_of.reindex(per12.index).values).mean()
print('Development AUC-PR by family:')
display(fam12.round(2))
best12 = summary12.index[0]
pair = tuple(part.split('=')[1] for part in best12.split(' | '))
locked12 = {'locked_pair': pair, 'a_priori_pair': M12_A_PRIORI, 'development_series': sorted(int(g) for g in per12.index)}
(REPORTS / 'm12_locked.json').write_text(json.dumps(locked12, indent=2))
print('Locked:', best12)

### 16.15. M12 confirmation: locked and a-priori pairs (re-run to continue)

Full evaluator on the confirmation series x 3 seeds; B1 is read from M11 (`m11_confirm_*.csv`), B0 from section 7.
Rows in `results/m12_confirm_<SESSION_TAG>.csv` (fixed columns).

In [ ]:
M12_CONFIRM = RESULTS_DIR / f'm12_confirm_{SESSION_TAG}.csv'
M12_CONFIRM_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share', 'seconds']
M12_CONFIRM_SEEDS = SEEDS                # the split between notebooks is by SESSION_TAG (m12_mine, 16.12)
assert (REPORTS / 'm12_locked.json').exists(), 'Run 16.14 first: the pair must be locked before confirmation'
locked12 = json.loads((REPORTS / 'm12_locked.json').read_text())
m12_conf_variants = {'M12 locked': tuple(locked12['locked_pair']), 'M12 a priori': tuple(locked12['a_priori_pair'])}
done = read_m12('confirm')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'confirmation' and int(r.seed) in M12_CONFIRM_SEEDS
        and m12_mine(r.group, r.seed) and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'confirmation models to do |', {k: m12_name(v) for k, v in m12_conf_variants.items()})
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    branches = m12_branches(m11_sources(code_model(run), series), series['period'])
    rows, cache = [], {}
    for name, p in m12_conf_variants.items():
        score = m12_scores(branches, [p])[m12_name(p)]
        if p not in cache:                                  # same pair locked and a priori: evaluate once
            cache[p] = dict(**score_metrics(score, series),
                            top_k_normal_share=alarm_stats(series, score)['top_k_normal_share'])
        rows.append(dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name, **cache[p]))
    for row in rows:
        row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame(rows).reindex(columns=M12_CONFIRM_COLUMNS).to_csv(M12_CONFIRM, mode='a', header=not M12_CONFIRM.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: VUS-PR " + ' | '.join(
        f"{r['variant']} {100 * r['VUS_PR']:.2f}" for r in rows) + f" | {rows[0]['seconds']:.0f} s", flush=True)

### 16.16. M12 results on the confirmation series

In [ ]:
conf12 = read_m12('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last')
locked12 = json.loads((REPORTS / 'm12_locked.json').read_text())
b1 = read_m11('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last').query("variant == 'B1 calibrated'")
b0 = code_runs[code_runs['group'].isin(conf12['group'].unique())].assign(variant='B0 CoAD')
b0['top_k_normal_share'] = np.nan
all12 = pd.concat([b0, b1, conf12], ignore_index=True).astype({'group': int, 'seed': int})
all12 = all12[all12['group'].isin(conf12['group'].unique()) & all12['seed'].isin(conf12['seed'].unique())]
order12 = ['B0 CoAD', 'B1 calibrated', 'M12 locked', 'M12 a priori']
print(f"{conf12['group'].nunique()} confirmation series, seeds {sorted(conf12['seed'].unique())} | locked "
      f"{m12_name(locked12['locked_pair'])} | a priori {m12_name(locked12['a_priori_pair'])}")
n_models = all12.groupby('variant')[['group', 'seed']].apply(lambda d: len(d.drop_duplicates()))
assert n_models.nunique() == 1, f'unequal model counts: {n_models.to_dict()} (finish 16.15 in every session)'
rows = []
for v in order12:
    d = all12[all12['variant'] == v]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
table12 = pd.DataFrame(rows).set_index('variant')
table12.loc['Table 1 (64 series, paper)'] = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}
display(table12.round(3))
if 'holm' not in globals():                       # defined in 9.3; repeated so 16.16 runs without it
    def holm(pvalues):
        pvalues = np.asarray(pvalues); adjusted, running = np.empty(len(pvalues)), 0.0
        for i, idx in enumerate(np.argsort(pvalues)):
            running = max(running, (len(pvalues) - i) * pvalues[idx]); adjusted[idx] = min(1.0, running)
        return adjusted
per12c = 100 * all12.pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
tests12 = []
for reference in ['B0 CoAD', 'B1 calibrated']:
    for v in order12:
        if v in ('B0 CoAD', reference):
            continue
        d = per12c[v] - per12c[reference]
        tests12.append({'vs': reference, 'variant': v, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                        'p (one-sided)': wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else np.nan})
tests12 = pd.DataFrame(tests12)
for reference in ['B0 CoAD', 'B1 calibrated']:
    m = tests12['vs'] == reference
    tests12.loc[m, 'p Holm'] = holm(tests12.loc[m, 'p (one-sided)'].fillna(1).to_numpy())
display(tests12.round(4))
print('VUS-PR by family (confirmation):')
display(per12c.groupby(family_of.reindex(per12c.index).values).mean()[order12].round(2))
print('VUS-PR gain over B1 per seed:')
ps12 = 100 * all12.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
display(ps12[order12[2:]].sub(ps12['B1 calibrated'], axis=0).groupby(level='seed').mean().round(2))
for v, label in [('M12 locked', 'Primary (locked pair > B1)'), ('M12 a priori', 'Secondary (a-priori pair > B1)')]:
    d = per12c[v] - per12c['B1 calibrated']
    p = wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else 1.0
    print(f'{label}, VUS-PR: {d.mean():+.2f}, higher on {int((d > 0).sum())}/{len(d)}, one-sided p = {p:.3f} -> '
          + ('HOLDS' if p < 0.05 and d.mean() > 0 else 'FAILS'))
table12.to_csv(REPORTS / 'm12_confirmation.csv'); tests12.to_csv(REPORTS / 'm12_tests.csv', index=False)

### 16.18. M13: external check, choice of the series (definitions)

**Why.** Every improvement so far comes from the 64 series of the paper, used many times: `calibrated` was found on
them (M9), and the confirmation half was already looked at in M11 and suggested M12. M13 runs the **fixed** formulas
on series never used before. The 64 series are **all** of MGAB, SED, SVDB, IOPS and TODS in TSB-AD-U; the other 806
series (18 families) are already in `data_source/TSB-AD-U.zip`. The UCR family is the UCR anomaly archive (KDD21).

**Choice (fixed before any run, written once to `eda/m13_external.csv`):** the 12 other families with at least 9
series (YAHOO, UCR, WSD, SMD, Exathlon, OPPORTUNITY, NAB, Stock, SMAP, LTDB, MSL, NEK); per family, series of at
most 100 000 points, byte-identical copies removed (TSB-AD-U holds several in Exathlon and YAHOO; the lowest id is
kept), in a random order (`default_rng(2027)`), keeping the first 5 whose window settings are valid
(window >= 2 patches, at least one training window). 60 series. The check uses no label values.

**Run (16.19):** CoAD in code mode trained as in part 7 (3 seeds), then on the same model: B0 (CoAD), B1
(`calibrated`), M12 locked (`reports/m12_locked.json`); plus the train-SD ratio of the two branches.

**Predictions written before running:**

| Code | Prediction | Criterion |
|---|---|---|
| E1 (primary) | `calibrated` > CoAD | VUS-PR, one-sided Wilcoxon p < 0.05 over the 60 series (mean over seeds) |
| E2 | consistent over families | mean gain positive in at least 7 of the 12 families |
| E3 (mechanism) | larger gain when the branches differ more in scale | Spearman(abs log(SD error branch / SD classify branch) on train, gain) > 0, one-sided p < 0.05 |
| E4 | M12 locked > CoAD (and, secondary, > `calibrated`) | as E1 |

No comparison with Table 1 here (Table 1 covers the 64 series only).

**Two notebooks at the same time:** cell 16.0 with `SETUP_TAG = 'gpu-a'` in one and `'gpu-b'` in the other, then
16.12 (for the M12 functions), 16.18, 16.19 in both; each takes half of the (series, seed) pairs, fixed by
`(group + seed) % 2`. 16.20 when both are done. Run 16.18 first in one notebook: the second one then reads the
same `eda/m13_external.csv`.

In [ ]:
import hashlib
from scipy.stats import spearmanr

M13_FILE = STORE / 'eda' / 'm13_external.csv'
M13_EXCLUDED = {'MGAB', 'SED', 'SVDB', 'IOPS', 'TODS'}        # the 64 series of the paper
M13_PER_FAMILY, M13_MIN_FAMILY, M13_MAX_LENGTH = 5, 9, 100_000

def m13_valid(series):
    """Window settings usable by the model (no label values used)."""
    return series['window'] >= 2 * PATCH and len(series['train']) >= series['window'] and series['period'] >= 2

if not M13_FILE.exists():
    files = sorted(RAW.glob('*.csv'))
    catalog = pd.DataFrame([dict(group=int(f.name.split('_')[0]), family=f.name.split('_')[1], file=f.name) for f in files])
    catalog = catalog[~catalog['family'].isin(M13_EXCLUDED)]
    sizes = catalog['family'].value_counts()
    catalog = catalog[catalog['family'].isin(sizes[sizes >= M13_MIN_FAMILY].index)]
    catalog['length'] = [sum(1 for _ in open(RAW / f)) - 1 for f in catalog['file']]
    # TSB-AD-U holds byte-identical copies (Exathlon, YAHOO): keep the lowest id of each content
    catalog['md5'] = [hashlib.md5((RAW / f).read_bytes()).hexdigest() for f in catalog['file']]
    n_before = len(catalog)
    catalog = catalog.sort_values('group').drop_duplicates('md5', keep='first')
    print(n_before - len(catalog), 'duplicate files dropped')
    sizes = catalog['family'].value_counts()
    catalog = catalog[catalog['family'].isin(sizes[sizes >= M13_MIN_FAMILY].index)]
    catalog = catalog[catalog['length'] <= M13_MAX_LENGTH]
    rng = np.random.default_rng(2027)
    rows = []
    for family in sorted(catalog['family'].unique()):
        groups = sorted(catalog.loc[catalog['family'] == family, 'group'])
        kept = []
        for g in rng.permutation(groups):
            try:
                ok = m13_valid(load_series(int(g)))
            except Exception:
                ok = False
            if ok:
                kept.append(int(g))
            if len(kept) == M13_PER_FAMILY:
                break
        rows += [dict(group=g, family=family, length=int(catalog.set_index('group').loc[g, 'length'])) for g in kept]
    pd.DataFrame(rows).to_csv(M13_FILE, index=False)
m13_series = pd.read_csv(M13_FILE)
M13_GROUPS = m13_series['group'].astype(int).tolist()
m13_family = m13_series.set_index('group')['family']
print(len(M13_GROUPS), 'external series:', m13_series.groupby('family').size().to_dict())

M13_RESULTS = RESULTS_DIR / f'm13_external_{SESSION_TAG}.csv'
M13_RUNS = STORE / 'runs' / 'm13'
M13_RUNS.mkdir(parents=True, exist_ok=True)
M13_COLUMNS = (['run_id', 'group', 'seed', 'family', 'variant', 'status', 'error'] + METRIC_KEYS
               + ['top_k_normal_share', 'log_sd_ratio', 'train_seconds'])
M13_SPLIT = {'gpu-a': 0, 'gpu-b': 1}       # two notebooks; any other SESSION_TAG runs every (series, seed)

def m13_mine(group, seed):
    part = M13_SPLIT.get(SESSION_TAG)
    return part is None or (int(group) + int(seed)) % 2 == part

def read_m13():
    files = sorted(RESULTS_DIR.glob('m13_external_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M13_COLUMNS)

M13_SEEDS = SEEDS        # [46, 47, 48]; seed 46 is always trained first, so stopping early still leaves complete seeds
mine = [(g, s) for s in M13_SEEDS for g in M13_GROUPS if m13_mine(g, s)]
print('SESSION_TAG', SESSION_TAG, '->', len(mine), 'of', len(M13_GROUPS) * len(SEEDS), '(series, seed) pairs in this notebook')

### 16.19. M13: train and score (re-run to continue)

One code-mode model per (series, seed) of this notebook, trained as in part 7; B0 is checked against the score of
`anomaly_score` (part 6). A series whose training or scoring fails gets a `failed` row and is retried once at the next run of this
cell; after two failures it is not retried again (reported in 16.20). Rows in `results/m13_external_<SESSION_TAG>.csv`, models in `runs/m13/`.

In [ ]:
assert (REPORTS / 'm12_locked.json').exists(), 'M13 uses the M12 locked pair: m12_locked.json is missing (16.14)'
m13_pair = tuple(json.loads((REPORTS / 'm12_locked.json').read_text())['locked_pair'])
missing = [n for n in ['train_model', 'load_series', 'anomaly_score', 'score_metrics', 'alarm_stats', 'm11_sources',
                       'm11_variants', 'm12_branches', 'm12_scores'] if n not in globals()]
assert not missing, f'not defined: {missing}. Run 16.0 (current version) and 16.12 first'
done = read_m13()
if len(done):
    ok_pairs = set(zip(done.loc[done['status'] == 'success', 'group'].astype(int), done.loc[done['status'] == 'success', 'seed'].astype(int)))
    n_failed = done[done['status'] == 'failed'].groupby(['group', 'seed']).size()
    given_up = {(int(g), int(s)) for (g, s), n in n_failed.items() if n >= 2}     # failed twice: not retried again
    done = ok_pairs | given_up
else:
    done = set()
todo = [(g, s) for g, s in mine if (g, s) not in done]
print(len(todo), 'models to do in this notebook | M12 locked pair:', m12_name(m13_pair))
session_start = time.time()
for i, (group, seed) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_m13_{group}_s{seed}"
    base = dict(run_id=run_id, group=group, seed=seed, family=m13_family[group])
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=True, seed=seed, log_every=0)
        train_seconds = round(time.time() - start, 1)
        out = M13_RUNS / run_id
        out.mkdir(parents=True)
        torch.save(model.state_dict(), out / 'model.pt')
        history.to_csv(out / 'loss.csv', index=False)
        period = series['period']
        src = m11_sources(model, series)
        b = m11_variants(src, period, combos=[])[0]
        gap = float(np.abs(b['B0 CoAD'] - anomaly_score(model, series)).max())
        assert gap < 1e-4, f'B0 differs from anomaly_score by {gap:.1e}'
        tr = src['train']
        sd_error = smooth(np.abs(tr['error'] - tr['error'].mean()), period).std()
        sd_classify = smooth(np.abs(tr['combined'] - tr['combined'].mean()), period).std()
        log_ratio = float(np.log((sd_error + EPS) / (sd_classify + EPS)))
        scores = {'B0 CoAD': b['B0 CoAD'], 'B1 calibrated': b['B1 calibrated'],
                  'M12 locked': m12_scores(m12_branches(src, period), [m13_pair])[m12_name(m13_pair)]}
        rows = []
        for name, score in scores.items():
            rows.append(dict(base, variant=name, status='success', **score_metrics(score, series),
                             top_k_normal_share=alarm_stats(series, score)['top_k_normal_share'],
                             log_sd_ratio=log_ratio, train_seconds=train_seconds))
        message = ' | '.join(f"{r['variant'].split()[0]} {100 * r['VUS_PR']:.2f}" for r in rows)
    except Exception as error:
        if isinstance(error, NameError):      # a setup problem, not a property of the series: stop, write nothing
            raise
        rows = [dict(base, variant='-', status='failed', error=f'{type(error).__name__}: {error}'[:300])]
        message = 'FAILED: ' + rows[0]['error']
    pd.DataFrame(rows).reindex(columns=M13_COLUMNS).to_csv(M13_RESULTS, mode='a', header=not M13_RESULTS.exists(), index=False)
    print(f'[{i}/{len(todo)}] {m13_family[group]} {group} seed {seed}: VUS-PR {message} | {time.time() - start:.0f} s', flush=True)

### 16.20. M13 results and predictions E1-E4

In [ ]:
m13 = read_m13()
m13 = pd.concat([m13[m13['status'] == 'failed'],
                 m13[m13['status'] == 'success'].drop_duplicates(['group', 'seed', 'variant'], keep='last')])
m13 = m13[m13['seed'].astype(int).isin(M13_SEEDS)]
ok = m13[m13['status'] == 'success'].astype({'group': int, 'seed': int})
ok_pairs = set(zip(ok['group'], ok['seed']))
bad = m13[(m13['status'] == 'failed') & [(int(g), int(s)) not in ok_pairs for g, s in zip(m13['group'], m13['seed'])]]
n_bad = bad.groupby(['group', 'seed']).size()
failed = bad[[n_bad[(g, s)] >= 2 for g, s in zip(bad['group'], bad['seed'])]].drop_duplicates(['group', 'seed'], keep='last')
n_pairs = len(M13_GROUPS) * len(M13_SEEDS)
n_done = len(ok_pairs) + len(failed)
print(f'Seeds analysed: {M13_SEEDS} | {n_done} of {n_pairs} (series, seed) pairs done, {len(failed)} failed')
if n_done < n_pairs:
    print('WARNING: not finished (run 16.19 in both notebooks); the numbers below are provisional')
if len(failed):
    display(failed[['group', 'seed', 'family', 'error']])
VARIANTS13 = ['B0 CoAD', 'B1 calibrated', 'M12 locked']
complete = ok.groupby('group')['variant'].nunique()
series_ok = complete[complete == len(VARIANTS13)].index
assert len(series_ok) >= 6, f'only {len(series_ok)} series with results for seeds {M13_SEEDS}: run 16.19 first'
rows = []
for v in VARIANTS13:
    d = ok[(ok['variant'] == v) & ok['group'].isin(series_ok)]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
print(f'{len(series_ok)} series with every variant, seeds {sorted(ok["seed"].unique())}')
display(pd.DataFrame(rows).set_index('variant').round(3))

per13 = 100 * ok[ok['group'].isin(series_ok)].pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
def one_sided(diff):
    return wilcoxon(diff, alternative='greater').pvalue if (diff != 0).any() else 1.0
tests13 = []
for v, ref in [('B1 calibrated', 'B0 CoAD'), ('M12 locked', 'B0 CoAD'), ('M12 locked', 'B1 calibrated')]:
    d = per13[v] - per13[ref]
    tests13.append({'variant': v, 'vs': ref, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                    'p (one-sided)': one_sided(d)})
display(pd.DataFrame(tests13).round(4))
gain = per13['B1 calibrated'] - per13['B0 CoAD']
by_family = pd.DataFrame({'B0 CoAD': per13['B0 CoAD'], 'B1 - B0': gain, 'M12 - B0': per13['M12 locked'] - per13['B0 CoAD'],
                          'M12 - B1': per13['M12 locked'] - per13['B1 calibrated']}).groupby(m13_family.reindex(per13.index).values).mean()
print('VUS-PR by family (mean over series and seeds):')
display(by_family.round(2))

ratio = ok[ok['variant'] == 'B0 CoAD'].groupby('group')['log_sd_ratio'].mean().abs().reindex(gain.index)
rho, p_rho = spearmanr(ratio, gain)
p_rho = p_rho / 2 if rho > 0 else 1 - p_rho / 2
e1 = one_sided(gain)
e2 = int((by_family['B1 - B0'] > 0).sum())
d4 = per13['M12 locked'] - per13['B0 CoAD']
print()
print(f'E1 (calibrated > CoAD): {gain.mean():+.2f}, higher on {int((gain > 0).sum())}/{len(gain)}, p = {e1:.4f} -> '
      + ('HOLDS' if e1 < 0.05 and gain.mean() > 0 else 'FAILS'))
print(f'E2 (gain positive in >= 7 of {len(by_family)} families): {e2} -> ' + ('HOLDS' if e2 >= 7 else 'FAILS'))
print(f'E3 (Spearman |log SD ratio| vs gain > 0): rho = {rho:+.3f}, one-sided p = {p_rho:.4f} -> '
      + ('HOLDS' if rho > 0 and p_rho < 0.05 else 'FAILS'))
print(f'E4 (M12 locked > CoAD): {d4.mean():+.2f}, higher on {int((d4 > 0).sum())}/{len(d4)}, p = {one_sided(d4):.4f} -> '
      + ('HOLDS' if one_sided(d4) < 0.05 and d4.mean() > 0 else 'FAILS'))
pd.DataFrame(rows).to_csv(REPORTS / 'm13_external.csv', index=False)
by_family.to_csv(REPORTS / 'm13_by_family.csv')

### 16.21. After M13: post-hoc analyses (hypotheses for the next test, not confirmations)

In [ ]:
# Run after 16.20 (needs per13, gain, ok, m13_family, M13_GROUPS). Post-hoc analyses: hypotheses for the next experiment, not confirmations.
signed = ok[ok['variant'] == 'B0 CoAD'].groupby('group')['log_sd_ratio'].mean().reindex(gain.index)
rho_s, p_s = spearmanr(signed, gain)
print(f'B1 - B0 vs signed log(SD error / SD classify): rho = {rho_s:+.3f}, p (two-sided) = {p_s:.2g}')
print('mean gain when error branch is narrower (log ratio < 0):', round(gain[signed < 0].mean(), 2), f'({int((signed < 0).sum())} series)',
      '| wider (> 0):', round(gain[signed > 0].mean(), 2), f'({int((signed > 0).sum())} series)')

print('Leave one family out, B1 - B0:')
fam13 = m13_family.reindex(gain.index)
display(pd.DataFrame([{'dropped': f, 'mean': gain[fam13 != f].mean(), 'p (one-sided)': one_sided(gain[fam13 != f])}
                      for f in sorted(fam13.unique())]).round(4))

def event_lengths(label):
    edges = np.diff(np.r_[0, np.asarray(label).astype(int), 0])
    return np.where(edges == -1)[0] - np.where(edges == 1)[0]
lengths = {}
for g in gain.index:
    ev = event_lengths(load_series(int(g))['label'])
    lengths[g] = np.median(ev) if len(ev) else np.nan
lengths = pd.Series(lengths)
d12 = per13['M12 locked'] - per13['B0 CoAD']
has = lengths.notna()
rho_l, p_l = spearmanr(lengths[has], d12[has])
print(f'M12 - B0 vs median anomaly length (test): rho = {rho_l:+.3f}, p = {p_l:.2g}, {int(has.sum())} series with anomalies')
bins = pd.cut(lengths, [0, 5, 50, 500, np.inf], labels=['1-5', '6-50', '51-500', '>500'])
display(pd.DataFrame({'M12 - B0': d12, 'B1 - B0': gain, 'length': bins}).groupby('length', observed=True)
        .agg(['mean', 'median', 'count']).round(2))

### 16.23. Context-mismatch (splice) test (re-run to continue)

**Question.** Does CoAD miss a segment whose values are normal but which does not fit its context (wrong phase or
wrong dynamics)? M9: on MGAB the raw reconstruction error finds only 19% of the events OE misses, while on SVDB and
IOPS it finds 90%. One candidate cause: the bidirectional GRU reconstructs a patch from both sides and can
interpolate a plausible segment over a subtle deviation. If CoAD misses context mismatches, a model change
(causal reconstruction, or a context-fit head as in NCAD) is worth trying; if not, both are dropped.

**Design** (inference on the saved code-mode models, no retraining, no labels except to keep away from labelled
anomalies):

* a segment of `L` points (`SPLICE_LENGTHS` x period, at least 8 points) at positions in the normal part of the test
  series, two windows apart, at least one window away from labelled anomalies, after the first period + window;
* the replaced content always comes from the **normal test values of the same series** (k = 1..4 periods later):
  * `matched` (control): among the in-phase donors `p + k * period`, the one closest to the original segment
    (RMSE); the most context-consistent normal segment available;
  * `shifted` (mismatch): among the out-of-phase donors `p + k * period + o` (o = 1/4, 1/2, 3/4 period), the one
    **most different** from the original segment; normal values, wrong phase / dynamics. (A plain half-period
    shift changed almost nothing on short noisy periods such as IOPS in the local test.)
  * `none`: no change (false-alarm level at the same positions);
* both ends are cross-faded over `w = max(2, L // 8)` points so the splice does not create a step; the remaining
  step is recorded (`jump_sd`, in units of the normal first-difference SD);
* all segments of one condition are inserted together (they are two windows apart, so no window sees two) and the
  whole modified series goes through the model;
* per segment and signal: peak percentile against the normal points of the **untouched** series, over the whole
  segment and over its **middle third** (away from both splice points). Found = percentile >= 0.99.

**Risks and how the design handles them:**

| Risk | Handling |
|---|---|
| The model reacts to the splice points, not to the context | cross-fade; middle-third measure; `jump_sd` reported; `matched` control has the same splice mechanics |
| "Wrong phase" is not defined for a chaotic series (MGAB) | `matched` is the closest normal segment; `shifted` is compared with it, not with "no change" |
| The test is too weak to show anything | sensitivity check C3 on the periodic families: if CoAD does not find `shifted` there either, C1-C2 are not interpreted |
| Artificial segments differ from real anomalies | stated as a limit; the test only decides whether to train a model change, it is not an improvement |
| Few series (MGAB has 9) | all 9 MGAB series x 3 seeds; per-series table; no claim beyond MGAB |
| Content difference, not context, drives detection | `rmse_sd` (donor vs original) reported per condition |

**Predictions (decision criteria) written before running**, raw reconstruction error, `L` = 1 period, middle third:

* (C1) on MGAB, found rate of `shifted` <= 0.30 (CoAD misses context mismatch);
* (C2) on MGAB, `shifted` - `matched` < 0.15 (barely separates mismatch from control);
* (C3, sensitivity) on the other families pooled, `shifted` - `matched` >= 0.30 (the test can show a mismatch).

Decision: C1 and C2 and C3 hold -> try the causal reconstruction (bidirectional=False) on MGAB; C1 or C2 fails ->
CoAD already sees context mismatch on MGAB: drop the context directions; C3 fails -> the test is uninformative.

Rows in `results/m14_splice_<SESSION_TAG>.csv`. Needs 14.1 and 16.1 (16.0 one-click setup is enough).

In [ ]:
# Self-contained after 16.1
family_of = manifest64.set_index('group')['family']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
M14_SPLICE = RESULTS_DIR / f'm14_splice_{SESSION_TAG}.csv'
SPLICE_LENGTHS = [0.5, 1]                      # segment length in periods
SPLICE_KINDS = ['none', 'matched', 'shifted']
SPLICE_SIGNALS = ['error', 'oe', 'residual', 'full_unsmoothed', 'full', 'calibrated']
SPLICE_MAX_SEGMENTS = 20
M14_COLUMNS = (['group', 'seed', 'family', 'kind', 'length_periods', 'length', 'n_segments', 'rmse_sd', 'jump_sd']
               + [f'{s}_{k}' for s in SPLICE_SIGNALS for k in ['found', 'found_middle', 'pct_middle']])

def splice_normal(series):
    """Test points usable as normal: at least one window away from labelled anomalies, after the first period."""
    window, period = series['window'], series['period']
    keep = np.ones(len(series['test']), bool)
    for a, b in segments(series['label']):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    return keep

def splice_length(series, length_periods):
    return max(8, int(round(length_periods * series['period'])))

def splice_plan(series, length_periods):
    """Positions and donors (no label values used beyond excluding labelled anomalies)."""
    x, period, window = series['test'], series['period'], series['window']
    L, n, keep = splice_length(series, length_periods), len(series['test']), splice_normal(series)
    offsets = sorted({max(1, period // 4), max(1, period // 2), max(1, 3 * period // 4)})
    rmse = lambda q: np.sqrt(np.mean((x[q:q + L] - x[p:p + L]) ** 2))
    plan, p = [], period + window
    while p + L + window < n and len(plan) < SPLICE_MAX_SEGMENTS:
        if keep[p - window:p + L + window].all():
            ok = lambda q: q + L <= n and keep[q:q + L].all()
            matched = [p + k * period for k in range(1, 5) if ok(p + k * period)]
            shifted = [p + k * period + o for k in range(1, 5) for o in offsets if ok(p + k * period + o)]
            if matched and shifted:
                # control: the closest in-phase normal segment; mismatch: the most different out-of-phase normal segment
                plan.append((p, min(matched, key=rmse), max(shifted, key=rmse)))
                p += L + 2 * window
                continue
        p += max(1, period)
    return plan, L

def splice(values, p, donor, L):
    """Replace values[p:p+L] by donor (length L), cross-fading over w points at both ends."""
    out = values.copy()
    w = max(2, L // 8)
    ramp = np.ones(L)
    ramp[:w] = np.linspace(0, 1, w + 2)[1:-1]
    ramp[-w:] = np.linspace(1, 0, w + 2)[1:-1]
    out[p:p + L] = (1 - ramp) * values[p:p + L] + ramp * donor
    return out

SPLICE_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    usable = [int(g) for g in d['group'] if len(splice_plan(load_series(int(g)), 1)[0]) >= 3]
    chosen = usable if family == 'MGAB' else usable[:2]
    print(f'{family}: splice series {chosen}')
    SPLICE_GROUPS += chosen

def read_splice():
    files = sorted(RESULTS_DIR.glob('m14_splice_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M14_COLUMNS)

done = read_splice()
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in SPLICE_GROUPS and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'models for the splice test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = code_model(run)
    x = series['test']
    base, train_parts = model_signals(model, series, values=x.copy())   # a copy: same processing as the spliced runs
    keep = splice_normal(series)
    pct = {s: percentile_against(base[s][keep]) for s in SPLICE_SIGNALS}
    sd, dsd = x[keep].std() + 1e-12, np.diff(x)[keep[1:]].std() + 1e-12
    rows = []
    for length_periods in SPLICE_LENGTHS:
        plan, L = splice_plan(series, length_periods)
        for kind in SPLICE_KINDS:
            values, rmse, jumps = x.copy(), [], []
            if kind != 'none':
                for p, q_matched, q_shifted in plan:
                    q = q_matched if kind == 'matched' else q_shifted
                    values = splice(values, p, x[q:q + L], L)
                    rmse.append(np.sqrt(np.mean((values[p:p + L] - x[p:p + L]) ** 2)) / sd)
                    jumps.append(max(abs(values[p] - values[p - 1]), abs(values[p + L] - values[p + L - 1])) / dsd)
                signals, _ = model_signals(model, series, values=values, train_parts=train_parts)
            else:
                signals = base
            row = dict(group=group, seed=seed, family=family_of[group], kind=kind, length_periods=length_periods,
                       length=L, n_segments=len(plan), rmse_sd=float(np.mean(rmse)) if rmse else 0.0,
                       jump_sd=float(np.mean(jumps)) if jumps else 0.0)
            third = max(1, L // 3)
            for s in SPLICE_SIGNALS:
                sig = signals[s]
                whole = np.array([pct[s](np.array([sig[p:p + L].max()]))[0] for p, _, _ in plan])
                middle = np.array([pct[s](np.array([sig[p + third:p + L - third].max()]))[0] for p, _, _ in plan])
                row[f'{s}_found'] = float((whole >= 0.99).mean())
                row[f'{s}_found_middle'] = float((middle >= 0.99).mean())
                row[f'{s}_pct_middle'] = float(middle.mean())
            rows.append(row)
    (pd.DataFrame(rows).reindex(columns=M14_COLUMNS)
     .to_csv(M14_SPLICE, mode='a', header=not M14_SPLICE.exists(), index=False))
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} conditions', flush=True)

### 16.24. Splice results and decision C1-C3

In [ ]:
sp = read_splice().drop_duplicates(['group', 'seed', 'kind', 'length_periods'], keep='last')
print(f"{sp['group'].nunique()} series, {sp[['group', 'seed']].drop_duplicates().shape[0]} models; "
      f"segments per series (median): {sp['n_segments'].median():.0f}")
print('Splice checks (mean): content change rmse_sd and remaining step jump_sd (1 = one normal step)')
display(sp[sp['kind'] != 'none'].groupby(['length_periods', 'kind'])[['rmse_sd', 'jump_sd']].mean().round(2))
sp['group_kind'] = np.where(sp['family'] == 'MGAB', 'MGAB', 'other')
for kind in ['found_middle', 'found']:
    print()
    print(f'{kind} (mean over series and seeds)')
    t = sp.groupby(['group_kind', 'length_periods', 'kind'])[[f'{s}_{kind}' for s in SPLICE_SIGNALS]].mean()
    t.columns = SPLICE_SIGNALS
    display(t.round(3))
print('Raw error found_middle by family (1 period):')
display(sp[sp['length_periods'] == 1].pivot_table(index='family', columns='kind', values='error_found_middle').round(3))
print('MGAB per series (raw error found_middle, 1 period, mean of seeds):')
display(sp[(sp['family'] == 'MGAB') & (sp['length_periods'] == 1)]
        .pivot_table(index='group', columns='kind', values='error_found_middle').round(3))

one = sp[sp['length_periods'] == 1].groupby(['group_kind', 'kind'])['error_found_middle'].mean()
assert {'MGAB', 'other'} <= set(one.index.get_level_values(0)), 'needs MGAB and other families: run 16.23 first'
c1 =one[('MGAB', 'shifted')]
c2 = one[('MGAB', 'shifted')] - one[('MGAB', 'matched')]
c3 = one[('other', 'shifted')] - one[('other', 'matched')]
print()
print(f'C1 (MGAB shifted <= 0.30): {c1:.3f} -> ' + ('HOLDS' if c1 <= 0.30 else 'FAILS'))
print(f'C2 (MGAB shifted - matched < 0.15): {c2:+.3f} -> ' + ('HOLDS' if c2 < 0.15 else 'FAILS'))
print(f'C3 (other families shifted - matched >= 0.30, sensitivity): {c3:+.3f} -> ' + ('HOLDS' if c3 >= 0.30 else 'FAILS'))
if c3 < 0.30:
    print('Decision: the test is uninformative (C3 fails); do not interpret C1-C2')
elif c1 <= 0.30 and c2 < 0.15:
    print('Decision: CoAD misses context mismatch on MGAB -> try causal reconstruction (bidirectional=False) on MGAB')
else:
    print('Decision: CoAD already sees context mismatch on MGAB -> drop the context directions')

### 16.25. M15: drift diagnosis of the false alarms (re-run to continue)

**Question.** Are CoAD's false alarms located where the local level or spread of the test series has moved away from
the training part (drift)? If so, a drift-aware model change in the spirit of MORA (a second reconstruction path with
a longer context, used to discount deviations that the longer context explains) is worth a pilot; if not, it is
dropped. M9: among the top-k points (k = number of anomalous points), 48% (SVDB) to 72% (TODS) are normal, but on
MGAB, SED and TODS almost all of them sit next to an event (spill-over of the smoothed score). False alarms **far**
from any event exist mainly on IOPS and SVDB, so the test looks at those.

**Design** (inference on the saved code-mode models, seed 46, no retraining; labels are used **only to diagnose**
which top-k points are false alarms, never in a score):

* models: the 64 paper series (code mode) and the 60 external M13 series (their models are in `runs/m13`), seed 46;
* score: the CoAD final score (`full`); top-k with k = number of anomalous test points (after the first period);
* **far false alarm**: a top-k point that is normal and more than one window from every labelled event;
  **reference normal points**: all other normal points more than one window from every event;
* drift at time t, from the data only: over the trailing window of `window` points (the model window),
  local mean `m(t)` and local log-SD `v(t)`; the same statistics on the training part give their mean and SD;
  `drift(t) = max(|z_level|, |z_scale|)`, the larger of the two standardised distances from the training values;
* per model: Mann-Whitney AUC of drift, far false alarms vs reference normal points (0.5 = no relation), the share
  of each group with drift > 3, and the share of all reference normal points with drift > 3 (how much the series
  drifts at all).

**Risks and how they are handled:**

| Risk | Handling |
|---|---|
| Spill-over next to events looks like drift | far false alarms and reference points are both more than one window from any event |
| A "false alarm" is an unlabelled anomaly | cannot be separated; stated as a limit (the result says where false alarms are, not that they are wrong) |
| Correlation, not cause: CoAD's score is a deviation from the training level, so it rises with drift by construction | the test only decides whether a drift-aware pilot is worth training; the pilot is the causal test |
| Few far false alarms make the AUC noisy | only models with at least 10 far false alarms enter R1-R2; per-family and per-series tables |
| The window choice drives the drift measure | fixed before running (the model window); not tuned |
| One seed | M9: top-k normal share is stable across seeds; seed fixed before running |

**Predictions (decision criteria) written before running**, on models with >= 10 far false alarms:

* (R1) median AUC (drift of far false alarms vs reference normal points) >= 0.70;
* (R2) in at least half of these models, the share with drift > 3 among far false alarms is >= 0.30 and at least
  twice the share among reference normal points;
* (R3, relevance) far false alarms are frequent enough to matter: on the pooled models, far false alarms are >= 15%
  of the top-k points.

Decision: R1 and R2 and R3 hold -> drift explains a meaningful part of the false alarms: pilot a MORA-style dual
context reconstruction on the drifting series; R1 or R2 fails -> drop the drift direction; R3 fails -> false alarms
far from events are too rare for this to matter.

Rows in `results/m15_drift_<SESSION_TAG>.csv`. Needs 14.1 and 16.1 (16.0 one-click setup is enough).

In [ ]:
from scipy.stats import mannwhitneyu
M15_DRIFT = RESULTS_DIR / f'm15_drift_{SESSION_TAG}.csv'
M15_SEED = 46
M15_COLUMNS = ['source', 'run_id', 'group', 'seed', 'family', 'k', 'top_k_normal_share', 'n_far', 'far_share',
               'auc', 'far_drift_median', 'ref_drift_median', 'far_out3', 'ref_out3']

def rolling_stats(x, w):
    """Trailing mean and log-SD over w points (NaN for the first w - 1 points)."""
    c1, c2 = np.cumsum(np.r_[0.0, x]), np.cumsum(np.r_[0.0, x * x])
    mean, out_v = np.full(len(x), np.nan), np.full(len(x), np.nan)
    s1, s2 = c1[w:] - c1[:-w], c2[w:] - c2[:-w]
    m = s1 / w
    mean[w - 1:] = m
    out_v[w - 1:] = 0.5 * np.log(np.maximum(s2 / w - m * m, 0) + 1e-12)
    return mean, out_v

def drift_of(series):
    w = series['window']
    m_tr, v_tr = rolling_stats(series['train'], w)
    m_te, v_te = rolling_stats(series['test'], w)
    z_level = np.abs(m_te - np.nanmean(m_tr)) / (np.nanstd(m_tr) + 1e-12)
    z_scale = np.abs(v_te - np.nanmean(v_tr)) / (np.nanstd(v_tr) + 1e-12)
    return np.fmax(z_level, z_scale)

def drift_row(series, score):
    label, period, window = np.asarray(series['label']).astype(int), series['period'], series['window']
    valid = np.arange(len(label)) >= max(period, window - 1)
    k = int(label[valid].sum())
    top = np.where(valid)[0][np.argsort(-score[valid])[:k]]
    near = np.zeros(len(label), bool)
    for a, b in segments(label):
        near[max(0, a - window):b + window] = True
    drift = drift_of(series)
    far_top = top[(label[top] == 0) & ~near[top]]
    is_top = np.zeros(len(label), bool)
    is_top[top] = True
    ref = np.where(valid & (label == 0) & ~near & ~is_top)[0]
    row = dict(k=k, top_k_normal_share=float((label[top] == 0).mean()) if k else np.nan,
               n_far=len(far_top), far_share=len(far_top) / max(k, 1),
               ref_drift_median=float(np.nanmedian(drift[ref])) if len(ref) else np.nan,
               ref_out3=float((drift[ref] > 3).mean()) if len(ref) else np.nan,
               auc=np.nan, far_drift_median=np.nan, far_out3=np.nan)
    if len(far_top) and len(ref):
        a, b = drift[far_top], drift[ref]
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        if len(a) and len(b):
            row['auc'] = float(mannwhitneyu(a, b, alternative='two-sided').statistic / (len(a) * len(b)))
            row['far_drift_median'] = float(np.median(a))
            row['far_out3'] = float((a > 3).mean())
    return row

# models: 64 paper series (code mode) + 60 external M13 series, seed M15_SEED
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
family64 = manifest64.set_index('group')['family']
jobs = [('tsbad64', r.run_id, int(r.group), int(r.seed), family64[int(r.group)], RUNS / r.run_id / 'model.pt')
        for r in code_runs.itertuples() if int(r.seed) == M15_SEED]
m13_files = sorted(RESULTS_DIR.glob('m13_external_*.csv'))
if m13_files:
    m13_runs = pd.concat([pd.read_csv(f) for f in m13_files], ignore_index=True)
    m13_runs = m13_runs[(m13_runs['status'] == 'success') & (m13_runs['variant'] == 'B0 CoAD')
                        & (m13_runs['seed'].astype(int) == M15_SEED)].drop_duplicates(['group', 'seed'], keep='last')
    jobs += [('m13', r.run_id, int(r.group), int(r.seed), r.family, STORE / 'runs' / 'm13' / r.run_id / 'model.pt')
             for r in m13_runs.itertuples()]
else:
    print('No M13 result files: only the 64 paper series')
missing = [j for j in jobs if not j[5].exists()]
if missing:
    print(f'WARNING: {len(missing)} saved models not found (skipped), e.g. {missing[0][5]}')
jobs = [j for j in jobs if j[5].exists()]

def read_drift():
    files = sorted(RESULTS_DIR.glob('m15_drift_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M15_COLUMNS)

done = set(read_drift()['run_id'])
todo = [j for j in jobs if j[1] not in done]
print(f'{len(jobs)} models ({sum(j[0] == "tsbad64" for j in jobs)} paper, {sum(j[0] == "m13" for j in jobs)} M13); {len(todo)} to do')
session_start = time.time()
for i, (source, run_id, group, seed, family, path) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    series = load_series(group)
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    signals, _ = model_signals(model, series)
    row = dict(source=source, run_id=run_id, group=group, seed=seed, family=family, **drift_row(series, signals['full']))
    pd.DataFrame([row]).reindex(columns=M15_COLUMNS).to_csv(M15_DRIFT, mode='a', header=not M15_DRIFT.exists(), index=False)
    print(f'[{i}/{len(todo)}] {source} {family} {group}: far false alarms {row["n_far"]}, AUC {row["auc"]:.2f}', flush=True)

### 16.26. Drift results and decision R1-R3

In [ ]:
dr = read_drift().drop_duplicates('run_id', keep='last')
print(f"{len(dr)} models ({(dr['source'] == 'tsbad64').sum()} paper, {(dr['source'] == 'm13').sum()} M13)")
print('Per family (mean): top-k normal share, far share of top-k, AUC, drift > 3 among far false alarms / reference')
display(dr.groupby(['source', 'family'])[['top_k_normal_share', 'far_share', 'n_far', 'auc', 'far_out3', 'ref_out3']]
        .mean().round(3))
use = dr[dr['n_far'] >= 10].dropna(subset=['auc'])
print(f'{len(use)} models with >= 10 far false alarms')
display(use[['source', 'family', 'group', 'n_far', 'auc', 'far_drift_median', 'ref_drift_median', 'far_out3', 'ref_out3']]
        .sort_values(['source', 'family', 'auc']).round(3))
r1 = use['auc'].median() if len(use) else np.nan
r2 = ((use['far_out3'] >= 0.30) & (use['far_out3'] >= 2 * use['ref_out3'])).mean() if len(use) else np.nan
r3 = dr['n_far'].sum() / max(dr['k'].sum(), 1)
print()
print(f'R1 (median AUC >= 0.70): {r1:.3f} -> ' + ('HOLDS' if r1 >= 0.70 else 'FAILS'))
print(f'R2 (share of models with far_out3 >= 0.30 and >= 2 x ref_out3, needs >= 0.5): {r2:.3f} -> ' + ('HOLDS' if r2 >= 0.5 else 'FAILS'))
print(f'R3 (far false alarms / top-k points, pooled, >= 0.15): {r3:.3f} -> ' + ('HOLDS' if r3 >= 0.15 else 'FAILS'))
if r3 < 0.15:
    print('Decision: far false alarms are too rare to matter; drop the drift direction')
elif r1 >= 0.70 and r2 >= 0.5:
    print('Decision: drift explains a meaningful part of the false alarms -> pilot a dual-context (MORA-style) reconstruction')
else:
    print('Decision: false alarms are not explained by drift -> drop the drift direction')

### 16.27. M16: residual branch independent of OE, four sources of information (re-run to continue)

**Weakness (error analysis M9, 8 262 real events).** The residual classifier is meant to be a second opinion, but it
is blind where OE is blind: when OE finds an event, the residual classifier finds 81-97% of them (IOPS, MGAB, SVDB);
when OE misses it, only 3-24%, although the raw reconstruction error finds 90% of those on SVDB and IOPS. Of the
2 915 events that OE misses and the raw error finds, the residual classifier finds 11%.

**Mechanism (code).** `residual_logit(x, x_rec)` classifies `encoder(x) - encoder(x_rec)`: the difference of two
features of **the encoder that OE uses**, trained only to recognise the four synthetic distortions. If those features
do not represent what is unusual about an event (OE misses it), the two features are close and the residual is
small, even when `x` and `x_rec` differ clearly.

**What changes.** Only the **source of information of the residual branch**. Four new residual heads, each with its
own layers (nothing shared with OE), trained the same way:

| Head | Input per patch (time values + STFT) | Reference |
|---|---|---|
| `diff` | `x - x_rec` (the residual in the data) | the residual itself, without OE's features |
| `concat` | `x` and `x_rec` side by side; the head learns how to compare them | DRAEM (Zavrtanik et al., ICCV 2021): discriminative network on the input and its reconstruction, trained on synthetic anomalies |
| `focus` | `x` and the error map `abs(x - x_rec)` | TranAD (Tuli et al., VLDB 2022; [48] of CoAD): reconstruction error of a first pass as a conditioning input ("focus score") |
| `own_feature` | `E(x) - E(x_rec)` with a **new** encoder `E` trained only for the residual | control: same form as CoAD, without sharing OE's encoder (is the sharing the problem, or the feature difference itself?) |

Everything else of CoAD is kept: OE, the reconstruction network, soft masking, the second reconstruction masked with
`A_c = (OE + new residual) / 2`, and the scoring.

**Protocol** (as M11: design, then confirmation; no labels of real anomalies are used to build anything):

* saved code-mode models, seed 46, **frozen**; only the new heads are trained, on the training windows, with CoAD's
  own synthetic distortions and layout (clean copy + distorted copy, patch labels, BCE with `pos_weight = 2`),
  `R16_EPOCHS` epochs, seed 46; their input is the first-pass reconstruction of the frozen model, as in CoAD's
  training; inputs are scaled by statistics of the clean training windows;
* `R16_ROLE = 'development'`: the 30 development series of M11, all four heads; then 16.28 **locks one head** with a
  rule fixed now: among the heads with P1 >= 0.50, the highest mean VUS-PR of `<head> calibrated`
  (`reports/m16_locked.json`);
* `R16_ROLE = 'confirmation'`: the 34 confirmation series of M11, the locked head only; P2 and P3 are judged there;
* score variants from the same model (code-mode processing: deviation from the training mean, moving average over
  one period, first period zeroed): `B0 CoAD`, `B1 calibrated`, `B1 error x2` (control: reconstruction weighted
  2 : 1, "is the new head only the error counted twice?"), and per head `<head> CoAD` and `<head> calibrated`;
* event check on the real events (labels used **only to evaluate**): peak percentile of each point signal against the
  normal test points (more than one window from any event, after the first period); found = percentile >= 0.99.

**Predictions written before running:**

* (P1, mechanism; per head, development) on the events that OE misses and the raw reconstruction error finds (both
  measured in this run), pooled: the new residual finds >= 0.50 (old residual: 0.11 in M9);
* (P2, confirmation, locked head) VUS-PR: `<head> calibrated` > `B1 calibrated` **and** > `B1 error x2`, Wilcoxon
  one-sided, Holm over the two tests, p < 0.05;
* (P3, confirmation) mean top-k normal share of `<head> calibrated` minus `B1 calibrated` <= 0.05.

Decision: no head passes P1 -> heads trained on synthetic distortions do not escape OE's blind spot: stop; P1 holds
but P2 fails on confirmation -> the new residual adds nothing beyond the error branch: stop; all hold -> joint training
with the locked head, then a new external set.

**Risks:**

| Risk | Handling |
|---|---|
| A head only learns the size of `x - x_rec`, i.e. duplicates the reconstruction branch | control `B1 error x2` in P2 |
| The heads only learn the synthetic distortions | P1 measures it directly on real events |
| Choosing the best of four heads overfits the development series | one head locked by a rule fixed before running; judged on the confirmation half |
| Frozen pilot differs from joint training (the soft-mask loop changes) | the pilot only decides whether joint training is worth it |
| Short anomalies (TODS, IOPS) are lost by the moving average anyway | stated; reported per family |
| MGAB: the raw error itself finds only 19% | stated; no gain expected there |
| Development and confirmation series were already used in M11 / M12 | any claim needs a new external set |
| Head size, epochs or learning rate tuned after seeing results | `R16_EPOCHS`, `R16_HIDDEN`, `R16_LR` fixed before running |

Rows in `results/m16_residual_<SESSION_TAG>.csv`. Needs 14.1 and 16.1 (16.0 one-click setup is enough).

In [ ]:
R16_ROLE = 'development'          # 'development': four heads on 30 series; 'confirmation': locked head on 34 series
M16_RESULTS = RESULTS_DIR / f'm16_residual_{SESSION_TAG}.csv'
M16_LOCK = REPORTS / 'm16_locked.json'
R16_SEED, R16_EPOCHS, R16_HIDDEN, R16_LR = 46, 30, 64, 1e-3
R16_ALL_HEADS = ['diff', 'concat', 'focus', 'own_feature']
if R16_ROLE == 'confirmation':
    assert M16_LOCK.exists(), 'run the development pilot and 16.28 first (it writes reports/m16_locked.json)'
    R16_HEADS = [json.loads(M16_LOCK.read_text())['locked_head']]
else:
    R16_HEADS = R16_ALL_HEADS
R16_BASE = ['B0 CoAD', 'B1 calibrated', 'B1 error x2']
R16_EVENT_KEYS = ['n_events', 'n_target', 'target_old', 'all_old'] + [f'{k}_{h}' for h in R16_ALL_HEADS
                                                                     for k in ['target', 'all', 'oe_found']]
M16_COLUMNS = (['run_id', 'role', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share']
               + R16_EVENT_KEYS + [f'loss_{h}' for h in R16_ALL_HEADS] + ['seconds'])
R16_FEAT = PATCH + PATCH * 2 * (N_FFT // 2 + 1)     # time values + STFT of one signal, per patch

def r16_patches(v):
    """(B, T) -> (B, N, R16_FEAT): time values and STFT of each patch."""
    time_patches, freq_patches = make_patches(v, stft_features(v))
    return torch.cat([time_patches, freq_patches], -1)

class R16Encoder(nn.Module):
    def __init__(self, n_in, hidden=R16_HIDDEN):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(n_in, hidden), nn.SELU())
        self.gru = nn.GRU(hidden, hidden, num_layers=1, batch_first=True, bidirectional=True)

    def forward(self, f):
        return self.gru(self.proj(f))[0]          # (B, N, 2 * hidden)

class R16Residual(nn.Module):
    """New residual branch: its own encoder and classifier; `kind` sets the source of information."""
    def __init__(self, kind, hidden=R16_HIDDEN):
        super().__init__()
        self.kind = kind
        n_in = {'diff': R16_FEAT, 'concat': 2 * R16_FEAT, 'focus': R16_FEAT + PATCH, 'own_feature': R16_FEAT}[kind]
        self.encoder = R16Encoder(n_in, hidden)
        self.classifier = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.SELU(), nn.Linear(hidden, 1))

    def forward(self, x, x_rec, stats):
        """x, x_rec (B, T) raw values -> logits (B, N, 1). stats: scales from the clean training windows."""
        xs, rs = (x - stats['mean']) / stats['sd'], (x_rec - stats['mean']) / stats['sd']
        r = (x - x_rec) / stats['res_sd']
        if self.kind == 'diff':
            h = self.encoder(r16_patches(r))
        elif self.kind == 'concat':
            h = self.encoder(torch.cat([r16_patches(xs), r16_patches(rs)], -1))
        elif self.kind == 'focus':
            B, T = r.shape
            h = self.encoder(torch.cat([r16_patches(xs), r.abs().reshape(B, T // PATCH, PATCH)], -1))
        else:                                        # own_feature: E(x) - E(x_rec), E not shared with OE
            h = self.encoder(r16_patches(xs)) - self.encoder(r16_patches(rs))
        return self.classifier(h)

@torch.no_grad()
def r16_first_pass(model, x):
    oe = torch.sigmoid(model.classifier(x))
    return oe, model.mae(x, oe)

def r16_train_heads(model, series, seed, kinds):
    """Train the new residual heads on the frozen model, on the same batches, with CoAD's synthetic distortions."""
    set_seed(seed)
    model.eval()
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    with torch.no_grad():                              # scales from the clean training windows (no labels)
        w = torch.from_numpy(windows).to(DEVICE)
        res = torch.cat([(b - r16_first_pass(model, b)[1]).flatten() for b in w.split(2 * BATCH_SIZE)])
        stats = dict(mean=float(w.mean()), sd=float(w.std()) + 1e-8, res_sd=float(res.std()) + 1e-8)
    heads = {k: R16Residual(k).to(DEVICE) for k in kinds}
    optimizers = {k: torch.optim.Adam(h.parameters(), lr=R16_LR) for k, h in heads.items()}
    pos_weight = torch.tensor([2.0], device=DEVICE)
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    losses = {k: [] for k in kinds}
    for epoch in range(R16_EPOCHS):
        totals, batches = {k: 0.0 for k in kinds}, 0
        for (clean,) in loader:
            x, labels, _ = training_batch(clean.to(DEVICE), series['period'], True)   # clean copy + distorted copy
            _, x_rec = r16_first_pass(model, x)
            y = patch_labels(labels)
            for k, h in heads.items():
                h.train()
                loss = F.binary_cross_entropy_with_logits(h(x, x_rec, stats), y, pos_weight=pos_weight)
                optimizers[k].zero_grad(set_to_none=True)
                loss.backward()
                optimizers[k].step()
                totals[k] += loss.item()
            batches += 1
        for k in kinds:
            losses[k].append(totals[k] / max(batches, 1))
    for h in heads.values():
        h.eval()
    return heads, stats, losses

@torch.no_grad()
def r16_components(model, heads, stats, values, window, stride):
    """Per-point OE, residuals, A_c and second-pass reconstructions: old (CoAD) and one per new head."""
    windows, pad = make_windows(values, window, stride)
    parts = {}
    def add(key, value, points=False):
        parts.setdefault(key, []).append((value if points else value.repeat_interleave(PATCH, dim=1).squeeze(-1)).cpu())
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        oe, first = r16_first_pass(model, x)
        res_old = torch.sigmoid(model.classifier.residual_logit(x, first))
        add('oe', oe)
        add('res_old', res_old)
        add('comb_old', (oe + res_old) / 2)
        add('rec_old', model.mae(x, (oe + res_old) / 2), points=True)          # D10 second pass
        for k, h in heads.items():
            res_new = torch.sigmoid(h(x, first, stats))
            add(f'res_{k}', res_new)
            add(f'comb_{k}', (oe + res_new) / 2)
            add(f'rec_{k}', model.mae(x, (oe + res_new) / 2), points=True)
    return {k: windows_to_series(torch.cat(v).numpy(), len(values), stride, pad) for k, v in parts.items()}

def r16_scores(test, train, series, kinds):
    """Point scores: baselines from the old branch, two variants per new head (code-mode processing)."""
    period, x, x_tr = series['period'], series['test'], series['train']
    out = {}
    for tag in ['old'] + list(kinds):
        r, r_tr = processed(np.abs(x - test[f'rec_{tag}']), np.abs(x_tr - train[f'rec_{tag}']), period)
        c, c_tr = processed(test[f'comb_{tag}'], train[f'comb_{tag}'], period)
        s_r, s_c = r_tr.std() + 1e-12, c_tr.std() + 1e-12
        if tag == 'old':
            out['B0 CoAD'], out['B1 calibrated'] = (r + c) / 2, (r / s_r + c / s_c) / 2
            out['B1 error x2'] = (2 * r / s_r + c / s_c) / 3
        else:
            out[f'{tag} CoAD'], out[f'{tag} calibrated'] = (r + c) / 2, (r / s_r + c / s_c) / 2
    return out

def r16_events(test, series, kinds):
    """Event check on the real events (evaluation only)."""
    label, period, window = np.asarray(series['label']).astype(int), series['period'], series['window']
    events = segments(label)
    ref = np.zeros(len(label), bool)
    ref[period:] = True
    for a, b in events:
        ref[max(0, a - window):b + window] = False
    out = {k: np.nan for k in R16_EVENT_KEYS}
    if not ref.any() or not events:
        return out
    signals = dict(oe=test['oe'], error=np.abs(series['test'] - test['rec_old']), res_old=test['res_old'],
                   **{f'res_{k}': test[f'res_{k}'] for k in kinds})
    found = {k: np.array([percentile_against(v[ref])(np.array([v[a:b].max()]))[0] >= 0.99 for a, b in events])
             for k, v in signals.items()}
    target = ~found['oe'] & found['error']
    out.update(n_events=len(events), n_target=int(target.sum()), target_old=int((found['res_old'] & target).sum()),
               all_old=int(found['res_old'].sum()))
    for k in kinds:
        f = found[f'res_{k}']
        out.update({f'target_{k}': int((f & target).sum()), f'all_{k}': int(f.sum()),
                    f'oe_found_{k}': int((f & found['oe']).sum())})
    return out

split = pd.read_csv(M11_SPLIT)
R16_GROUPS = split.loc[split['role'] == R16_ROLE, 'group'].astype(int).tolist()
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
family_of = manifest64.set_index('group')['family']

def read_m16():
    files = sorted(RESULTS_DIR.glob('m16_residual_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M16_COLUMNS)

done = read_m16()
done = set(done.loc[done['role'] == R16_ROLE, 'run_id'])
todo = [r for r in code_runs.itertuples()
        if int(r.group) in R16_GROUPS and int(r.seed) == R16_SEED and r.run_id not in done]
print(f'{R16_ROLE}: {len(R16_GROUPS)} series, heads {R16_HEADS}; {len(todo)} models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    t0 = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = code_model(run)
    for p in model.parameters():
        p.requires_grad_(False)
    r16_heads, r16_stats, r16_losses = r16_train_heads(model, series, seed, R16_HEADS)
    test = r16_components(model, r16_heads, r16_stats, series['test'], series['window'], series['period'])
    train = r16_components(model, r16_heads, r16_stats, series['train'], series['window'], series['period'])
    scores = r16_scores(test, train, series, R16_HEADS)
    assert np.abs(scores['B0 CoAD'] - anomaly_score(model, series)).max() < 1e-4, 'B0 does not reproduce the CoAD score'
    events = r16_events(test, series, R16_HEADS)
    losses = {f'loss_{k}': f'{v[0]:.4f}->{v[-1]:.4f}' for k, v in r16_losses.items()}
    rows = [dict(run_id=run.run_id, role=R16_ROLE, group=group, seed=seed, family=family_of[group], variant=v,
                 **score_metrics(s, series), top_k_normal_share=alarm_stats(series, s)['top_k_normal_share'],
                 **events, **losses, seconds=round(time.time() - t0, 1)) for v, s in scores.items()]
    (pd.DataFrame(rows).reindex(columns=M16_COLUMNS)
     .to_csv(M16_RESULTS, mode='a', header=not M16_RESULTS.exists(), index=False))
    vus = {r['variant']: round(100 * r['VUS_PR'], 1) for r in rows if r['variant'].endswith('calibrated')}
    tgt = {k: events.get(f'target_{k}') for k in R16_HEADS}
    print(f"[{i}/{len(todo)}] {family_of[group]} {group} ({time.time() - t0:.0f} s): target events {events['n_target']}, "
          f"found old {events['target_old']}, new {tgt}; VUS-PR {vus}", flush=True)

### 16.28. Pilot results, head lock (development) and decision P1-P3

In [ ]:
m16 = read_m16().drop_duplicates(['run_id', 'role', 'variant'], keep='last')
for role_name in ['development', 'confirmation']:
    d = m16[m16['role'] == role_name]
    if not len(d):
        continue
    heads = [h for h in R16_ALL_HEADS if f'{h} calibrated' in set(d['variant'])]
    variants = R16_BASE + [f'{h} {s}' for h in heads for s in ['CoAD', 'calibrated']]
    per = 100 * d.pivot_table(index='group', columns='variant', values='VUS_PR')
    print(f"===== {role_name}: {per.shape[0]} series (seed {R16_SEED}), heads {heads}")
    summary = d.groupby('variant')[list(METRICS4) + ['top_k_normal_share']].mean()
    summary[list(METRICS4)] *= 100
    display(summary.rename(columns=METRICS4).reindex(variants).round(3))
    fam = d.drop_duplicates('group').set_index('group')['family']
    print('VUS-PR by family:')
    display(per.groupby(fam.reindex(per.index).values).mean()[[v for v in variants if v in per]].round(2))
    ev = d.drop_duplicates('run_id')
    n_t, n_e = ev['n_target'].sum(), ev['n_events'].sum()
    p1 = {h: ev[f'target_{h}'].sum() / max(n_t, 1) for h in heads}
    print(f'Events: {n_e} events, {n_t} that OE misses and the raw error finds')
    display(pd.DataFrame({'found among OE-missed / error-found (P1)': {**{'old residual': ev['target_old'].sum() / max(n_t, 1)}, **p1},
                          'found among all events': {**{'old residual': ev['all_old'].sum() / max(n_e, 1)},
                                                     **{h: ev[f'all_{h}'].sum() / max(n_e, 1) for h in heads}}}).round(3))
    def one_sided(x):
        return wilcoxon(x, alternative='greater').pvalue if (x != 0).any() else 1.0
    rows = []
    for h in heads:
        for ref in ['B1 calibrated', 'B1 error x2']:
            x = per[f'{h} calibrated'] - per[ref]
            rows.append(dict(head=h, vs=ref, difference=x.mean(), higher=f'{int((x > 0).sum())}/{len(x)}', p=one_sided(x)))
    tests = pd.DataFrame(rows)
    display(tests.round(4))
    tk = d.groupby('variant')['top_k_normal_share'].mean()
    for h in heads:
        print(f"  {h}: P1 {p1[h]:.3f} ({'HOLDS' if p1[h] >= 0.5 else 'FAILS'}), "
              f"top-k normal share change {tk[f'{h} calibrated'] - tk['B1 calibrated']:+.3f}")
    if role_name == 'development':
        passing = [h for h in heads if p1[h] >= 0.5]
        if not passing:
            print('Decision: no head passes P1 -> heads trained on synthetic distortions do not escape OE blind spot: stop')
        elif M16_LOCK.exists():
            print('Locked head (fixed earlier, not changed):', json.loads(M16_LOCK.read_text())['locked_head'])
        else:
            locked = max(passing, key=lambda h: per[f'{h} calibrated'].mean())
            M16_LOCK.write_text(json.dumps({'locked_head': locked, 'p1': p1,
                                            'vus_calibrated': {h: per[f'{h} calibrated'].mean() for h in heads}}, indent=2))
            print(f'Locked head (rule: P1 >= 0.50, highest mean VUS-PR of <head> calibrated): {locked} -> {M16_LOCK.name}. '
                  "Next: set R16_ROLE = 'confirmation' in 16.27 and run 16.27, then 16.28")
    else:
        h = heads[0]
        t = tests[tests['head'] == h].copy()
        order = t['p'].sort_values().index
        t.loc[order, 'p Holm'] = np.maximum.accumulate([min(1, p * (len(t) - k)) for k, p in enumerate(t.loc[order, 'p'])])
        p2 = bool((t['p Holm'] < 0.05).all() and (t['difference'] > 0).all())
        p3 = tk[f'{h} calibrated'] - tk['B1 calibrated']
        print(f'Confirmation, head {h}:')
        display(t.round(4))
        print('P2 (> B1 calibrated and > B1 error x2, Holm p < 0.05): ' + ('HOLDS' if p2 else 'FAILS'))
        print(f'P3 (top-k normal share increase <= 0.05): {p3:+.3f} -> ' + ('HOLDS' if p3 <= 0.05 else 'FAILS'))
        print('Decision: ' + ('go -> joint training with this head, then a new external set' if p2 and p3 <= 0.05
                              else 'stop: the new residual does not improve on the calibrated baseline'))

### 16.29. M16b: residual head trained to complement OE (re-run to continue)

**Why.** M16 (16.27) changes only the **input** of the residual branch. The weakness measured in M9 is more specific:
the residual branch fails exactly where OE fails. M16b keeps the input of `diff` (`x - x_rec`) and changes the
**training objective**: synthetic anomalous patches that the frozen OE misses get a larger weight in the loss.

**Reference.** The idea of training a second model on the failures of a first one: Just Train Twice (Liu et al.,
ICML 2021: up-weight the examples that a first model misclassifies) and Learning from Failure (Nam et al., NeurIPS
2020); in general boosting (Freund and Schapire, 1997) and negative correlation learning (Liu and Yao, 1999).
Applied here to CoAD's residual branch with OE as the first model and **synthetic** labels instead of true labels:
this transfer is ours and may not work (OE failing on synthetic anomalies is not the same as failing on real ones).

**Design** (same frozen models, series, seed, distortions and settings as 16.27; written before 16.28 was read):

* two heads trained on the **same batches**: `diff` (as in 16.27, retrained here so that the comparison is paired:
  16.27 does not save its heads) and `comp`, same input and architecture;
* `comp` loss: BCE with `pos_weight = 2`, multiplied per patch by `w = 1 + R16B_GAMMA * (1 - p_OE)` on synthetic
  anomalous patches and `w = 1` on normal patches; `p_OE` is the frozen OE probability of that patch;
  `R16B_GAMMA = 4`, fixed before running;
* `comp` vs `diff` measures the contribution of the complementary objective alone;
* same score variants and event check as 16.27.

**Predictions written before running** (development series):

* (Q1) `comp` finds >= 0.50 of the events that OE misses and the raw error finds, and at least 0.10 more than `diff`;
* (Q2) mean VUS-PR of `comp calibrated` >= that of `diff calibrated`;
* (Q3) mean top-k normal share of `comp calibrated` - `diff calibrated` <= 0.05.

Decision: Q1, Q2, Q3 hold -> `comp` goes to the confirmation series (`R16B_ROLE = 'confirmation'`) and is judged
with P2 and P3 of 16.27; otherwise the complementary objective adds nothing: stop.

**When to run:** only if 16.28 (development) gives no head with P1 >= 0.50, or P1 close to it (0.30-0.50).

**Risks:**

| Risk | Handling |
|---|---|
| The head learns only the synthetic distortions that OE finds hard, not real anomalies | Q1 measures it on real events |
| Up-weighting anomalies raises false alarms | Q3 |
| gamma tuned after seeing results | fixed at 4 before running |
| Results of 16.27 seen before M16b runs | predictions and gamma written before 16.28 was read; `diff` is retrained here for a paired comparison |

Needs 16.27 run in the same session first (for its definitions; with all its models done it only prints
"0 models to do"). Rows in `results/m16b_comp_<SESSION_TAG>.csv`.

In [ ]:
assert 'R16Residual' in globals(), 'run 16.27 first in this session (it defines the residual heads)'
R16B_ROLE = 'development'          # 'confirmation' only if Q1-Q3 hold
M16B_RESULTS = RESULTS_DIR / f'm16b_comp_{SESSION_TAG}.csv'
R16B_GAMMA = 4.0
R16B_HEADS = ['diff', 'comp']
R16B_EVENT_KEYS = ['n_events', 'n_target', 'target_old', 'all_old'] + [f'{k}_{h}' for h in R16B_HEADS
                                                                       for k in ['target', 'all', 'oe_found']]
M16B_COLUMNS = (['run_id', 'role', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share']
                + R16B_EVENT_KEYS + [f'loss_{h}' for h in R16B_HEADS] + ['seconds'])

def r16b_train_heads(model, series, seed):
    """diff and comp on the same batches; comp up-weights synthetic anomalous patches that the frozen OE misses."""
    set_seed(seed)
    model.eval()
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    with torch.no_grad():
        w = torch.from_numpy(windows).to(DEVICE)
        res = torch.cat([(b - r16_first_pass(model, b)[1]).flatten() for b in w.split(2 * BATCH_SIZE)])
        stats = dict(mean=float(w.mean()), sd=float(w.std()) + 1e-8, res_sd=float(res.std()) + 1e-8)
    heads = {k: R16Residual('diff').to(DEVICE) for k in R16B_HEADS}
    optimizers = {k: torch.optim.Adam(h.parameters(), lr=R16_LR) for k, h in heads.items()}
    pos_weight = torch.tensor([2.0], device=DEVICE)
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    losses = {k: [] for k in R16B_HEADS}
    for epoch in range(R16_EPOCHS):
        totals, batches = {k: 0.0 for k in R16B_HEADS}, 0
        for (clean,) in loader:
            x, labels, _ = training_batch(clean.to(DEVICE), series['period'], True)
            p_oe, x_rec = r16_first_pass(model, x)
            y = patch_labels(labels)
            weight = torch.where(y > 0, 1 + R16B_GAMMA * (1 - p_oe), torch.ones_like(y))
            for k, h in heads.items():
                h.train()
                per_patch = F.binary_cross_entropy_with_logits(h(x, x_rec, stats), y, pos_weight=pos_weight,
                                                               reduction='none')
                loss = (per_patch * weight).sum() / weight.sum() if k == 'comp' else per_patch.mean()
                optimizers[k].zero_grad(set_to_none=True)
                loss.backward()
                optimizers[k].step()
                totals[k] += loss.item()
            batches += 1
        for k in R16B_HEADS:
            losses[k].append(totals[k] / max(batches, 1))
    for h in heads.values():
        h.eval()
    return heads, stats, losses

def read_m16b():
    files = sorted(RESULTS_DIR.glob('m16b_comp_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M16B_COLUMNS)

R16B_GROUPS = split.loc[split['role'] == R16B_ROLE, 'group'].astype(int).tolist()
done = read_m16b()
done = set(done.loc[done['role'] == R16B_ROLE, 'run_id'])
todo = [r for r in code_runs.itertuples()
        if int(r.group) in R16B_GROUPS and int(r.seed) == R16_SEED and r.run_id not in done]
print(f'M16b {R16B_ROLE}: {len(R16B_GROUPS)} series, gamma {R16B_GAMMA}; {len(todo)} models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    t0 = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = code_model(run)
    for p in model.parameters():
        p.requires_grad_(False)
    r16b_heads, r16b_stats, r16b_losses = r16b_train_heads(model, series, seed)
    test = r16_components(model, r16b_heads, r16b_stats, series['test'], series['window'], series['period'])
    train = r16_components(model, r16b_heads, r16b_stats, series['train'], series['window'], series['period'])
    scores = r16_scores(test, train, series, R16B_HEADS)
    assert np.abs(scores['B0 CoAD'] - anomaly_score(model, series)).max() < 1e-4, 'B0 does not reproduce the CoAD score'
    events = r16_events(test, series, R16B_HEADS)
    losses = {f'loss_{k}': f'{v[0]:.4f}->{v[-1]:.4f}' for k, v in r16b_losses.items()}
    rows = [dict(run_id=run.run_id, role=R16B_ROLE, group=group, seed=seed, family=family_of[group], variant=v,
                 **score_metrics(s, series), top_k_normal_share=alarm_stats(series, s)['top_k_normal_share'],
                 **events, **losses, seconds=round(time.time() - t0, 1)) for v, s in scores.items()]
    (pd.DataFrame(rows).reindex(columns=M16B_COLUMNS)
     .to_csv(M16B_RESULTS, mode='a', header=not M16B_RESULTS.exists(), index=False))
    vus = {r['variant']: round(100 * r['VUS_PR'], 1) for r in rows if r['variant'].endswith('calibrated')}
    print(f"[{i}/{len(todo)}] {family_of[group]} {group} ({time.time() - t0:.0f} s): target events {events['n_target']}, "
          f"found old {events['target_old']}, diff {events.get('target_diff')}, comp {events.get('target_comp')}; "
          f"VUS-PR {vus}", flush=True)

### 16.30. M16b results and decision Q1-Q3

In [ ]:
mb = read_m16b().drop_duplicates(['run_id', 'role', 'variant'], keep='last')
for role_name in ['development', 'confirmation']:
    d = mb[mb['role'] == role_name]
    if not len(d):
        continue
    per = 100 * d.pivot_table(index='group', columns='variant', values='VUS_PR')
    variants = ['B0 CoAD', 'B1 calibrated', 'B1 error x2', 'diff CoAD', 'diff calibrated', 'comp CoAD', 'comp calibrated']
    print(f'===== M16b {role_name}: {per.shape[0]} series (seed {R16_SEED}), gamma {R16B_GAMMA}')
    summary = d.groupby('variant')[list(METRICS4) + ['top_k_normal_share']].mean()
    summary[list(METRICS4)] *= 100
    display(summary.rename(columns=METRICS4).reindex(variants).round(3))
    fam = d.drop_duplicates('group').set_index('group')['family']
    print('VUS-PR by family:')
    display(per.groupby(fam.reindex(per.index).values).mean()[variants].round(2))
    ev = d.drop_duplicates('run_id')
    n_t, n_e = ev['n_target'].sum(), ev['n_events'].sum()
    rate = {h: ev[f'target_{h}'].sum() / max(n_t, 1) for h in ['old'] + R16B_HEADS}
    print(f'Events: {n_e} events, {n_t} that OE misses and the raw error finds')
    display(pd.DataFrame({'found among OE-missed / error-found': rate,
                          'found among all events': {h: ev[f'all_{h}'].sum() / max(n_e, 1) for h in ['old'] + R16B_HEADS}}).round(3))
    x = per['comp calibrated'] - per['diff calibrated']
    tk = d.groupby('variant')['top_k_normal_share'].mean()
    q1 = rate['comp'] >= 0.5 and rate['comp'] - rate['diff'] >= 0.10
    q2 = x.mean() >= 0
    q3 = tk['comp calibrated'] - tk['diff calibrated']
    print(f"comp calibrated - diff calibrated: {x.mean():+.2f} VUS-PR, higher on {int((x > 0).sum())}/{len(x)}")
    if role_name == 'development':
        print(f"Q1 (comp >= 0.50 and >= diff + 0.10): comp {rate['comp']:.3f}, diff {rate['diff']:.3f} -> " + ('HOLDS' if q1 else 'FAILS'))
        print(f"Q2 (comp calibrated >= diff calibrated): {x.mean():+.2f} -> " + ('HOLDS' if q2 else 'FAILS'))
        print(f"Q3 (top-k normal share change <= 0.05): {q3:+.3f} -> " + ('HOLDS' if q3 <= 0.05 else 'FAILS'))
        print('Decision: ' + ("comp goes to confirmation: set R16B_ROLE = 'confirmation' in 16.29, run 16.29 and 16.30"
                              if q1 and q2 and q3 <= 0.05 else 'the complementary objective adds nothing: stop'))
    else:
        def one_sided(v):
            return wilcoxon(v, alternative='greater').pvalue if (v != 0).any() else 1.0
        t = pd.DataFrame([dict(vs=ref, difference=(per['comp calibrated'] - per[ref]).mean(),
                               higher=f"{int((per['comp calibrated'] - per[ref] > 0).sum())}/{len(per)}",
                               p=one_sided(per['comp calibrated'] - per[ref])) for ref in ['B1 calibrated', 'B1 error x2']])
        order = t['p'].sort_values().index
        t.loc[order, 'p Holm'] = np.maximum.accumulate([min(1, p * (len(t) - k)) for k, p in enumerate(t.loc[order, 'p'])])
        display(t.round(4))
        p2 = bool((t['p Holm'] < 0.05).all() and (t['difference'] > 0).all())
        p3 = tk['comp calibrated'] - tk['B1 calibrated']
        print('P2 (> B1 calibrated and > B1 error x2, Holm p < 0.05): ' + ('HOLDS' if p2 else 'FAILS'))
        print(f'P3 (top-k normal share increase <= 0.05): {p3:+.3f} -> ' + ('HOLDS' if p3 <= 0.05 else 'FAILS'))
        print('Decision: ' + ('go -> joint training with comp, then a new external set' if p2 and p3 <= 0.05
                              else 'stop: comp does not improve on the calibrated baseline'))

### 16.31. M17: scoring checks on the saved models (re-run to continue)

**Why (reviewers).** Three questions about the score that need no retraining:

1. **Mechanism of `calibrated`** (post-hoc in M13: gain grows when the reconstruction branch is the narrower one,
   signed log(SD error / SD classify), rho = -0.64). Tested here on data not used to form it: the 34 confirmation
   series of M11.
2. **Robust scale**: `calibrated` divides by the training SD, which contamination and heavy tails inflate.
   `B1 MAD` divides by 1.4826 x MAD instead (no parameter).
3. **Reconstruction weight**: on the M16 design series, counting the reconstruction branch twice beat `calibrated`
   (+0.75). Weight w in {1, 2, 3} on the calibrated reconstruction branch, w locked on the design series.

**Design.** Inference only: code-mode models of the 64 paper series (design and confirmation halves of M11, 3 seeds)
and of the 60 M13 series (`runs/m13`, 3 seeds). Same code-mode processing as CoAD. Variants: `B0 CoAD`,
`B1 SD` (= `calibrated`), `B1 MAD`, `B1 err w2`, `B1 err w3`; per model the signed log SD ratio and log MAD ratio
of the two processed branches on the training part.

**Predictions written before running:**

* (G1) confirmation series (mean of seeds per series): Spearman(signed log SD ratio, B1 - B0) < 0, one-sided
  p < 0.05, and more than half of the series have a log ratio < 0;
* (G2) `B1 MAD` >= `B1 SD` on the confirmation series (mean VUS-PR) and `B1 MAD` - `B0` > 0 in >= 6 of 12 M13 families;
* (G3) the weight w locked on the design series (highest mean VUS-PR; w = 1 means no change) beats `B1 SD` on the
  confirmation series (Wilcoxon one-sided, Holm with G2's test, p < 0.05) and on M13 (p < 0.05, >= 7/12 families).

Decision: G1 holds -> the mechanism of `calibrated` is confirmed on unseen series; otherwise the M13 explanation is
withdrawn. G2 / G3: a variant is reported as an improvement only if every part holds.

**Risks.** 34 series is small; the confirmation half was already seen in M11-M12 (not for this question); M13 was seen
in M13 (reported as an external check only). Rows in `results/m17_scoring_<SESSION_TAG>.csv`.

In [ ]:
M17_RESULTS = RESULTS_DIR / f'm17_scoring_{SESSION_TAG}.csv'
M17_VARIANTS = ['B0 CoAD', 'B1 SD', 'B1 MAD', 'B1 err w2', 'B1 err w3']
M17_COLUMNS = (['source', 'role', 'run_id', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS
               + ['top_k_normal_share', 'log_sd_ratio', 'log_mad_ratio'])

def m17_branches(model, series):
    period, window = series['period'], series['window']
    te = predict_components(model, series['test'], window, period)
    tr = predict_components(model, series['train'], window, period)
    r, r_tr = processed(np.abs(series['test'] - te['reconstruction']), np.abs(series['train'] - tr['reconstruction']), period)
    c, c_tr = processed(te['combined'], tr['combined'], period)
    return r, r_tr, c, c_tr

def m17_mad(v):
    mad = 1.4826 * np.median(np.abs(v - np.median(v)))
    return mad if mad > 1e-12 else v.std() + 1e-12

def m17_scores(r, r_tr, c, c_tr):
    sr, sc = r_tr.std() + 1e-12, c_tr.std() + 1e-12
    mr, mc = m17_mad(r_tr), m17_mad(c_tr)
    scores = {'B0 CoAD': (r + c) / 2, 'B1 SD': (r / sr + c / sc) / 2, 'B1 MAD': (r / mr + c / mc) / 2,
              'B1 err w2': (2 * r / sr + c / sc) / 3, 'B1 err w3': (3 * r / sr + c / sc) / 4}
    return scores, float(np.log(sr / sc)), float(np.log(mr / mc))

split17 = pd.read_csv(M11_SPLIT).set_index('group')['role']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
family64 = manifest64.set_index('group')['family']
jobs17 = [('tsbad64', split17.get(int(r.group), 'unknown'), r.run_id, int(r.group), int(r.seed), family64[int(r.group)],
           RUNS / r.run_id / 'model.pt') for r in code_runs.itertuples()]
m13_files = sorted(RESULTS_DIR.glob('m13_external_*.csv'))
if m13_files:
    m13r = pd.concat([pd.read_csv(f) for f in m13_files], ignore_index=True)
    m13r = m13r[(m13r['status'] == 'success') & (m13r['variant'] == 'B0 CoAD')].drop_duplicates(['group', 'seed'], keep='last')
    jobs17 += [('m13', 'external', r.run_id, int(r.group), int(r.seed), r.family, STORE / 'runs' / 'm13' / r.run_id / 'model.pt')
               for r in m13r.itertuples()]
missing17 = [j for j in jobs17 if not j[6].exists()]
if missing17:
    print(f'WARNING: {len(missing17)} saved models not found (skipped), e.g. {missing17[0][6]}')
jobs17 = [j for j in jobs17 if j[6].exists()]

def read_m17():
    files = sorted(RESULTS_DIR.glob('m17_scoring_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M17_COLUMNS)

done17 = set(read_m17()['run_id'])
todo17 = [j for j in jobs17 if j[2] not in done17]
print(f'{len(jobs17)} models ({sum(j[0] == "tsbad64" for j in jobs17)} paper, {sum(j[0] == "m13" for j in jobs17)} M13); {len(todo17)} to do')
session_start = time.time()
for i, (source, role17, run_id, group, seed, family, path) in enumerate(todo17, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    series = load_series(group)
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    scores, log_sd, log_mad = m17_scores(*m17_branches(model, series))
    assert np.abs(scores['B0 CoAD'] - anomaly_score(model, series)).max() < 1e-4, 'B0 does not reproduce the CoAD score'
    rows = [dict(source=source, role=role17, run_id=run_id, group=group, seed=seed, family=family, variant=v,
                 **score_metrics(s, series), top_k_normal_share=alarm_stats(series, s)['top_k_normal_share'],
                 log_sd_ratio=log_sd, log_mad_ratio=log_mad) for v, s in scores.items()]
    pd.DataFrame(rows).reindex(columns=M17_COLUMNS).to_csv(M17_RESULTS, mode='a', header=not M17_RESULTS.exists(), index=False)
    print(f'[{i}/{len(todo17)}] {source} {family} {group} s{seed}: log SD ratio {log_sd:+.2f}; VUS-PR '
          + ', '.join(f"{r['variant']} {100 * r['VUS_PR']:.1f}" for r in rows), flush=True)

### 16.32. M17 results and decisions G1-G3

In [ ]:
from scipy.stats import spearmanr
m17 = read_m17().drop_duplicates(['run_id', 'variant'], keep='last')
def per_series(d):
    return 100 * d.pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
def one_sided(x):
    return wilcoxon(x, alternative='greater').pvalue if (x != 0).any() else 1.0
for name, d in [('design', m17[m17['role'] == 'development']), ('confirmation', m17[m17['role'] == 'confirmation']),
                ('M13 external', m17[m17['source'] == 'm13'])]:
    if len(d):
        print(f'{name}: {d["group"].nunique()} series, {d["run_id"].nunique()} models')
        display(per_series(d).mean()[M17_VARIANTS].round(2).to_frame('VUS-PR').T)

conf = m17[m17['role'] == 'confirmation']
pc = per_series(conf)
ratio = conf.drop_duplicates('run_id').groupby('group')['log_sd_ratio'].mean().reindex(pc.index)
gain = pc['B1 SD'] - pc['B0 CoAD']
rho, p_rho = spearmanr(ratio, gain)
p_rho = p_rho / 2 if rho < 0 else 1 - p_rho / 2
neg = (ratio < 0).mean()
g1 = rho < 0 and p_rho < 0.05 and neg > 0.5
print(f'G1: Spearman(signed log SD ratio, B1 - B0) = {rho:+.3f}, one-sided p = {p_rho:.4f}; share with log ratio < 0 = {neg:.2f} -> '
      + ('HOLDS' if g1 else 'FAILS'))

dev = per_series(m17[m17['role'] == 'development'])
w_scores = {1: dev['B1 SD'].mean(), 2: dev['B1 err w2'].mean(), 3: dev['B1 err w3'].mean()}
w_lock = max(w_scores, key=w_scores.get)
w_name = 'B1 SD' if w_lock == 1 else f'B1 err w{w_lock}'
print(f'Weight locked on the design series: w = {w_lock} ({w_name}); design means {dict((k, round(v, 2)) for k, v in w_scores.items())}')

ext = m17[m17['source'] == 'm13']
pe = per_series(ext) if len(ext) else None
fam_e = ext.drop_duplicates('group').set_index('group')['family'] if len(ext) else None
tests = [dict(test='MAD vs SD (confirmation)', diff=(pc['B1 MAD'] - pc['B1 SD']).mean(), p=one_sided(pc['B1 MAD'] - pc['B1 SD']))]
if w_lock != 1:
    tests.append(dict(test=f'{w_name} vs SD (confirmation)', diff=(pc[w_name] - pc['B1 SD']).mean(), p=one_sided(pc[w_name] - pc['B1 SD'])))
tests = pd.DataFrame(tests)
order = tests['p'].sort_values().index
tests.loc[order, 'p Holm'] = np.maximum.accumulate([min(1, p * (len(tests) - k)) for k, p in enumerate(tests.loc[order, 'p'])])
display(tests.round(4))
if pe is not None:
    fam_mad = (pe['B1 MAD'] - pe['B0 CoAD']).groupby(fam_e.reindex(pe.index).values).mean()
    g2 = (pc['B1 MAD'].mean() >= pc['B1 SD'].mean()) and (fam_mad > 0).sum() >= 6
    print(f"G2: MAD - SD on confirmation {(pc['B1 MAD'] - pc['B1 SD']).mean():+.2f}; MAD - B0 positive in {(fam_mad > 0).sum()}/12 M13 families -> "
          + ('HOLDS' if g2 else 'FAILS'))
    if w_lock == 1:
        print('G3: the design series chose w = 1 (no change) -> nothing to confirm')
    else:
        x = pe[w_name] - pe['B1 SD']
        fam_w = x.groupby(fam_e.reindex(pe.index).values).mean()
        p_conf = tests.loc[tests['test'].str.startswith(w_name), 'p Holm'].iloc[0]
        g3 = p_conf < 0.05 and one_sided(x) < 0.05 and (fam_w > 0).sum() >= 7
        print(f'G3: {w_name} - SD: confirmation Holm p = {p_conf:.4f}; M13 {x.mean():+.2f}, p = {one_sided(x):.4f}, '
              f'positive in {(fam_w > 0).sum()}/12 families -> ' + ('HOLDS' if g3 else 'FAILS'))
    flagged = ['NAB', 'WSD', 'YAHOO', 'Stock']
    keep = ~fam_e.reindex(pe.index).isin(flagged).values
    print('M13 without the families the paper excludes (NAB, WSD, YAHOO, Stock):',
          {v: round((pe[v] - pe['B0 CoAD'])[keep].mean(), 2) for v in M17_VARIANTS[1:]})

### 16.33. M18: order-free attribution of the signal loss (Shapley over processing steps; re-run to continue)

**Why (reviewers).** M9 attributes each missed event to the first step that loses it (processing, then smoothing,
then fusion), so the shares depend on that order. Here each step is switched on or off and the loss is attributed by
Shapley values over the 3! = 6 orders.

**Steps** on the two raw branches (raw error `|x - x_rec|` and `A_c`), as in code mode:

* `X` processing: `|v - mean of v on train|` (on) or `v` (off);
* `S` smoothing: moving average over one period (on) or none (off); the first period is zeroed in both cases;
* `G` fusion: mean of the two processed branches (on), or "found by either branch" (off).

An event is found when its peak percentile against the reference normal points is >= a threshold (0.95 / 0.99 /
0.999); two references: all normal points after the first period (as M9) and normal points more than one window from
any event (as M16). Per event, `d(X, S, G)` in {0, 1} for the 8 configurations; the Shapley value of a step is the
mean over the 6 orders of the drop in `d` when that step is added. The sum of the three equals
`d(none) - d(all)`. Labels are used only for this diagnosis.

**Prediction written before running:** on IOPS, SVDB and SED, smoothing `S` takes >= 50% of the total loss at all
three thresholds, with the M9 reference. Decision: holds -> "lost mainly by smoothing" is order-free; fails -> the
report's attribution is revised.

Models: code mode, seed 46, 64 series. Rows (one per model, reference and threshold) in
`results/m18_shapley_<SESSION_TAG>.csv`.

In [ ]:
from itertools import permutations
M18_RESULTS = RESULTS_DIR / f'm18_shapley_{SESSION_TAG}.csv'
M18_SEED, M18_THRESHOLDS = 46, [0.95, 0.99, 0.999]
M18_COLUMNS = ['run_id', 'group', 'family', 'reference', 'threshold', 'n_events', 'd_none', 'd_all', 'phi_X', 'phi_S', 'phi_G']

def m18_step(v, v_tr, period, x_on, s_on):
    w = np.abs(v - v_tr.mean()) if x_on else v.copy()
    if s_on:
        w = moving_average(w, period)
    w[:period] = 0
    return w

def m18_rows(model, series, run_id, family):
    period, window = series['period'], series['window']
    te = predict_components(model, series['test'], window, period)
    tr = predict_components(model, series['train'], window, period)
    raw = {'r': (np.abs(series['test'] - te['reconstruction']), np.abs(series['train'] - tr['reconstruction'])),
           'c': (te['combined'], tr['combined'])}
    label = np.asarray(series['label']).astype(int)
    events = segments(label)
    refs = {'all normal (M9)': (label == 0) & (np.arange(len(label)) >= period)}
    far = refs['all normal (M9)'].copy()
    for a, b in events:
        far[max(0, a - window):b + window] = False
    refs['far normal (M16)'] = far
    rows = []
    for ref_name, ref in refs.items():
        if not ref.any() or not events:
            continue
        peaks = {}
        for x_on in (0, 1):
            for s_on in (0, 1):
                br = {k: m18_step(v, v_tr, period, x_on, s_on) for k, (v, v_tr) in raw.items()}
                br['fused'] = (br['r'] + br['c']) / 2
                peaks[(x_on, s_on)] = {k: np.array([percentile_against(v[ref])(np.array([v[a:b].max()]))[0] for a, b in events])
                                       for k, v in br.items()}
        for thr in M18_THRESHOLDS:
            def d(x_on, s_on, g_on):
                p = peaks[(x_on, s_on)]
                return (p['fused'] >= thr) if g_on else ((p['r'] >= thr) | (p['c'] >= thr))
            phi = {k: np.zeros(len(events)) for k in 'XSG'}
            for order in permutations('XSG'):
                state = {'X': 0, 'S': 0, 'G': 0}
                for step in order:
                    before = d(state['X'], state['S'], state['G']).astype(float)
                    state[step] = 1
                    phi[step] += (before - d(state['X'], state['S'], state['G'])) / 6
            rows.append(dict(run_id=run_id, group=series['group'], family=family, reference=ref_name, threshold=thr,
                             n_events=len(events), d_none=float(d(0, 0, 0).sum()), d_all=float(d(1, 1, 1).sum()),
                             phi_X=float(phi['X'].sum()), phi_S=float(phi['S'].sum()), phi_G=float(phi['G'].sum())))
    return rows

def read_m18():
    files = sorted(RESULTS_DIR.glob('m18_shapley_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M18_COLUMNS)

if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
done18 = set(read_m18()['run_id'])
todo18 = [r for r in code_runs.itertuples() if int(r.seed) == M18_SEED and r.run_id not in done18]
print(len(todo18), 'models for M18')
session_start = time.time()
for i, run in enumerate(todo18, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group = int(run.group)
    series = load_series(group)
    rows = m18_rows(code_model(run), series, run.run_id, manifest64.set_index('group')['family'][group])
    pd.DataFrame(rows).reindex(columns=M18_COLUMNS).to_csv(M18_RESULTS, mode='a', header=not M18_RESULTS.exists(), index=False)
    print(f'[{i}/{len(todo18)}] group {group}: {len(rows)} rows', flush=True)

### 16.34. M18 results and decision

In [ ]:
m18 = read_m18().drop_duplicates(['run_id', 'reference', 'threshold'], keep='last')
agg = m18.groupby(['reference', 'family', 'threshold'])[['n_events', 'd_none', 'd_all', 'phi_X', 'phi_S', 'phi_G']].sum()
loss = (agg['d_none'] - agg['d_all']).where(lambda v: v >= 5)      # shares only when >= 5 events are lost
table = pd.DataFrame({'found, no step': agg['d_none'] / agg['n_events'], 'found, all steps': agg['d_all'] / agg['n_events'],
                      'events lost': agg['d_none'] - agg['d_all'],
                      'share X': agg['phi_X'] / loss, 'share S (smoothing)': agg['phi_S'] / loss, 'share G': agg['phi_G'] / loss})
display(table.round(3))
m9 = table.xs('all normal (M9)', level='reference')
checks = [m9.loc[(f, t), 'share S (smoothing)'] for f in ['IOPS', 'SVDB', 'SED'] for t in M18_THRESHOLDS if (f, t) in m9.index]
checks = [c for c in checks if pd.notna(c)]
ok = bool(checks) and all(c >= 0.5 for c in checks)
print(f'Prediction (smoothing >= 50% of the loss on IOPS, SVDB, SED at all thresholds, M9 reference; {len(checks)} cells with >= 5 lost events): '
      + ('HOLDS' if ok else 'FAILS'))

### 16.35. M19: TODS training contamination (retraining; re-run to continue)

**Why (reviewers).** 13 of the 15 TODS series have labelled anomalies in their training part, and 52% of TODS events
are found by no branch (M9). Is that a data / protocol confound or a limit of the model?

**Design.** 15 TODS series x seeds 46-48, code mode, same settings as section 7; only the training windows change:

* `label_clean` (diagnosis, uses the training labels): replace every labelled training point by linear interpolation
  between the nearest unlabelled points. (Dropping contaminated windows was the first design, but in the local test
  every TODS training window contained a labelled point, so nothing was left to train on.)
* `z_clean` (label-free candidate): the same repair for every training point with |z| > 4, z from the training median
  and 1.4826 x MAD (threshold fixed before running);
* control: the saved code-mode model of the same (series, seed), scored by the same code in the same cell
  (GPU nondeterminism between runs is a known caveat, up to several points per series).

Per model: VUS-PR and the share of events found by no branch (OE, residual, raw error, unsmoothed final; peak percentile
>= 0.99 against all normal points, as M9); number of training points repaired.

**Predictions written before running:**

* (T1) `label_clean` - control: mean VUS-PR over TODS >= +2 **or** "no branch" share drops by >= 0.10 -> contamination
  is a confound for the TODS conclusions; otherwise the TODS failures are attributed to amplitude / model limits;
* (T2) `z_clean` - control: >= +2 VUS-PR and Wilcoxon one-sided p < 0.05 over the 15 series -> a label-free training
  filter becomes a candidate improvement (to be tested on other families before any claim).

Cost: 90 trainings (about 1-2 h on a T4). Rows in `results/m19_tods_<SESSION_TAG>.csv`.

In [ ]:
M19_RESULTS = RESULTS_DIR / f'm19_tods_{SESSION_TAG}.csv'
M19_VARIANTS = ['control', 'label_clean', 'z_clean']
M19_COLUMNS = ['group', 'seed', 'variant', 'points_total', 'points_repaired'] + METRIC_KEYS + ['n_events', 'no_branch_share', 'final_found_share', 'seconds']
TODS_GROUPS = manifest64.loc[manifest64['family'] == 'TODS', 'group'].astype(int).tolist()
raw_files = {int(f.name.split('_')[0]): f for f in RAW.glob('*.csv')}

def m19_train_labels(series):
    return pd.read_csv(raw_files[series['group']]).iloc[:len(series['train']), -1].to_numpy().astype(int)

def m19_repaired_train(series, variant):
    """Training values with the flagged points replaced by linear interpolation; returns (values, number repaired)."""
    x = np.asarray(series['train'], dtype=float).copy()
    if variant == 'label_clean':
        bad = m19_train_labels(series).astype(bool)
    else:
        z = np.abs(x - np.median(x)) / (1.4826 * np.median(np.abs(x - np.median(x))) + 1e-12)
        bad = z > 4
    if bad.any() and (~bad).sum() >= 2:
        idx = np.arange(len(x))
        x[bad] = np.interp(idx[bad], idx[~bad], x[~bad])
    return x, int(bad.sum())

def m19_train(series, seed, windows):
    """train_model of section 5.2 (code mode) on a given set of training windows."""
    set_seed(seed)
    model = CoADReimpl(True).to(DEVICE)
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99)
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    for epoch in range(EPOCHS):
        model.train()
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], True, True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                loss = coad_loss(model, x, labels, target, True)[0]
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
    return model

def m19_eval(model, series):
    signals, _ = model_signals(model, series)
    label = np.asarray(series['label']).astype(int)
    normal = (label == 0) & (np.arange(len(label)) >= series['period'])
    events = segments(label)
    found = {k: np.array([percentile_against(signals[k][normal])(np.array([signals[k][a:b].max()]))[0] >= 0.99 for a, b in events])
             for k in ['oe', 'residual', 'error', 'full_unsmoothed', 'full']}
    any_branch = found['oe'] | found['residual'] | found['error'] | found['full_unsmoothed']
    return dict(**score_metrics(anomaly_score(model, series), series), n_events=len(events),
                no_branch_share=float((~any_branch).mean()) if events else np.nan,
                final_found_share=float(found['full'].mean()) if events else np.nan)

def read_m19():
    files = sorted(RESULTS_DIR.glob('m19_tods_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M19_COLUMNS)

if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
saved = {(int(r.group), int(r.seed)): r for r in code_runs.itertuples()}
done19 = read_m19()
done19 = set(zip(done19['group'].astype(int), done19['seed'].astype(int), done19['variant'])) if len(done19) else set()
todo19 = [(g, s, v) for s in SEEDS for g in TODS_GROUPS for v in M19_VARIANTS if (g, s, v) not in done19 and (g, s) in saved]
print(f'{len(TODS_GROUPS)} TODS series; {len(todo19)} (series, seed, variant) to do')
session_start = time.time()
for i, (group, seed, variant) in enumerate(todo19, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    t0 = time.time()
    series = load_series(group)
    n_total = len(series['train'])
    if variant == 'control':
        model, n_fixed = code_model(saved[(group, seed)]), 0
    else:
        repaired, n_fixed = m19_repaired_train(series, variant)
        windows, _ = make_windows(repaired, series['window'], series['train_stride'])
        model = m19_train(series, seed, windows)
    row = dict(group=group, seed=seed, variant=variant, points_total=n_total, points_repaired=n_fixed,
               **m19_eval(model, series), seconds=round(time.time() - t0, 1))
    pd.DataFrame([row]).reindex(columns=M19_COLUMNS).to_csv(M19_RESULTS, mode='a', header=not M19_RESULTS.exists(), index=False)
    print(f"[{i}/{len(todo19)}] TODS {group} s{seed} {variant}: repaired {n_fixed}/{n_total} points, VUS-PR {100 * row['VUS_PR']:.2f}, "
          f"no-branch share {row['no_branch_share']:.3f} ({row['seconds']:.0f} s)", flush=True)

### 16.36. M19 results and decisions T1-T2

In [ ]:
m19 = read_m19().drop_duplicates(['group', 'seed', 'variant'], keep='last')
per19 = m19.pivot_table(index=['group', 'seed'], columns='variant', values=['VUS_PR', 'no_branch_share'])
vus = (100 * per19['VUS_PR'].groupby('group').mean()).reindex(columns=M19_VARIANTS)
nb = per19['no_branch_share'].groupby('group').mean().reindex(columns=M19_VARIANTS)
assert vus.notna().all().all(), 'every variant needs results for every series: run 16.35 to the end first'
print(f"{vus.shape[0]} TODS series; share of training points repaired (median): "
      f"{(m19['points_repaired'] / m19['points_total']).groupby(m19['variant']).median().round(4).to_dict()}")
display(pd.DataFrame({'VUS-PR': vus.mean(), 'no-branch share': nb.mean()}).reindex(M19_VARIANTS).round(3))
d_label, d_z = vus['label_clean'] - vus['control'], vus['z_clean'] - vus['control']
nb_drop = nb['control'].mean() - nb['label_clean'].mean()
t1 = d_label.mean() >= 2 or nb_drop >= 0.10
p_z = wilcoxon(d_z, alternative='greater').pvalue if (d_z != 0).any() else 1.0
t2 = d_z.mean() >= 2 and p_z < 0.05
print(f'T1 (label_clean: {d_label.mean():+.2f} VUS-PR, no-branch share drop {nb_drop:+.3f}) -> '
      + ('HOLDS: contamination is a confound' if t1 else 'FAILS: TODS failures attributed to amplitude / model limits'))
print(f'T2 (z_clean: {d_z.mean():+.2f} VUS-PR, higher on {int((d_z > 0).sum())}/{len(d_z)}, p = {p_z:.4f}) -> '
      + ('HOLDS: label-free filter is a candidate' if t2 else 'FAILS'))

### 16.39. M20: partial check of the KDD21 column of Table 1 (training; re-run to continue)

**Why (reviewer).** Table 1 reports CoAD on two benchmarks, KDD21 and TSB-AD. The project only checked the TSB-AD
half, so "Table 1 is not reproducible" must be limited to TSB-AD unless KDD21 is checked too.

**Data.** The authors' loader needs their preprocessed KDD21 files (`dataset/UCR/processed/*.npy`), which are not
published. TSB-AD-U contains the same KDD21 series as its family `UCR`, repackaged by TSB-AD (its own train split and
min-max scaling on the training part). This check uses those files, which is a **preprocessing difference** from the
authors' setup and must be stated with the result.

**Evaluation.** For KDD21 the authors call the evaluator with `use_ucr=True`. In `evaluation/evaluator.py` this flag
only adds a UCR peak-accuracy metric: F1, AUC-PR, R-AUC-PR and VUS-PR are computed the same way (VUS always with
`subsequence_length=None`), so the four Table 1 metrics are comparable.

**Design.**

* 30 series drawn from the TSB-AD-U `UCR` family with `default_rng(2028)`: length <= 100 000 points, byte-identical
  files removed (md5), the 5 series already used in M13 excluded, valid window settings (as M13);
* CoAD code mode, as section 7, seed 46 only (Table 1 averages 5 seeds; one seed is a stated limit);
* mean of each metric with a bootstrap 95% CI over series (10 000 resamples, seed 0).

**Prediction written before running:** the Table 1 KDD21 value of VUS-PR (45.67) lies outside our bootstrap 95% CI.
Decision: outside -> the deviation from Table 1 extends to KDD21 (with the preprocessing caveat); inside -> the claim
stays limited to TSB-AD. F1, AUC-PR and R-AUC-PR are reported the same way, as secondary results.

**Risks.** 30 of the 250 KDD21 series give a wide CI; a different preprocessing and period estimate from the
authors'; one seed. Cost: about 30-45 GPU minutes. Rows in `results/m20_kdd21_<SESSION_TAG>.csv`.

In [ ]:
M20_FILE = STORE / 'eda' / 'm20_kdd21.csv'
M20_RESULTS = RESULTS_DIR / f'm20_kdd21_{SESSION_TAG}.csv'
M20_N, M20_SEED, M20_MAX_LENGTH = 30, 46, 100_000
TABLE1_KDD21 = {'F1': 52.82, 'AUC-PR': 48.10, 'R-AUC-PR': 46.35, 'VUS-PR': 45.67}
M20_COLUMNS = ['run_id', 'group', 'seed', 'status', 'error'] + METRIC_KEYS + ['train_seconds']
if not M20_FILE.exists():
    import hashlib
    files = sorted(f for f in RAW.glob('*.csv') if f.name.split('_')[1] == 'UCR')
    used = set()
    m13_file = STORE / 'eda' / 'm13_external.csv'
    if m13_file.exists():
        used = set(pd.read_csv(m13_file)['group'].astype(int))
    cat = pd.DataFrame([dict(group=int(f.name.split('_')[0]), file=f.name) for f in files])
    cat = cat[~cat['group'].isin(used)]
    cat['length'] = [sum(1 for _ in open(RAW / f)) - 1 for f in cat['file']]
    cat['md5'] = [hashlib.md5((RAW / f).read_bytes()).hexdigest() for f in cat['file']]
    cat = cat.sort_values('group').drop_duplicates('md5')
    cat = cat[cat['length'] <= M20_MAX_LENGTH]
    chosen = []
    for g in np.random.default_rng(2028).permutation(cat['group'].to_numpy()):
        try:
            s = load_series(int(g))
            ok = s['window'] >= 2 * PATCH and len(s['train']) >= s['window'] and s['period'] >= 2
        except Exception:
            ok = False
        if ok:
            chosen.append(int(g))
        if len(chosen) == M20_N:
            break
    pd.DataFrame({'group': chosen}).to_csv(M20_FILE, index=False)
M20_GROUPS = pd.read_csv(M20_FILE)['group'].astype(int).tolist()
print(len(M20_GROUPS), 'KDD21 series (TSB-AD-U family UCR):', M20_GROUPS)

def read_m20():
    files = sorted(RESULTS_DIR.glob('m20_kdd21_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M20_COLUMNS)

done20 = read_m20()
done20 = set(done20.loc[done20['status'] == 'success', 'group'].astype(int)) if len(done20) else set()
todo20 = [g for g in M20_GROUPS if g not in done20]
print(len(todo20), 'models to train')
session_start = time.time()
for i, group in enumerate(todo20, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_m20_{group}_s{M20_SEED}"
    try:
        series = load_series(group)
        t0 = time.time()
        model, _ = train_model(series, faithful=True, seed=M20_SEED, log_every=0)
        row = dict(run_id=run_id, group=group, seed=M20_SEED, status='success', error='',
                   **score_metrics(anomaly_score(model, series), series), train_seconds=round(time.time() - t0, 1))
    except Exception as error:
        if isinstance(error, NameError):
            raise
        row = dict(run_id=run_id, group=group, seed=M20_SEED, status='failed', error=repr(error)[:200])
    pd.DataFrame([row]).reindex(columns=M20_COLUMNS).to_csv(M20_RESULTS, mode='a', header=not M20_RESULTS.exists(), index=False)
    print(f"[{i}/{len(todo20)}] KDD21 group {group}: {row['status']}, VUS-PR "
          f"{100 * row.get('VUS_PR', float('nan')):.2f}", flush=True)

### 16.40. M20 results and decision

In [ ]:
m20 = read_m20()
m20 = m20[m20['status'] == 'success'].drop_duplicates('group', keep='last')
print(f'{len(m20)} of {len(M20_GROUPS)} KDD21 series done')
rng = np.random.default_rng(0)
rows = []
for key, name in METRICS4.items():
    v = 100 * m20[key].to_numpy()
    boot = rng.choice(v, size=(10_000, len(v)), replace=True).mean(1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rows.append({'metric': name, 'Table 1 (KDD21)': TABLE1_KDD21[name], 'ours (mean)': v.mean(), 'CI 2.5%': lo, 'CI 97.5%': hi,
                 'difference': v.mean() - TABLE1_KDD21[name], 'Table 1 outside CI': not (lo <= TABLE1_KDD21[name] <= hi)})
res20 = pd.DataFrame(rows).set_index('metric')
display(res20.round(2))
outside = bool(res20.loc['VUS-PR', 'Table 1 outside CI'])
print('Prediction (Table 1 VUS-PR outside our 95% CI): ' + ('HOLDS -> the deviation extends to KDD21 (TSB-AD-U preprocessing, 1 seed, 30 series)'
      if outside else 'FAILS -> the deviation claim stays limited to TSB-AD'))
print('Direction on TSB-AD (section 3): F1 / AUC-PR lower, R-AUC-PR / VUS-PR higher than Table 1. Same here: ',
      {k: ('lower' if res20.loc[k, 'difference'] < 0 else 'higher') for k in res20.index})

### 16.41. M21: early-stopping audit (training; re-run to continue)

**Why (reviewer).** The paper states that it uses early stopping for model selection. The authors'
`train_classify_mask.py` trains a fixed number of epochs with no validation-based stopping or best-model restore.
This is one more paper/code difference and a possible source of the deviation from Table 1.

**Design** (label-free; the validation loss uses the same synthetic distortions as training):

* the training part is split in time: first 80% for training, last 20% for validation (no shared time stamps);
  series whose last 20% is shorter than one window are skipped (reported);
* `es`: train on the 80%, compute CoAD's loss on a **fixed** set of distorted validation batches after every epoch,
  stop after 10 epochs without improvement (max 300 = section 7), restore the best epoch;
* `cut80`: train on the same 80% for the full 300 epochs, no early stopping (control: separates the effect of early
  stopping from the effect of training on less data);
* reference: the saved code-mode results (100% of the training part, 300 epochs) of the same series and seeds;
* default series: the 9 MGAB series (largest paper/code gap) x seeds 46-48; set `M21_FAMILIES` to extend.

**Prediction written before running:** |mean VUS-PR(`es`) - mean VUS-PR(`cut80`)| < the seed SD of code mode on
MGAB (3.57, section 3.3). Decision: holds -> early stopping is ruled out as a material source of the deviation on
MGAB; fails -> it is reported as a candidate source, and the 64 series should be run with seed 46.

**Risks.** The validation rule (last 20%, patience 10) is one plausible protocol, not the authors' (unknown).
Cost: about 54 trainings on MGAB, 1.5-2.5 GPU hours. Rows in `results/m21_es_<SESSION_TAG>.csv`.

In [ ]:
import copy
M21_RESULTS = RESULTS_DIR / f'm21_es_{SESSION_TAG}.csv'
M21_FAMILIES, M21_VARIANTS, M21_PATIENCE, M21_VAL_SHARE = ['MGAB'], ['es', 'cut80'], 10, 0.2
M21_COLUMNS = ['group', 'seed', 'family', 'variant', 'status', 'note', 'epochs_run', 'best_epoch'] + METRIC_KEYS + ['seconds']
M21_GROUPS = manifest64.loc[manifest64['family'].isin(M21_FAMILIES), 'group'].astype(int).tolist()

def m21_train(series, seed, train_values, val_values, early_stop):
    set_seed(seed)
    model = CoADReimpl(True).to(DEVICE)
    windows, _ = make_windows(train_values, series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99)
    use_amp = DEVICE.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    val_batches = []
    if early_stop:                                   # fixed distorted validation batches (label-free)
        vw, _ = make_windows(val_values, series['window'], series['train_stride'])
        state = (random.getstate(), np.random.get_state(), torch.get_rng_state())
        set_seed(seed + 10_000)
        for chunk in torch.from_numpy(vw).split(BATCH_SIZE):
            val_batches.append(training_batch(chunk.to(DEVICE), series['period'], True, True))
        random.setstate(state[0]); np.random.set_state(state[1]); torch.set_rng_state(state[2])
    best, best_epoch, best_state, waited = float('inf'), 0, None, 0
    for epoch in range(EPOCHS):
        model.train()
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], True, True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                loss = coad_loss(model, x, labels, target, True)[0]
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        if early_stop:
            model.eval()
            with torch.no_grad():
                val = float(np.mean([coad_loss(model, *b, True)[0].item() for b in val_batches]))
            if val < best:
                best, best_epoch, best_state, waited = val, epoch + 1, copy.deepcopy(model.state_dict()), 0
            else:
                waited += 1
                if waited >= M21_PATIENCE:
                    break
    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
    return model, epoch + 1, (best_epoch if early_stop else epoch + 1)

def read_m21():
    files = sorted(RESULTS_DIR.glob('m21_es_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M21_COLUMNS)

done21 = read_m21()
done21 = set(zip(done21['group'].astype(int), done21['seed'].astype(int), done21['variant'])) if len(done21) else set()
todo21 = [(g, s, v) for s in SEEDS for g in M21_GROUPS for v in M21_VARIANTS if (g, s, v) not in done21]
print(f'{len(M21_GROUPS)} series ({M21_FAMILIES}); {len(todo21)} trainings to do')
session_start = time.time()
for i, (group, seed, variant) in enumerate(todo21, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    t0 = time.time()
    series = load_series(group)
    x = np.asarray(series['train'], dtype=float)
    cut = int(round((1 - M21_VAL_SHARE) * len(x)))
    base = dict(group=group, seed=seed, family=manifest64.set_index('group')['family'][group], variant=variant)
    if len(x) - cut < series['window'] or cut < series['window']:
        row = dict(base, status='skipped', note=f'validation part {len(x) - cut} < window {series["window"]}')
    else:
        model, epochs_run, best_epoch = m21_train(series, seed, x[:cut], x[cut:], early_stop=(variant == 'es'))
        row = dict(base, status='success', note='', epochs_run=epochs_run, best_epoch=best_epoch,
                   **score_metrics(anomaly_score(model, series), series), seconds=round(time.time() - t0, 1))
    pd.DataFrame([row]).reindex(columns=M21_COLUMNS).to_csv(M21_RESULTS, mode='a', header=not M21_RESULTS.exists(), index=False)
    print(f"[{i}/{len(todo21)}] {base['family']} {group} s{seed} {variant}: {row['status']} {row['note']} "
          f"epochs {row.get('epochs_run')}, best {row.get('best_epoch')}, VUS-PR {100 * row.get('VUS_PR', float('nan')):.2f}", flush=True)

### 16.42. M21 results and decision

In [ ]:
m21 = read_m21().drop_duplicates(['group', 'seed', 'variant'], keep='last')
print(m21.groupby(['variant', 'status']).size().to_dict())
ok21 = m21[m21['status'] == 'success']
ref = (read_reimpl_results().query("mode == 'code' and status == 'success'").drop_duplicates(['group', 'seed'], keep='last'))
ref = ref[ref['group'].astype(int).isin(ok21['group'].astype(int).unique()) & ref['seed'].astype(int).isin(ok21['seed'].astype(int).unique())]
table = pd.concat([ok21.assign(variant=ok21['variant']), ref.assign(variant='code (100%, 300 ep)')])
summary = table.groupby('variant')[list(METRICS4)].mean() * 100
display(summary.rename(columns=METRICS4).round(2))
print('Epochs run by es (median, min-max):', ok21.loc[ok21['variant'] == 'es', 'epochs_run'].median(),
      ok21.loc[ok21['variant'] == 'es', 'epochs_run'].min(), '-', ok21.loc[ok21['variant'] == 'es', 'epochs_run'].max())
per = 100 * ok21.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
d = (per['es'] - per['cut80']).groupby('group').mean()
p = wilcoxon(d).pvalue if (d != 0).any() else 1.0
print(f'es - cut80 (VUS-PR, mean over seeds per series): {d.mean():+.2f}, higher on {int((d > 0).sum())}/{len(d)}, two-sided p = {p:.3f}')
print('Prediction (|es - cut80| < 3.57): ' + ('HOLDS -> early stopping ruled out as a material source on MGAB' if abs(d.mean()) < 3.57
      else 'FAILS -> report early stopping as a candidate source; run the 64 series with seed 46'))